# Build GTFS File: LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1
---

**GTFS (General Transit Feed Specification)** adalah standar format data yang digunakan untuk mendeskripsikan informasi transportasi publik seperti rute, jadwal, dan halte/stasiun dalam bentuk file teks yang terstruktur. GTFS memungkinkan operator transportasi umum untuk mempublikasikan data jadwal dan rute mereka dalam format yang bisa dibaca oleh berbagai aplikasi dan platform secara seragam. Dengan GTFS, data bisa langsung diintegrasikan ke Google Maps, Apple Maps, atau aplikasi perjalanan lainnya tanpa konversi manual. Format ini memungkinkan pengembang untuk:

```
- Menampilkan rute & jadwal transit di aplikasi peta (Google Maps, Moovit, dll)
- Membangun aplikasi perencanaan perjalanan multimoda
- Menganalisis cakupan dan aksesibilitas layanan transportasi
- Berbagi data transit secara terbuka untuk publik
```

GTFS terdiri dari kumpulan file **CSV (Comma-Separated Values)** dengan ekstensi `.txt` yang dikompres menjadi satu file `.zip`. Setiap file memiliki skema kolom tertentu yang telah distandarisasi. GTFS terdiri dari dua Varian Utama yaitu:

- `GTFS Static`, data statis yang menggambarkan "rencana" layanan transportasi tidak berubah secara real-time. Biasanya diperbarui secara berkala (mingguan/bulanan) ketika ada perubahan jadwal atau rute.
- `GTFS Realtime`, data dinamis yang menggambarkan kondisi aktual layanan saat ini (Realtime). Menggunakan format Protocol Buffers (protobuf), diperbarui setiap beberapa detik hingga menit. Dengan GTFS-Realtime, Informasi keterlambatan, pembatalan, perubahan jadwal perjalanan dapat diketahui secara live, selain itu dapat mengetahui posisi GPS kendaraan secara real-time di peta. GTFS Realtime tidak bisa berdiri sendiri, membutuhkan GTFS Static sebagai acuan.

*Dokumentasi Resmi:* [GTFS Reference](https://gtfs.org/documentation/overview/)

---
Pada notebook ini akan dilakukan proses pembuatan GTFS Static untuk operasi layanan LRT Jabodebek. Berikut ini file yang akan di-build pada notebook ini:

| File | Status | Deskripsi Singkat |
|------|--------|------------------|
| `agency.txt` | ✅ Required | Informasi operator (PT Kereta Api Indonesia (Persero) Divisi LRT Jabodebek |
| `stops.txt` | ✅ Required | Data halte/stasiun (nama, koordinat, zona) |
| `routes.txt` | ✅ Required | Data rute/layanan (warna, tipe, referensi agency) |
| `trips.txt` | ✅ Required | Data perjalanan spesifik (route, service_id, direction) |
| `stop_times.txt` | ✅ Required | Jadwal kedatangan/keberangkatan di tiap halte/stasiun |
| `calendar.txt` | ⚠️ Conditional | Jadwal operasional reguler (Senin-Minggu) |
| `fare_attributes.txt` | ❌ Optional | Informasi tarif/ticketing |
| `fare_rules.txt` | ❌ Optional | Aturan penerapan tarif |
| `shapes.txt` | ❌ Optional | Koordinat polyline untuk visualisasi rute di peta |

---

Setelah notebook selesai dijalankan, project ini akan memiliki struktur seperti ini:

```
gtfs_lrt_jabodebek/
├── agency.txt
├── calendar.txt
├── fare_attributes.txt
├── fare_rules.txt
├── routes.txt
├── shapes.txt
├── stops.txt
├── stop_times.txt
└── trips.txt
```

**Field Types**

Berikut ini merupakan tipe data yang digunakan pada format file GTFS:

| Tipe Data | Deskripsi | Contoh | Catatan Penting |
|-----------|-----------|--------|----------------|
| **`Color`** | Warna yang dienkode sebagai angka heksadesimal 6-digit. | `FFFFFF` (putih), `000000` (hitam), `0039A6` (biru MTA NY) | Jangan sertakan tanda `#` di awal. Gunakan [htmlcolorcodes.com](https://htmlcolorcodes.com) untuk generate nilai valid. |
| **`Currency code`** | Kode mata uang alfabetis ISO 4217. | `IDR` (Rupiah), `USD` (Dolar AS), `EUR` (Euro) | Lihat daftar lengkap: [ISO 4217 Active Codes](https://en.wikipedia.org/wiki/ISO_4217#Active_codes) |
| **`Currency amount`** | Nilai desimal yang menunjukkan jumlah mata uang. | `14000.00`, `2.50` | Jumlah desimal mengikuti standar ISO 4217 untuk mata uang terkait. **Jangan proses sebagai `float`** — gunakan tipe `decimal`/`currency` untuk hindari error pembulatan finansial. |
| **`Date`** | Hari layanan dalam format `YYYYMMDD`. | `20260416` (16 April 2026) | Waktu dalam hari layanan boleh > `24:00:00`, sehingga satu hari layanan dapat mencakup informasi hari berikutnya. |
| **`Email`** | Alamat email valid. | `cs@kai.id` | Gunakan format standar RFC 5322. |
| **`Enum`** | Opsi dari sekumpulan konstanta yang telah didefinisikan di kolom "Description". | `0` = Trem, `1` = Metro, `3` = Bus (untuk field `route_type`) | Selalu merujuk ke dokumentasi resmi untuk nilai yang diperbolehkan. |
| **`ID`** | Nilai ID internal (tidak untuk ditampilkan ke penumpang), berupa urutan karakter UTF-8 apa pun. | `BK-WD-DKA-JTM-001`, `STOP-DKA-001` | 🔹 Disarankan hanya gunakan karakter ASCII yang dapat dicetak.<br>🔹 **"Unique ID"**: Harus unik dalam satu file.<br>🔹 **"Foreign ID"**: Merujuk ke ID di file lain (misal: `routes.agency_id` merujuk ke `agency.agency_id`). |
| **`Language code`** | Kode bahasa IETF BCP 47. | `id` (Indonesia), `en` (Inggris), `en-US` (Inggris AS) | Panduan: [RFC 4646](http://www.rfc-editor.org/rfc/bcp/bcp47.txt) & [W3C Language Tags](https://www.w3.org/International/articles/language-tags/) |
| **`Latitude`** | Lintang WGS84 dalam desimal derajat. | `-6.2088` (Jati Mulya) | 📍 Rentang nilai: `-90.0` ≤ latitude ≤ `90.0` |
| **`Longitude`** | Bujur WGS84 dalam desimal derajat. | `106.8456` (Jati Mulya) | 📍 Rentang nilai: `-180.0` ≤ longitude ≤ `180.0` |
| **`Float`** | (*floating point*). | `3.14159`, `-0.001` | Hindari untuk nilai mata uang tetap gunakan `Currency amount` dengan tipe `decimal`. |
| **`Integer`** | Integer | `0`, `42`, `-7` | Cocok untuk counter, urutan, atau nilai diskrit. |
| **`Phone number`** | Nomor telepon. | `+62215000332`, `1500332` | Format bebas, tetapi disarankan gunakan format internasional E.164 untuk kompatibilitas global. |
| **`Time`** | Waktu dalam format `HH:MM:SS` (atau `H:MM:SS`). Diukur dari "noon minus 12h" hari layanan (efektif tengah malam, kecuali saat perubahan DST). | `14:30:00` (2:30 PM), `25:35:00` (1:35 AM hari berikutnya) | Untuk waktu setelah tengah malam hari layanan, gunakan nilai > `24:00:00`. Contoh: Kereta tiba pukul 01:15 dini hari → tulis `25:15:00`. |
| **`Local time`** | Waktu dalam format `HH:MM:SS` (atau `H:MM:SS`). Merepresentasikan waktu dinding (*wall-clock time*) sesuai zona waktu lokasi yang ditentukan. | `08:00:00` (jam 8 pagi waktu setempat) | Berbeda dengan `Time`, tipe ini tidak terkait hari layanan — murni waktu lokal absolut. |
| **`Text`** | String karakter UTF-8 yang ditujukan untuk ditampilkan dan harus dapat dibaca manusia. | `Stasiun Jati Mulya`, `LRT Jabodebek` | Gunakan encoding `utf-8` saat ekspor file untuk dukung karakter khusus & aksara lokal. |
| **`Timezone`** | Zona waktu TZ dari [IANA Time Zone Database](https://www.iana.org/time-zones). | `Asia/Jakarta`, `America/New_York` | Nama zona waktu tidak pernah mengandung spasi, tapi boleh menggunakan underscore (`_`). Lihat daftar lengkap: [List of tz zones](https://en.wikipedia.org/wiki/List_of_tz_database_time_zones) |
| **`URL`** | URL lengkap yang menyertakan `http://` atau `https://`, dengan karakter khusus yang di-*escape* dengan benar. | `https://lrtjabodebek.kai.id/stations` | 🔗 Panduan pembuatan URL valid: [W3C URI Recommendations](http://www.w3.org/Addressing/URL/4_URI_Recommentations.html) |


Lebih langkap cek *dokumentasi resmi:* [GTFS Reference](https://gtfs.org/documentation/overview/)

---

## agency.txt (Required)

File `agency.txt` berisi informasi tentang operator transit yang menyediakan layanan dalam dataset GTFS ini. Untuk LRT Jabodebek, file ini mendeskripsikan identitas PT Kereta Api Indonesia (Persero) Divisi LRT Jabodebek sebagai penyedia layanan. Referensi data agency.txt ini diambil dari website resmi [LRT Jabodebek](https://lrtjabodebek.kai.id/).

Direferensi oleh: `routes.txt` (melalui field `agency_id`)

---

**Skema kolom agency.txt**

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `agency_id` | Unique ID | **Kondisional** | ID unik pengenal merek/operator transportasi. **Wajib** jika dataset berisi lebih dari satu operator. Disarankan diisi meskipun hanya satu operator. |
| `agency_name` | Text | **Wajib** | Nama lengkap operator transportasi. |
| `agency_url` | URL | **Wajib** | Alamat website resmi operator. |
| `agency_timezone` | Timezone | **Wajib** | Zona waktu lokasi operator. Jika ada beberapa operator dalam satu dataset, semua harus menggunakan zona waktu yang sama. |
| `agency_lang` | Language Code | Opsional | Bahasa utama yang digunakan operator. Membantu aplikasi menerapkan aturan penulisan dan pengaturan bahasa yang tepat. |
| `agency_phone` | Phone Number | Opsional | Nomor telepon layanan operator. Boleh mengandung tanda baca pemisah angka.|
| `agency_fare_url` | URL | Opsional | URL halaman web tempat penumpang dapat membeli tiket atau melihat informasi tarif layanan. |
| `agency_email` | Email | Opsional | Alamat email layanan pelanggan yang aktif dipantau, sebagai kontak langsung bagi penumpang. |

### Panduan Pengisian `agency.txt` — LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

---

**`agency_id`**

Diisi dengan `KAI` untuk layanan LRT Jabodebek.

---

**`agency_name`**

Diisi dengan operator atau perusahaan yang mengoperasikan layanan LRT Jabodebek yaitu `PT Kereta Api Indonesia (Persero) Divisi LRT Jabodebek`

---

**`agency_url`**

Diisi dengan website resmi dari LRT Jabodebek yaitu `https://lrtjabodebek.kai.id/`

---

**`agency_timezone`**

Diisi dengan zona waktu lokasi layanan LRT Jabodebek yaitu `Asia/Jakarta`

---

**`agency_lang`**

Diisi dengan bahasa utama yang digunakan operator LRT Jabodebek yaitu bahasa indonesia `id`

---

**`agency_phone`**

Diisi dengan nomor telephone call center dari operator LRT Jabodebek yaitu `021121`

---

**`agency_fare_url`**

Diisi dengan URL halaman web tempat penumpang dapat membeli tiket atau melihat informasi tarif layanan, pada LRT Jabodebek bisa diakses pada website resmi LRT Jabodebek pada bagian `https://lrtjabodebek.kai.id/informasi-tarif`

---

**`agency_email`**

Diisi dengan alamat email customer care dari operator layanan LRT Jabodebek yaitu `cs@kai.id`

In [1]:
import pandas as pd
import os
import csv

# Data agency LRT Jabodebek
agency_data = {
    'agency_id': ['KAI'],
    'agency_name': ['PT Kereta Api Indonesia (Persero) Divisi LRT Jabodebek'],
    'agency_url': ['https://lrtjabodebek.kai.id/'],
    'agency_timezone': ['Asia/Jakarta'],
    'agency_lang': ['id'],
    'agency_phone': ['021121'],
    'agency_fare_url': ['https://lrtjabodebek.kai.id/informasi-tarif'],
    'agency_email': ['cs@kai.id']
}

df_agency = pd.DataFrame(agency_data)

df_agency.to_csv(
    "agency.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

# Tampilkan hasil
print("✅ File agency.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_agency

✅ File agency.txt berhasil dibuat!

📄 Preview isi file:


,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,agency_fare_url,agency_email
0,KAI,PT Kereta Api Indonesia (Persero) Divisi LRT J...,https://lrtjabodebek.kai.id/,Asia/Jakarta,id,021121,https://lrtjabodebek.kai.id/informasi-tarif,cs@kai.id


## stops.txt (Conditionally Required)

File `stops.txt` berisi informasi tentang lokasi halte/stasiun tempat transportasi publik menaikkan atau menurunkan penumpang atau semua lokasi dalam jaringan transportasi mulai dari stasiun induk, peron, hingga pintu masuk/keluar. Untuk LRT Jabodebek, file ini mendeskripsikan 18 stasiun Lin Bekasi dan Lin Cibubur Fase 1.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `stop_id` | `stop_times.txt` | Foreign ID — setiap baris stop_times merujuk ke stop tempat kendaraan berhenti |
| `stop_id` | `transfers.txt` | Foreign ID — mendefinisikan aturan transfer antara dua stop |
| `stop_id` | `pathways.txt` | Foreign ID — mendefinisikan jalur fisik (tangga, lift, koridor) antar stop/entrance |
| `stop_id` | `levels.txt` | Foreign ID — menghubungkan lantai/level ke stop tertentu |
| `stop_id` | `locations.geojson` | Foreign ID — area geografis untuk on-demand service |
| `zone_id` | `fare_rules.txt` | Foreign ID — menentukan tarif berdasarkan zona asal dan tujuan |
| `level_id` | `levels.txt` | Foreign ID — merujuk ke level/lantai di dalam stasiun |
| `parent_station` | `stops.txt` *(self-referencing)* | Foreign ID — stop/platform/entrance merujuk ke stasiun induknya |

---

**Skema Kolom stops.txt**

| Nama Kolom | Tipe Data | status | Deskripsi | Contoh Value |
|------------|-----------|--------|-----------|--------------|
| `stop_id` | ID (unique) | **Wajib** | ID unik untuk mengidentifikasi halte/stasiun. | `SDKA` |
| `stop_code` | Text | Opsional | Kode singkat lokasi yang ditampilkan ke penumpang misalnya di papan informasi. | `BK01` |
| `stop_name` | Text | **Kondisional** | Nama lokasi sesuai yang tertera di jadwal, website, atau papan petunjuk resmi. **Wajib** untuk `location_type` 0, 1, dan 2. Opsional untuk tipe 3 dan 4. | `Jati Mulya` |
| `tts_stop_name` | Text | Opsional | Versi `stop_name` yang mudah dibaca oleh sistem text-to-speech (phonemisasi). | |
| `stop_desc` | Text | Opsional | Deskripsi lokasi yang informatif. Tidak boleh duplikasi dari `stop_name`. | `Stasiun terminus dan terintegrasi dengan halte bus` |
| `stop_lat` | Latitude | **Kondisional** | Koordinat lintang lokasi (WGS84). Untuk platform/halte, koordinat harus merujuk ke tiang halte atau tempat penumpang naik — bukan ke jalur kendaraan. **Wajib** untuk tipe 0, 1, dan 2. | `-6.2832` |
| `stop_lon` | Longitude | **Kondisional** | Koordinat bujur lokasi (WGS84). Aturan sama dengan `stop_lat`. **Wajib** untuk tipe 0, 1, dan 2. | `106.7394` |
| `zone_id` | ID | Opsional | ID zona tarif untuk halte. Diabaikan jika lokasi adalah stasiun atau pintu masuk stasiun. | `ZONE-JAKTIM` |
| `stop_url` | URL | Opsional | URL informasi spesifik halte/stasiun. | `https://lrtjabodebek.kai.id/stations/jati-mulya` |
| `location_type` | Enum | Opsional | Tipe lokasi: `0`=halte/platform, `1`=stasiun, `2`=pintu masuk, `3`=node, `4`=area boarding. Lebih detail lihat tabel jenis lokasi di bawah. | `1` |
| `parent_station` | Foreign ID → stops.stop_id | **Kondisional** | ID stasiun induk (untuk hierarki dalam stasiun) **Wajib** untuk tipe 2, 3, dan 4. Opsional untuk tipe 0. **Dilarang** diisi untuk tipe 1. | `SDKA` |
| `stop_timezone` | Timezone | Opsional | Zona waktu halte/stasiun (jika berbeda dari agency). | `Asia/Jakarta` |
| `wheelchair_boarding` | Enum | Opsional | Aksesibilitas kursi roda: `0`=tidak diketahui, `1`=tersedia, `2`=tidak tersedia. | `1` |
| `level_id` | Foreign ID → levels.level_id | Opsional | Referensi ke `levels.txt` untuk lantai dalam stasiun. | `LEVEL-G` |
| `platform_code` | Text | Opsional | Kode peron — hanya nomor/hurufnya saja, misalnya `1` atau `G`. Jangan sertakan kata "peron" atau "jalur". | `1` |
| `stop_access` | Enum | **Kondisional** | Cara akses ke platform dari luar. Lihat tabel nilai enum di bawah. **Dilarang** untuk tipe 1, 2, 3, dan 4, serta jika `parent_station` kosong. |

---

**Jenis Lokasi (`location_type`)**

| Nilai | Nama | Deskripsi |
|:---:|---|---|
| `0` atau kosong | Stop / Platform | Tempat penumpang naik/turun kendaraan. Disebut platform jika berada di dalam stasiun induk. |
| `1` | Station | Bangunan atau area fisik yang menampung satu atau lebih platform. |
| `2` | Entrance / Exit | Pintu masuk atau keluar stasiun dari jalan. Jika satu pintu terhubung ke beberapa stasiun, harus dipilih satu sebagai induk. |
| `3` | Generic Node | Lokasi dalam stasiun yang tidak termasuk tipe lain digunakan untuk menghubungkan jalur pejalan kaki (`pathways.txt`). |
| `4` | Boarding Area | Lokasi spesifik di dalam platform tempat penumpang naik/turun kendaraan. |

---

**Aksesibilitas Kursi Roda (`wheelchair_boarding`)**

| Konteks | Nilai | Deskripsi |
|---|:---:|---|
| **Stop tanpa induk** | `0` atau kosong | Tidak ada informasi aksesibilitas |
| | `1` | Beberapa kendaraan bisa diakses kursi roda |
| | `2` | Tidak bisa diakses kursi roda |
| **Stop anak (punya induk)** | `0` atau kosong | Mewarisi nilai dari stasiun induk (parent station) |
| | `1` | Ada jalur aksesibel dari luar stasiun ke platform ini |
| | `2` | Tidak ada jalur aksesibel ke platform ini |
| **Entrance / Exit** | `0` atau kosong | Mewarisi nilai dari stasiun induk (parent station) |
| | `1` | Pintu masuk aksesibel kursi roda |
| | `2` | Tidak ada jalur aksesibel dari pintu masuk ke platform |

---
**Akses Stop (`stop_access`)**

| Nilai | Deskripsi |
|:---:|---|
| `0` | Platform tidak bisa diakses langsung dari jalan — harus melalui pintu masuk stasiun atau jalur yang terdefinisi di `pathways.txt` |
| `1` | Aplikasi boleh memberi petunjuk langsung ke platform, tanpa melalui pintu masuk atau pathway |
| `kosong` | Akses tidak terdefinisi |

---


### Panduan Pengisian `stops.txt` — LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

---

**`Daftar Stasiun`**

`Lin Bekasi`

| No | Nama Stasiun | Short Name | Kode Stasiun |
|:---:|---|:---:|:---:|
| 1 | Dukuh Atas Bank Syariah Indonesia | DKA | BK01 |
| 2 | Setiabudi | SET | BK02 |
| 3 | Rasuna Said | RAS | BK03 |
| 4 | Kuningan | KUA | BK04 |
| 5 | Pancoran bank bjb | PAN | BK05 |
| 6 | Cikoko | CKK | BK06 |
| 7 | Ciliwung | CIL | BK07 |
| 8 | Cawang | CWG | BK08 |
| 9 | Halim | HAL | BK09 |
| 10 | Jati Bening Baru | JBU | BK10 |
| 11 | Cikunir 1 | CK1 | BK11 |
| 12 | Cikunir 2 | CK2 | BK12 |
| 13 | Bekasi Barat | BEK | BK13 |
| 14 | Jati Mulya | JTM | BK14 |

`Lin Cibubur`

| No | Nama Stasiun | Short Name | Kode Stasiun |
|:---:|---|:---:|:---:|
| 1 | Dukuh Atas Bank Syariah Indonesia | DKA | CB01 |
| 2 | Setiabudi | SET | CB02 |
| 3 | Rasuna Said | RAS | CB03 |
| 4 | Kuningan | KUA | CB04 |
| 5 | Pancoran bank bjb | PAN | CB05 |
| 6 | Cikoko | CKK | CB06 |
| 7 | Ciliwung | CIL | CB07 |
| 8 | Cawang | CWG | CB08 |
| 9 | TMII | TMI | CB09 |
| 10 | Kampung Rambutan | KAM | CB10 |
| 11 | Ciracas | CRC | CB11 |
| 12 | Harjamukti | HAR | CB12 |

---

**`stop_id`**

Format penamaan terdiri dari 3 jenis sesuai `location_type`:

| Jenis | Format | Contoh | Keterangan |
|---|---|---|---|
| Parent station | `S(kode)` | `SDKA`, `SSET`, `SRAS` | Satu per stasiun |
| Platform | `G(kode)(urut 2 digit)` | `GDKA01`, `GDKA02` | Urut mulai dari `01` per platform |
| Entrance | `E(kode)(urut 2 digit)` | `EDKA01`, `EDKA02` | Urut mulai dari `01`, sesuai label Akses A, B, C, … |

---

**`stop_name`**

Diisi dengan nama lengkap stasiun sesuai daftar resmi di atas, dengan prefiks **"Stasiun LRT"**. Untuk entrance ditambahkan sufiks label akses.

| Jenis | Format | Contoh |
|---|---|---|
| Parent station | `Stasiun LRT (nama stasiun)` | `Stasiun LRT Dukuh Atas Bank Syariah Indonesia` |
| Platform | `Stasiun LRT (nama stasiun)` | `Stasiun LRT Dukuh Atas Bank Syariah Indonesia` |
| Entrance | `Stasiun LRT (nama stasiun) Akses (label)` | `Stasiun LRT Dukuh Atas Bank Syariah Indonesia` |

---

**`location_type`**

`location_type` disesuaikan dengan kaidah dari dokumentasi GTFS, , untuk parent_stasion=1, untuk platform=0, dan untuk entrance=2.

| Jenis | Nilai | Contoh `stop_id` |
|---|:---:|---|
| Parent station | `1` | `SDKA`, `SSET`, `SRAS`, … |
| Platform | `0` | `GDKA01`, `GSET01`, … |
| Entrance | `2` | `EDKA01`, `ESET01`, … |

---

**`parent_station`**

Diisi dengan nilai `stop_id` dari stasiun induk dari setiap `location_type` dengan nilai 0 (platform) dan 2 (entrance) yaitu (SDKA, SSET, SRAS, SKUA, SPAN, SCKK, SCIL, SCWG, SHAL, SJBU, SCK1, SCK2, SBEK, SJTM, STMI, SKAM, SCRC, SHAR)

| Jenis | Isi | Contoh |
|---|---|---|
| Parent station | **Kosong** — dilarang diisi per spesifikasi GTFS | — |
| Platform | `stop_id` dari parent station stasiun bersangkutan | `GDKA01` → `SDKA` |
| Entrance | `stop_id` dari parent station stasiun bersangkutan | `EDKA01` → `SDKA` |

---

**`platform_code`**

Diisi hanya untuk baris **platform** (`location_type=0`). Cukup nomor urutnya saja tanpa kata "peron" atau "jalur".

| Contoh `stop_id` | Nilai `platform_code` |
|---|:---:|
| `GCWG01` | `1` |
| `GCWG02` | `2` |
| `GCWG03` | `3` |
| Parent station & entrance | *(kosong)* |

---

**`stop_lat & stop_lon`**

Koordinat diambil dari node OpenStreetMap menggunakan dua API berikut secara berurutan:

| Prioritas | API | Keterangan |
|:---:|---|---|
| 1 | **OSM API** — `api.openstreetmap.org` | Sumber utama |
| 2 | **Overpass API** — `overpass-api.de` | Fallback jika OSM API gagal |

Node OSM yang digunakan mencakup parent station, platform dan entrance, untuk lokasi stop yang belum memiliki node OSM, koordinat diisi secara manual. Berikut ini daftar node stops dari LRT Jabodebek yang tersedia di OpenStreetMap:

| Nama Stop | Tipe | Platform | Node OSM |
|---|---|:---:|---|
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia | Parent station | — | https://www.openstreetmap.org/node/8174072570 |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia | Platform | 1 | https://www.openstreetmap.org/node/11040189499 |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia | Platform | 2 | https://www.openstreetmap.org/node/11040189500 |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses A | Entrance | — | https://www.openstreetmap.org/node/13784553059 |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses B | Entrance | — | https://www.openstreetmap.org/node/13784634607 |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses C | Entrance | — | https://www.openstreetmap.org/node/ |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses D | Entrance | — | https://www.openstreetmap.org/node/ |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13784553057 |
| Stasiun LRT Dukuh Atas Bank Syariah Indonesia Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/13784634608 |
| Stasiun LRT Setiabudi | Parent station | — | https://www.openstreetmap.org/node/6720467135 |
| Stasiun LRT Setiabudi | Platform | 1 | https://www.openstreetmap.org/node/9124369263 |
| Stasiun LRT Setiabudi | Platform | 2 | https://www.openstreetmap.org/node/9124364689 |
| Stasiun LRT Setiabudi Akses A | Entrance | — | https://www.openstreetmap.org/node/11507279415 |
| Stasiun LRT Setiabudi Akses B | Entrance | — | https://www.openstreetmap.org/node/11804907916 |
| Stasiun LRT Setiabudi Akses C | Entrance | — | https://www.openstreetmap.org/node/11507279424 |
| Stasiun LRT Setiabudi Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13784454296 |
| Stasiun LRT Setiabudi Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/11844544368 |
| Stasiun LRT Rasuna Said | Parent station | — | https://www.openstreetmap.org/node/6720467136 |
| Stasiun LRT Rasuna Said | Platform | 1 | https://www.openstreetmap.org/node/11040189501 |
| Stasiun LRT Rasuna Said | Platform | 2 | https://www.openstreetmap.org/node/11040189502 |
| Stasiun LRT Rasuna Said Akses A | Entrance | — | https://www.openstreetmap.org/node/11516232038 |
| Stasiun LRT Rasuna Said Akses B | Entrance | — | https://www.openstreetmap.org/node/11516232036 |
| Stasiun LRT Rasuna Said Akses C1 | Entrance | — | https://www.openstreetmap.org/node/12031514934 |
| Stasiun LRT Rasuna Said Akses C2 | Entrance | — | https://www.openstreetmap.org/node/12031514936 |
| Stasiun LRT Rasuna Said Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/12031514940 |
| Stasiun LRT Rasuna Said Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/11785366792 |
| Stasiun LRT Kuningan | Parent station | — | https://www.openstreetmap.org/node/8174072567 |
| Stasiun LRT Kuningan | Platform | 1 | https://www.openstreetmap.org/node/11040189504 |
| Stasiun LRT Kuningan | Platform | 2 | https://www.openstreetmap.org/node/11040189503 |
| Stasiun LRT Kuningan Akses A | Entrance | — | https://www.openstreetmap.org/node/11529377125 |
| Stasiun LRT Kuningan Akses C1 | Entrance | — | https://www.openstreetmap.org/node/12031334773 |
| Stasiun LRT Kuningan Akses C2 | Entrance | — | https://www.openstreetmap.org/node/12031334771 |
| Stasiun LRT Kuningan Akses D | Entrance | — | https://www.openstreetmap.org/node/12031297565 |
| Stasiun LRT Kuningan Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/12031334808 |
| Stasiun LRT Kuningan Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/12031334805 |
| Stasiun LRT Pancoran bank bjb | Parent station | — | https://www.openstreetmap.org/node/6720467132 |
| Stasiun LRT Pancoran bank bjb | Platform | 1 | https://www.openstreetmap.org/node/11040204405 |
| Stasiun LRT Pancoran bank bjb | Platform | 2 | https://www.openstreetmap.org/node/11040204406 |
| Stasiun LRT Pancoran bank bjb Akses A | Entrance | — | https://www.openstreetmap.org/node/13784406397 |
| Stasiun LRT Pancoran bank bjb Akses B1 | Entrance | — | https://www.openstreetmap.org/node/12108411891 |
| Stasiun LRT Pancoran bank bjb Akses B2 | Entrance | — | https://www.openstreetmap.org/node/7098603621 |
| Stasiun LRT Pancoran bank bjb Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13784418901 |
| Stasiun LRT Cikoko | Parent station | — | https://www.openstreetmap.org/node/6720467131 |
| Stasiun LRT Cikoko | Platform | 1 | https://www.openstreetmap.org/node/11040204408 |
| Stasiun LRT Cikoko | Platform | 2 | https://www.openstreetmap.org/node/11040204407 |
| Stasiun LRT Cikoko Akses A | Entrance | — | https://www.openstreetmap.org/node/11365102450 |
| Stasiun LRT Cikoko Akses B | Entrance | — | https://www.openstreetmap.org/node/13784386630 |
| Stasiun LRT Cikoko Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/13784353781 |
| Stasiun LRT Ciliwung | Parent station | — | https://www.openstreetmap.org/node/8487603262 |
| Stasiun LRT Ciliwung | Platform | 1 | https://www.openstreetmap.org/node/9124369397 |
| Stasiun LRT Ciliwung | Platform | 2 | https://www.openstreetmap.org/node/9124369223 |
| Stasiun LRT Ciliwung Akses A | Entrance | — | https://www.openstreetmap.org/node/5639219805 |
| Stasiun LRT Ciliwung Akses C | Entrance | — | https://www.openstreetmap.org/node/13779321273 |
| Stasiun LRT Cawang | Parent station | — | https://www.openstreetmap.org/node/12260707174 |
| Stasiun LRT Cawang | Platform | 1 | https://www.openstreetmap.org/node/11040204410 |
| Stasiun LRT Cawang | Platform | 2 | https://www.openstreetmap.org/node/11040204409 |
| Stasiun LRT Cawang | Platform | 3 | https://www.openstreetmap.org/node/11040204411 |
| Stasiun LRT Cawang | Platform | 4 | https://www.openstreetmap.org/node/11040204412 |
| Stasiun LRT Cawang Akses A1 | Entrance | — | https://www.openstreetmap.org/node/13778631933 |
| Stasiun LRT Cawang Akses A2 | Entrance | — | https://www.openstreetmap.org/node/13778713696 |
| Stasiun LRT Cawang Akses B | Entrance | — | https://www.openstreetmap.org/node/13778687810 |
| Stasiun LRT Cawang Akses C | Entrance | — | https://www.openstreetmap.org/node/13778668058 |
| Stasiun LRT Cawang Akses D | Entrance | — | https://www.openstreetmap.org/node/13778982410 |
| Stasiun LRT Cawang Akses E | Entrance | — | https://www.openstreetmap.org/node/13778676174 |
| Stasiun LRT Cawang Akses F | Entrance | — | https://www.openstreetmap.org/node/ |
| Stasiun LRT Cawang Akses G | Entrance | — | https://www.openstreetmap.org/node/13778930391 |
| Stasiun LRT Cawang Akses H | Entrance | — | https://www.openstreetmap.org/node/13778684903 |
| Stasiun LRT Cawang Akses J | Entrance | — | https://www.openstreetmap.org/node/13778684902 |
| Stasiun LRT Cawang Akses K | Entrance | — | https://www.openstreetmap.org/node/13779063111 |
| Stasiun LRT Cawang Elevator Akses H | Entrance | — | https://www.openstreetmap.org/node/13778767928 |
| Stasiun LRT Cawang Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/13779161674 |
| Stasiun LRT Halim | Parent station | — | https://www.openstreetmap.org/node/9761865798 |
| Stasiun LRT Halim | Platform | 1 | https://www.openstreetmap.org/node/11040189497 |
| Stasiun LRT Halim | Platform | 2 | https://www.openstreetmap.org/node/11040189498 |
| Stasiun LRT Halim Akses A | Entrance | — | https://www.openstreetmap.org/node/9761865809 |
| Stasiun LRT Jati Bening Baru | Parent station | — | https://www.openstreetmap.org/node/8174072566 |
| Stasiun LRT Jati Bening Baru | Platform | 1 | https://www.openstreetmap.org/node/11040189495 |
| Stasiun LRT Jati Bening Baru | Platform | 2 | https://www.openstreetmap.org/node/12578386110 |
| Stasiun LRT Jati Bening Baru Akses A | Entrance | — | https://www.openstreetmap.org/node/11743373671 |
| Stasiun LRT Jati Bening Baru Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13778684254 |
| Stasiun LRT Cikunir 1 | Parent station | — | https://www.openstreetmap.org/node/8174072565 |
| Stasiun LRT Cikunir 1 | Platform | 1 | https://www.openstreetmap.org/node/11040189493 |
| Stasiun LRT Cikunir 1 | Platform | 2 | https://www.openstreetmap.org/node/11040189494 |
| Stasiun LRT Cikunir 1 Akses A | Entrance | — | https://www.openstreetmap.org/node/11382851957 |
| Stasiun LRT Cikunir 1 Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13778642095 |
| Stasiun LRT Cikunir 2 | Parent station | — | https://www.openstreetmap.org/node/8174072564 |
| Stasiun LRT Cikunir 2 | Platform | 1 | https://www.openstreetmap.org/node/11040189491 |
| Stasiun LRT Cikunir 2 | Platform | 2 | https://www.openstreetmap.org/node/11040189492 |
| Stasiun LRT Cikunir 2 Akses A | Entrance | — | https://www.openstreetmap.org/node/11583698331 |
| Stasiun LRT Cikunir 2 Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13778630202 |
| Stasiun LRT Bekasi Barat | Parent station | — | https://www.openstreetmap.org/node/8174072563 |
| Stasiun LRT Bekasi Barat | Platform | 1 | https://www.openstreetmap.org/node/4909917394 |
| Stasiun LRT Bekasi Barat | Platform | 2 | https://www.openstreetmap.org/node/4982025827 |
| Stasiun LRT Bekasi Barat Akses A | Entrance | — | https://www.openstreetmap.org/node/12264011246 |
| Stasiun LRT Bekasi Barat Akses B | Entrance | — | https://www.openstreetmap.org/node/13778592814 |
| Stasiun LRT Bekasi Barat Akses C | Entrance | — | https://www.openstreetmap.org/node/11415068296 |
| Stasiun LRT Bekasi Barat Elevator Akses A-B | Entrance | — | https://www.openstreetmap.org/node/13778592813 |
| Stasiun LRT Jati Mulya | Parent station | — | https://www.openstreetmap.org/node/7616402144 |
| Stasiun LRT Jati Mulya | Platform | 1 | https://www.openstreetmap.org/node/11040189489 |
| Stasiun LRT Jati Mulya | Platform | 2 | https://www.openstreetmap.org/node/11040189490 |
| Stasiun LRT Jati Mulya Akses A | Entrance | — | https://www.openstreetmap.org/node/13778581063 |
| Stasiun LRT Jati Mulya Akses B | Entrance | — | https://www.openstreetmap.org/node/13778581062 |
| Stasiun LRT Jati Mulya Elevator Akses A-B | Entrance | — | https://www.openstreetmap.org/node/13778581067 |
| Stasiun LRT TMII | Parent station | — | https://www.openstreetmap.org/node/4907843137 |
| Stasiun LRT TMII | Platform | 1 | https://www.openstreetmap.org/node/11040189488 |
| Stasiun LRT TMII | Platform | 2 | https://www.openstreetmap.org/node/11040189487 |
| Stasiun LRT TMII Akses A | Entrance | — | https://www.openstreetmap.org/node/13767166161 |
| Stasiun LRT Kampung Rambutan | Parent station | — | https://www.openstreetmap.org/node/6720467137 |
| Stasiun LRT Kampung Rambutan | Platform | 1 | https://www.openstreetmap.org/node/11040189486 |
| Stasiun LRT Kampung Rambutan | Platform | 2 | https://www.openstreetmap.org/node/11040189485 |
| Stasiun LRT Kampung Rambutan Akses A | Entrance | — | https://www.openstreetmap.org/node/11576541975 |
| Stasiun LRT Kampung Rambutan Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/13766839045 |
| Stasiun LRT Ciracas | Parent station | — | https://www.openstreetmap.org/node/8174072568 |
| Stasiun LRT Ciracas | Platform | 1 | https://www.openstreetmap.org/node/11040189470 |
| Stasiun LRT Ciracas | Platform | 2 | https://www.openstreetmap.org/node/12578386108 |
| Stasiun LRT Ciracas Akses A | Entrance | — | https://www.openstreetmap.org/node/11453116133 |
| Stasiun LRT Ciracas Akses B | Entrance | — | https://www.openstreetmap.org/node/11453116136 |
| Stasiun LRT Ciracas Elevator Akses A | Entrance | — | https://www.openstreetmap.org/node/11453116129 |
| Stasiun LRT Harjamukti | Parent station | — | https://www.openstreetmap.org/node/6720467138 |
| Stasiun LRT Harjamukti | Platform | 1 | https://www.openstreetmap.org/node/11040189468 |
| Stasiun LRT Harjamukti | Platform | 2 | https://www.openstreetmap.org/node/11040189467 |
| Stasiun LRT Harjamukti Akses A | Entrance | — | https://www.openstreetmap.org/node/13752213754 |
| Stasiun LRT Harjamukti Akses B | Entrance | — | https://www.openstreetmap.org/node/13752225295 |
| Stasiun LRT Harjamukti Elevator Akses B | Entrance | — | https://www.openstreetmap.org/node/13752241957 |

---

**`wheelchair_boarding`**

Diisi berdasarkan informasi aksesibilitas dari website resmi LRT Jabodebek. Nilai yang berlaku:

| Nilai | Arti | Berlaku untuk |
|:---:|---|---|
| `1` | Aksesibel kursi roda | Parent station yang memiliki fasilitas aksesibel |
| `0` | Mewarisi nilai dari parent station | Platform & entrance (jika mengikuti induknya) |
| `2` | Tidak aksesibel kursi roda | Jika ada platform/entrance yang tidak aksesibel |

In [2]:
# Import Library
import requests
import csv
import time

# Konfigurasi API OpenStreetMap
OSM_API      = "https://api.openstreetmap.org/api/0.6/node/{node_id}.json"
OVERPASS_API = "https://overpass-api.de/api/interpreter"
HEADERS      = {"User-Agent": "gtfs-stops-generator/1.0"}

_cache = {}  # simpan hasil fetch agar tidak request ulang

In [3]:
# Function request titik koordinat (stop_lat & stop_lon) OSM API
def get_osm_coords(label, node_id):
    """Fetch (lat, lon) dari OSM node ID. Fallback ke Overpass jika OSM gagal."""
    if node_id is None:
        return None, None
    if node_id in _cache:
        return _cache[node_id]

    # Request OSM API
    try:
        url  = OSM_API.format(node_id=node_id)
        resp = requests.get(url, timeout=10, headers=HEADERS)
        resp.raise_for_status()
        elem = resp.json()["elements"][0]
        lat, lon = elem["lat"], elem["lon"]
        _cache[node_id] = (lat, lon)
        print(f"  ✓ [OSM] {label} (node_id={node_id}) → ({lat}, {lon})")
        time.sleep(0.2)
        return lat, lon
    except Exception as e:
        print(f"  … OSM gagal ({e}), coba Overpass …")

    # Request Overpass API
    try:
        query = f"[out:json];node({node_id});out;"
        resp  = requests.post(OVERPASS_API, data={"data": query},
                              timeout=15, headers=HEADERS)
        resp.raise_for_status()
        elem  = resp.json()["elements"][0]
        lat, lon = elem["lat"], elem["lon"]
        _cache[node_id] = (lat, lon)
        print(f"  ✓ [Overpass] {label} {node_id} → ({lat}, {lon})")
        time.sleep(0.5)
        return lat, lon
    except Exception as e:
        print(f"  ✗ Overpass juga gagal: {e}")

    _cache[node_id] = (None, None)
    return None, None

In [4]:
# Pengisian Data disesuaikan dengan field panduan diatas 

STATION_DATA = [
    {
        "stop_id_prefix": "DKA", #Kode Stasiun
        "stop_name"     : "Stasiun LRT Dukuh Atas Bank Syariah Indonesia", # stop_name
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/dukuh-atas", #URL Stasiun
        "stop_code"     : "BK01/CB01", # stop_code biasanya tertampil di peta informasi
        "parent_wc"     : 1, # wheelchair_boarding parent station
        "parent_node_id": 8174072570, # node-id parent stasiun
        "platforms": [  
            #(platform_code dan node-id platform stasiun)
            ("1", 11040189499), # platform_code dan node-id platform stasiun
            ("2", 11040189500),
        ],
        # (label, osm_node_id) — isi None jika belum ada node OSM
        "entrances": [
            #(nama entrance, node-id entrance stasiun, wheelchair_boarding entrance)
            ("Akses A", 13784553059, 2),
            ("Akses B", 13784634607, 2),
            ("Akses C1", None, 2, -6.205195774034116, 106.82652890878188),
            ("Akses C2", None, 2, -6.205046450328923, 106.82578634921708),
            # ("Akses C1", , 2),
            # ("Akses C2", , 2),
            ("Elevator Akses A", 13784553057, 1),
            ("Elevator Akses B", 13784634608, 1),
        ],
    },
    {
        "stop_id_prefix": "SET",
        "stop_name"     : "Stasiun LRT Setiabudi",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/setiabudi",
        "stop_code"     : "BK02/CB02",
        "parent_wc"     : 1,
        "parent_node_id": 6720467135,
        "platforms": [
            ("1", 9124369263),
            ("2", 9124364689),
        ],
        "entrances": [
            ("Akses A", 11507279415, 2), 
            ("Akses B", 11804907916, 2), 
            ("Akses C", 11507279424, 2), 
            ("Elevator Akses A", 13784454296, 1),
            ("Elevator Akses B", 11844544368, 1)
        ],
    },
    {
        "stop_id_prefix": "RAS",
        "stop_name"     : "Stasiun LRT Rasuna Said",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/rasuna-said",
        "stop_code"     : "BK03/CB03",
        "parent_wc"     : 1,
        "parent_node_id": 6720467136,
        "platforms": [
            ("1", 11040189501),
            ("2", 11040189502),
        ],
        "entrances": [
            ("Akses A", 11516232038, 2), 
            ("Akses B", 11516232036, 2), 
            ("Akses C1", 12031514934, 2),
            ("Akses C2", 12031514936, 2), 
            ("Elevator Akses A", 12031514940, 1),
            ("Elevator Akses B", 11785366792, 1)
        ],
    },
    {
        "stop_id_prefix": "KUA",
        "stop_name"     : "Stasiun LRT Kuningan",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/kuningan",
        "stop_code"     : "BK04/CB04",
        "parent_wc"     : 1,
        "parent_node_id": 8174072567,
        "platforms": [
            ("1", 11040189504),
            ("2", 11040189503),
        ],
        "entrances": [
            ("Akses A", 11529377125, 2), 
            ("Akses C1", 12031334773, 2), 
            ("Akses C2", 12031334771, 2), 
            ("Akses D", 12031297565, 2), 
            ("Elevator Akses A", 12031334808, 1),
            ("Elevator Akses B", 12031334805, 1)
        ],
    },
    {
        "stop_id_prefix": "PAN",
        "stop_name"     : "Stasiun LRT Pancoran bank bjb",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/pancoran",
        "stop_code"     : "BK05/CB05",
        "parent_wc"     : 1,
        "parent_node_id": 6720467132,
        "platforms": [
            ("1", 11040204405),
            ("2", 11040204406),
        ],
        "entrances": [
            ("Akses A", 13784406397, 2), 
            ("Akses B1", 12108411891, 2), 
            ("Akses B2", 7098603621, 2), 
            ("Elevator Akses A", 13784418901, 1),
        ],
    },
    {
        "stop_id_prefix": "CKK",
        "stop_name"     : "Stasiun LRT Cikoko",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/cikoko",
        "stop_code"     : "BK06/CB06",
        "parent_wc"     : 1,
        "parent_node_id": 6720467131,
        "platforms": [
            ("1", 11040204408),
            ("2", 11040204407),
        ],
        "entrances": [
            ("Akses A", 11365102450, 2),
            ("Akses B", 13784386630, 2), 
            ("Elevator Akses B", 13784353781, 1),
        ],
    },
    {
        "stop_id_prefix": "CIL",
        "stop_name"     : "Stasiun LRT Ciliwung",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/ciliwung",
        "stop_code"     : "BK07/CB07",
        "parent_wc"     : 1,
        "parent_node_id": 8487603262,
        "platforms": [
            ("1", 9124369397),
            ("2", 9124369223),
        ],
        "entrances": [
            ("Akses A", 5639219805, 2),
            ("Akses C", 13779321273, 2)
        ],
    },
    {
        "stop_id_prefix": "CWG",
        "stop_name"     : "Stasiun LRT Cawang",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/cawang",
        "stop_code"     : "BK08/CB08",
        "parent_wc"     : 1,
        "parent_node_id": 12260707174,
        "platforms": [
            ("1", 11040204410),
            ("2", 11040204409),
            ("3", 11040204411),
            ("4", 11040204412)
        ],
        "entrances": [
            ("Akses A1", 13778631933, 2),
            ("Akses A2", 13778713696, 2),
            ("Akses B", 13778687810, 2),
            ("Akses C", 13778668058, 2), 
            ("Akses D", 13778982410, 2),
            ("Akses E", 13778676174, 2), 
            ("Akses F", None, 2, -6.246114095313879, 106.87210715933598),
            # ("Akses F", , 2),
            ("Akses G", 13778930391, 2), 
            ("Akses H", 13778684903, 2),
            ("Akses J", 13778684902, 2), 
            ("Akses K", 13779063111, 2),
            ("Elevator Akses B", 13779161674, 1),
            ("Elevator Akses H", 13778767928, 1),
        ],
    },
    {
        "stop_id_prefix": "HAL",
        "stop_name"     : "Stasiun LRT Halim",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/halim",
        "stop_code"     : "BK09",
        "parent_wc"     : 1,
        "parent_node_id": 9761865798,
        "platforms": [
            ("1", 11040189497),
            ("2", 11040189498),
        ],
        "entrances": [
            ("Akses A", 9761865809, 2)
        ],
    },
    {
        "stop_id_prefix": "JBU",
        "stop_name"     : "Stasiun LRT Jati Bening Baru",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/jati-bening-baru",
        "stop_code"     : "BK10",
        "parent_wc"     : 1,
        "parent_node_id": 8174072566,
        "platforms": [
            ("1", 11040189495),
            ("2", 12578386110),
        ],
        "entrances": [
            ("Akses A", 11743373671, 2),
            ("Elevator Akses A", 13778684254, 1),
        ],
    },
    {
        "stop_id_prefix": "CK1",
        "stop_name"     : "Stasiun LRT Cikunir 1",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/cikunir-1",
        "stop_code"     : "BK11",
        "parent_wc"     : 1,
        "parent_node_id": 8174072565,
        "platforms": [
            ("1", 11040189493),
            ("2", 11040189494),
        ],
        "entrances": [
            ("Akses A", 11382851957, 2), 
            ("Elevator Akses A", 13778642095, 1),
        ],
    },
    {
        "stop_id_prefix": "CK2",
        "stop_name"     : "Stasiun LRT Cikunir 2",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/cikunir-2",
        "stop_code"     : "BK12",
        "parent_wc"     : 1,
        "parent_node_id": 8174072564,
        "platforms": [
            ("1", 11040189491),
            ("2", 11040189492),
        ],
        "entrances": [
            ("Akses A", 11583698331, 2),
            ("Elevator Akses A", 13778630202, 1),
        ],
    },
    {
        "stop_id_prefix": "BEK",
        "stop_name"     : "Stasiun LRT Bekasi Barat",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/bekasi-barat",
        "stop_code"     : "BK13",
        "parent_wc"     : 1,
        "parent_node_id": 8174072563,
        "platforms": [
            ("1", 4909917394),
            ("2", 4982025827),
        ],
        "entrances": [
            ("Akses A", 12264011246, 2),
            ("Akses B", 13778592814, 2), 
            ("Akses C", 11415068296, 2), 
            ("Elevator Akses A-B", 13778592813, 1),
        ],
    },
    {
        "stop_id_prefix": "JTM",
        "stop_name"     : "Stasiun LRT Jati Mulya",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/jati-mulya",
        "stop_code"     : "BK14",
        "parent_wc"     : 1,
        "parent_node_id": 7616402144,
        "platforms": [
            ("1", 11040189489),
            ("2", 11040189490),
        ],
        "entrances": [
            ("Akses A", 13778581063, 2),
            ("Akses B", 13778581062, 2), 
            ("Elevator Akses A-B", 13778581067, 1),
        ],
    },
    {
        "stop_id_prefix": "TMI",
        "stop_name"     : "Stasiun LRT TMII",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/taman-mini",
        "stop_code"     : "CB09",
        "parent_wc"     : 1,
        "parent_node_id": 4907843137,
        "platforms": [
            ("1", 11040189488),
            ("2", 11040189487),
        ],
        "entrances": [
            ("Akses A", 13767166161, 2)
        ],
    },
    {
        "stop_id_prefix": "KAM",
        "stop_name"     : "Stasiun LRT Kampung Rambutan",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/kampung-rambutan",
        "stop_code"     : "CB10",
        "parent_wc"     : 1,
        "parent_node_id": 6720467137,
        "platforms": [
            ("1", 11040189486),
            ("2", 11040189485),
        ],
        "entrances": [
            ("Akses A", 11576541975, 2),
            ("Elevator Akses A", 13766839045, 1),
        ],
    },
    {
        "stop_id_prefix": "CRC",
        "stop_name"     : "Stasiun LRT Ciracas",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/ciracas",
        "stop_code"     : "CB11",
        "parent_wc"     : 1,
        "parent_node_id": 8174072568,
        "platforms": [
            ("1", 11040189470),
            ("2", 12578386108),
        ],
        "entrances": [
            ("Akses A", 11453116133, 2),
            ("Akses B", 11453116136, 2), 
            ("Elevator Akses A", 11453116129, 1),
        ],
    },
    {
        "stop_id_prefix": "HAR",
        "stop_name"     : "Stasiun LRT Harjamukti",
        "stop_url"      : "https://lrtjabodebek.kai.id/stations/harjamukti",
        "stop_code"     : "CB12",
        "parent_wc"     : 1,
        "parent_node_id": 6720467138,
        "platforms": [
            ("1", 11040189468),
            ("2", 11040189467),
        ],
        "entrances": [
            ("Akses A", 13752213754, 2),
            ("Akses B", 13752225295, 2), 
            ("Elevator Akses B", 13752241957, 1),
        ],
    },
]

print(f"✅ {len(STATION_DATA)} stasiun terdefinisi.")

✅ 18 stasiun terdefinisi.


In [5]:
FIELDNAMES = [
    "stop_id", "stop_code", "stop_name", "stop_url", "stop_lat", "stop_lon",
    "zone_id","location_type", "parent_station", "platform_code",
    "wheelchair_boarding"
]

def build_rows():
    rows = []
    for st in STATION_DATA:
        pfx       = st["stop_id_prefix"]
        name      = st["stop_name"]
        parent_id = f"S{pfx}"

        print(f"\n{'─'*50}")
        print(f"📍 {name}")

        # Parent station (location_type = 1)
        print(f"   Parent node: {st['parent_node_id']}")
        lat, lon = get_osm_coords(name, st["parent_node_id"])
        rows.append({
            "stop_id"            : parent_id,
            "stop_code"          : st["stop_code"],
            "stop_name"          : name,
            "stop_url"           : st["stop_url"],
            "stop_lat"           : lat if lat is not None else "",
            "stop_lon"           : lon if lon is not None else "",
            "zone_id"            : pfx,
            "location_type"      : 1,
            "parent_station"     : "",
            "platform_code"      : "",
            "wheelchair_boarding": st["parent_wc"]
        })

        # Platform (location_type = 0)
        for idx, (pcode, node_id) in enumerate(st["platforms"], start=1):
            num = str(idx).zfill(2)
            lat, lon = get_osm_coords(f"{name} Platform {pcode}", node_id)
            rows.append({
                "stop_id"            : f"G{pfx}{num}",
                "stop_code"          : "",
                "stop_name"          : name,
                "stop_url"           : "",
                "stop_lat"           : lat if lat is not None else "",
                "stop_lon"           : lon if lon is not None else "",
                "zone_id"            : pfx,
                "location_type"      : 0,
                "parent_station"     : parent_id,
                "platform_code"      : pcode,
                "wheelchair_boarding": 0
            })

        # Entrance (location_type = 2)
        for idx, entrance in enumerate(st["entrances"], start=1):
            num   = str(idx).zfill(2)
            label = entrance[0]
            node_id    = entrance[1]
            wheelchair = entrance[2]
        
            # ── Cek apakah ada koordinat manual ──────────────────────────
            if node_id is None and len(entrance) == 5:
                # Format: (label, None, wheelchair, lat, lon)
                lat, lon = entrance[3], entrance[4]
                print(f"  ✓ [Manual] {name} {label} → koordinat manual ({lat}, {lon})")
            else:
                # Format: (label, node_id, wheelchair) → ambil dari OSM
                lat, lon = get_osm_coords(f"{name} {label}", node_id)
        
            rows.append({
                "stop_id"            : f"E{pfx}{num}",
                "stop_code"          : "",
                "stop_name"          : f"{name} {label}",
                "stop_url"           : "",
                "stop_lat"           : lat if lat is not None else "",
                "stop_lon"           : lon if lon is not None else "",
                "zone_id"            : pfx,
                "location_type"      : 2,
                "parent_station"     : parent_id,
                "platform_code"      : "",
                "wheelchair_boarding": wheelchair
            })

        # Entrance (location_type = 2)
        # for idx, (label, node_id, wheelchair) in enumerate(st["entrances"], start=1):
        #     num = str(idx).zfill(2)
        #     lat, lon = get_osm_coords(f"{name} {label}", node_id)
        #     rows.append({
        #         "stop_id"            : f"E{pfx}{num}",
        #         "stop_code"          : "",
        #         "stop_name"          : f"{name} {label}",
        #         "stop_url"           : "",
        #         "stop_lat"           : lat if lat is not None else "",
        #         "stop_lon"           : lon if lon is not None else "",
        #         "zone_id"            : pfx,
        #         "location_type"      : 2,
        #         "parent_station"     : parent_id,
        #         "platform_code"      : "",
        #         "wheelchair_boarding": wheelchair
        #     })

    return rows

print("✅ Fungsi build_rows() siap.")

✅ Fungsi build_rows() siap.


In [6]:
_cache.clear()  # reset cache jika cell dijalankan ulang
all_rows = build_rows()

print(f"\n{'='*50}")
print(f"✅ Total baris dihasilkan: {len(all_rows)}")


──────────────────────────────────────────────────
📍 Stasiun LRT Dukuh Atas Bank Syariah Indonesia
   Parent node: 8174072570
  ✓ [OSM] Stasiun LRT Dukuh Atas Bank Syariah Indonesia (node_id=8174072570) → (-6.204828, 106.8255301)
  ✓ [OSM] Stasiun LRT Dukuh Atas Bank Syariah Indonesia Platform 1 (node_id=11040189499) → (-6.2048511, 106.8255247)
  ✓ [OSM] Stasiun LRT Dukuh Atas Bank Syariah Indonesia Platform 2 (node_id=11040189500) → (-6.2048049, 106.8255355)
  ✓ [OSM] Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses A (node_id=13784553059) → (-6.204904, 106.8253762)
  ✓ [OSM] Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses B (node_id=13784634607) → (-6.2046245, 106.8250622)
  ✓ [Manual] Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses C1 → koordinat manual (-6.205195774034116, 106.82652890878188)
  ✓ [Manual] Stasiun LRT Dukuh Atas Bank Syariah Indonesia Akses C2 → koordinat manual (-6.205046450328923, 106.82578634921708)
  ✓ [OSM] Stasiun LRT Dukuh Atas Bank Syariah Indone

In [7]:
import pandas as pd

df_stops = pd.DataFrame(all_rows, columns=FIELDNAMES)
df_stops

,stop_id,stop_code,stop_name,stop_url,stop_lat,stop_lon,zone_id,location_type,parent_station,platform_code,wheelchair_boarding
0,SDKA,BK01/CB01,Stasiun LRT Dukuh Atas Bank Syariah Indonesia,https://lrtjabodebek.kai.id/stations/dukuh-atas,-6.204828,106.825530,DKA,1,,,1
1,GDKA01,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia,,-6.204851,106.825525,DKA,0,SDKA,1,0
2,GDKA02,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia,,-6.204805,106.825536,DKA,0,SDKA,2,0
3,EDKA01,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.204904,106.825376,DKA,2,SDKA,,2
4,EDKA02,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.204625,106.825062,DKA,2,SDKA,,2
...,...,...,...,...,...,...,...,...,...,...,...
119,GHAR01,,Stasiun LRT Harjamukti,,-6.373896,106.895652,HAR,0,SHAR,1,0
120,GHAR02,,Stasiun LRT Harjamukti,,-6.373890,106.895686,HAR,0,SHAR,2,0
121,EHAR01,,Stasiun LRT Harjamukti Akses A,,-6.373657,106.895475,HAR,2,SHAR,,2
122,EHAR02,,Stasiun LRT Harjamukti Akses B,,-6.374178,106.895575,HAR,2,SHAR,,2


In [8]:
# Validasi titik koordinat setiap stops dengan build graph folium
import pandas as pd
import folium

# pastikan lat/lon numeric
df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")

# drop yang invalid
df_stops = df_stops.dropna(subset=["stop_lat", "stop_lon"])

# center map (pakai rata-rata koordinat)
center_lat = df_stops["stop_lat"].mean()
center_lon = df_stops["stop_lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# plot semua stop
for _, row in df_stops.iterrows():
    folium.CircleMarker(
        location=[row["stop_lat"], row["stop_lon"]],
        radius=4,
        popup=f"""
        ID: {row['stop_id']}<br>
        Name: {row['stop_name']}<br>
        Type: {row.get('location_type', '')}<br>
        Parent: {row.get('parent_station', '')}
        """,
        fill=True,
    ).add_to(m)

# simpan ke html
# m.save("stops_map.html")
m

In [9]:
df_stops.to_csv(
    "stops.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

In [10]:
df_stops = pd.read_csv(
    "stops.txt",
    dtype={
        "stop_id"             : str,
        "stop_code"           : str,
        "stop_name"           : str,
        "stop_url"            : str,
        "stop_lat"            : float,
        "stop_lon"            : float,
        "zone_id"             : str,
        "location_type"       : "Int64",
        "parent_station"      : str,
        "platform_code"       : str,
        "wheelchair_boarding" : "Int64"
    },
    keep_default_na=False
)
print(f"Shape: {df_stops.shape[0]} baris × {df_stops.shape[1]} kolom")
df_stops.head(50)

Shape: 124 baris × 11 kolom


,stop_id,stop_code,stop_name,stop_url,stop_lat,stop_lon,zone_id,location_type,parent_station,platform_code,wheelchair_boarding
0,SDKA,BK01/CB01,Stasiun LRT Dukuh Atas Bank Syariah Indonesia,https://lrtjabodebek.kai.id/stations/dukuh-atas,-6.204828,106.825530,DKA,1,,,1
1,GDKA01,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia,,-6.204851,106.825525,DKA,0,SDKA,1,0
2,GDKA02,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia,,-6.204805,106.825536,DKA,0,SDKA,2,0
3,EDKA01,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.204904,106.825376,DKA,2,SDKA,,2
4,EDKA02,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.204625,106.825062,DKA,2,SDKA,,2
5,EDKA03,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.205196,106.826529,DKA,2,SDKA,,2
6,EDKA04,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.205046,106.825786,DKA,2,SDKA,,2
7,EDKA05,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.204831,106.824944,DKA,2,SDKA,,1
8,EDKA06,,Stasiun LRT Dukuh Atas Bank Syariah Indonesia ...,,-6.204566,106.825083,DKA,2,SDKA,,1
9,SSET,BK02/CB02,Stasiun LRT Setiabudi,https://lrtjabodebek.kai.id/stations/setiabudi,-6.209318,106.830221,SET,1,,,1


## routes.txt (Required)

File `routes.txt` berisi informasi tentang identitas jalur atau rute layanan transportasi tersebut seperti nama rute, operator, jenis moda yang digunakan, warna identitas rute. File ini menjadi penghubung utama antara identitas layanan dengan jadwal perjalanan aktual.

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `route_id` | `trips.txt` | Foreign ID — setiap trip merujuk ke rute induknya |
| `route_id` | `fare_rules.txt` | Foreign ID — tarif dikaitkan ke rute tertentu |
| `route_id` | `attributions.txt` | Foreign ID — atribusi data per rute |
| `agency_id` | `agency.txt` | Foreign ID — merujuk ke operator |

---

**Skema kolom routes.txt**

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `route_id` | Unique ID | **Wajib** | ID unik pengenal rute. Direferensikan di `trips.txt` dan `fare_rules.txt`. |
| `agency_id` | Foreign ID → `agency.agency_id` | **Kondisional** | ID operator rute. **Wajib** jika dataset berisi lebih dari satu operator. |
| `route_short_name` | Text | **Kondisional** | Nama singkat rute yang dikenal penumpang, maksimal 12 karakter. Contoh: `BK`, `32`, `CB`. **Wajib** jika `route_long_name` kosong. |
| `route_long_name` | Text | **Kondisional** | Nama lengkap rute, biasanya mencantumkan tujuan atau deskripsi jalur. **Wajib** jika `route_short_name` kosong. |
| `route_desc` | Text | Opsional | Deskripsi informatif rute. Tidak boleh duplikasi dari `route_short_name` atau `route_long_name`. |
| `route_type` | Enum | **Wajib** | Jenis moda transportasi. Lihat tabel jenis rute di bawah. |
| `route_url` | URL | Opsional | URL halaman web khusus rute ini. Harus berbeda dari `agency_url`. |
| `route_color` | Color | Opsional | Warna latar rute dalam format hex 6 digit tanpa `#`. Default putih (`FFFFFF`). |
| `route_text_color` | Color | Opsional | Warna teks di atas `route_color` dalam format hex 6 digit tanpa `#`. Default hitam (`000000`). Harus kontras dengan `route_color`. |
| `route_sort_order` | Non-negative integer | Opsional | Urutan tampilan rute — nilai lebih kecil ditampilkan lebih dulu. |
| `continuous_pickup` | Enum | **Kondisional** | Apakah penumpang bisa naik di sembarang titik sepanjang rute. Lihat tabel nilai enum di bawah. **Dilarang** jika `stop_times.start_pickup_drop_off_window` atau `end_pickup_drop_off_window` terdefinisi. |
| `continuous_drop_off` | Enum | **Kondisional** | Apakah penumpang bisa turun di sembarang titik sepanjang rute. Aturan sama dengan `continuous_pickup`. |
| `network_id` | ID | **Kondisional** | ID grup rute — beberapa rute bisa punya `network_id` yang sama. **Dilarang** jika file `route_networks.txt` atau `networks.txt` ada. |
| `cemv_support` | Enum | Opsional | Dukungan pembayaran contactless EMV (kartu/perangkat Visa, Mastercard, dst) untuk rute ini. Jika diisi bersamaan dengan `agency.cemv_support`, nilai di sini yang **lebih diprioritaskan**. |

---

**Jenis Rute (`route_type`)**

| Nilai | Jenis | 
|:---:|---|
| `0` | Tram / Light Rail | 
| `1` | Subway / Metro |
| `2` | Kereta jarak jauh |
| `3` | Bus | 
| `4` | Ferry |
| `5` | Cable tram |
| `6` | Aerial lift / Gondola | 
| `7` | Funicular |
| `11` | Trolleybus |
| `12` | Monorail | 

---

**Nilai `continuous_pickup` & `continuous_drop_off`**

| Nilai | Arti |
|:---:|---|
| `0` | Bisa naik/turun di sembarang titik sepanjang rute |
| `1` atau kosong | Tidak bisa — hanya di halte/stasiun yang ditentukan |
| `2` | Perlu telepon operator terlebih dahulu |
| `3` | Perlu koordinasi langsung dengan pengemudi |

### Panduan Pengisian `routes.txt` — LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

---

**`route_id`**

Diisi sesuai dengan singkatan dari LRT Jabodebek Lin Bekasi dan Lin Cibubur yaitu: `BK` dan `CB` 

---

**`agency_id`**

Disesuaikan dengan `agency_id` yang direferensikan dari file agency.txt yaitu `KAI`

---

**`route_short_name`**

Diisi dengan singkatan dari Lin Bekasi dan Lin Cibubur yaitu `BK` dan `CB`

---

**`route_long_name`**

Diisi dengan `Lin Bekasi` dan `Lin Cibubur`

---

**`route_desc`**

Diisi dengan deskripsi singkat yaitu: `LRT Jabodebek Lin Bekasi (Dukuh Atas-Jati Mulya)` dan `LRT Jabodebek Lin Cibubur (Dukuh Atas-Harjamukti)`

---

**`route_type`**

Diisi dengan nilai `0` karena jenis moda dari LRT Jabodebek adalah Light Rail

---

**`route_color`**

Diisi dengan nilai kode warna rute untuk LRT Jabodebek Lin Bekasi dan Lin Cibubur sesuai dengan peta resmi: [TransportForJakarta](https://transportforjakarta.or.id/lrtjabodebek/) untuk Lin Bekasi yaitu kode warna Green: `0E6938` dan untuk Lin Cibubur yaitu kode warna Blue `20409A`

**`route_text_color`**

Diisi dengan nilai warna teks di atas route_color, disesuaikan dengan Logo LRT Jabodebek Lin Bekasi dan Lin Cibubur [Logo](https://transportforjakarta.or.id/lrtjabodebek/) warna nya putih: `FFFFFF`

**`network_id`**

Diisi dengan `LRTJAB`, field ini nanti yang akan digunakan `fare_leg_rules.txt` sebagai ID jaringan transportasi LRT Jabodebek ini.

In [11]:
import pandas as pd
import csv

# Data routes LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1
routes_data = {
    'route_id'         : ['BK', 'CB'],
    'agency_id'        : ['KAI', 'KAI'],
    'route_short_name' : ['BK', 'CB'],
    'route_long_name'  : ['Lin Bekasi', 'Lin Cibubur'],
    'route_desc'       : ['LRT Jabodebek Lin Bekasi (Dukuh Atas-Jati Mulya)', 'LRT Jabodebek Lin Cibubur (Dukuh Atas-Harjamukti)'],
    'route_type'       : [0, 0],
    'route_url'        : ['https://id.wikipedia.org/wiki/Lin_Bekasi_(LRT_Jabodebek)', 'https://id.wikipedia.org/wiki/Lin_Cibubur_(LRT_Jabodebek)'],
    'route_color'      : ['0E6938', '20409A'],
    'route_text_color' : ['FFFFFF', 'FFFFFF'],
    'route_sort_order' : [1, 1],
    'network_id'       : ['LRTJAB', 'LRTJAB']
}

df_routes = pd.DataFrame(routes_data)
df_routes.to_csv(
    "routes.txt",
    index=False,
    encoding='utf-8',
    lineterminator='\n',
    quoting=csv.QUOTE_MINIMAL
)

print("✅ File routes.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_routes

✅ File routes.txt berhasil dibuat!

📄 Preview isi file:


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order,network_id
0,BK,KAI,BK,Lin Bekasi,LRT Jabodebek Lin Bekasi (Dukuh Atas-Jati Mulya),0,https://id.wikipedia.org/wiki/Lin_Bekasi_(LRT_...,0E6938,FFFFFF,1,LRTJAB
1,CB,KAI,CB,Lin Cibubur,LRT Jabodebek Lin Cibubur (Dukuh Atas-Harjamukti),0,https://id.wikipedia.org/wiki/Lin_Cibubur_(LRT...,20409A,FFFFFF,1,LRTJAB


## trips.txt (Required)

File `trips.txt` mendefinisikan setiap **perjalanan individual** yang dilakukan dalam satu rute. Jika `routes.txt` mendeskripsikan *"jalur apa"*, maka `trips.txt` mendeskripsikan *"perjalanan spesifik kapan dan ke arah mana"*. Satu rute bisa memiliki ratusan trip dalam sehari, setiap keberangkatan dari stasiun awal adalah satu trip tersendiri. Pada konteks LRT Jabodebek, satu trip mewakilkan satu perjalanan untuk satu Lin semisal Lin Bekasi dari Dukuh Atas Bank Syariah Indonesia ke Jati Mulya atau sebaliknya dari Jati Mulya ke Dukuh Atas Bank Syariah Indonesia pada waktu spesifik tertentu.
    

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `trip_id` | `stop_times.txt` | Foreign ID — setiap waktu pemberhentian merujuk ke trip |
| `trip_id` | `frequencies.txt` | Foreign ID — frekuensi keberangkatan per trip |
| `trip_id` | `transfers.txt` | Foreign ID — aturan transfer antar trip |
| `service_id` | `calendar.txt` | Foreign ID — jadwal hari operasi |
| `service_id` | `calendar_dates.txt` | Foreign ID — pengecualian jadwal |
| `shape_id` | `shapes.txt` | Foreign ID — jalur geometri di peta |
| `route_id` | `routes.txt` | Foreign ID — rute induk trip |

---

**Skema kolom trips.txt** 

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `route_id` | Foreign ID → `routes.route_id` | **Wajib** | ID rute yang dilayani trip ini. Untuk LRT Jabodebek diisi `BK` dan `CB`. |
| `service_id` | Foreign ID → `calendar.service_id` | **Wajib** | ID jadwal operasi. Merujuk ke `calendar.txt` atau `calendar_dates.txt`. Memberikan informasi trip ini berjalan everydays atau weekdays atau weekends saja |
| `trip_id` | Unique ID | **Wajib** | ID unik setiap perjalanan. |
| `trip_headsign` | Text | Opsional | Teks tujuan yang tampil di papan depan kendaraan |
| `trip_short_name` | Text | Opsional | Nama publik trip seperti nomor kereta  |
| `direction_id` | Enum | Opsional | Arah perjalanan. Misal untuk LRT Jabodebek Lin Bekasi nilai `0` = satu arah (Dukuh Atas Bank Syariah Indonesia → Jati Mulya), `1` = arah sebaliknya. |
| `block_id` | ID | Opsional | ID blok operasi untuk menandai trip yang dilayani oleh kendaraan yang sama secara berurutan. |
| `shape_id` | Foreign ID → `shapes.shape_id` | **Kondisional** | ID bentuk jalur di peta. **Wajib** jika ada `continuous_pickup/drop_off`. Disarankan diisi untuk tampilan rute di peta. |
| `wheelchair_accessible` | Enum | Opsional | Aksesibilitas kursi roda. Lihat tabel nilai di bawah. |
| `bikes_allowed` | Enum | Opsional | Apakah sepeda diperbolehkan untuk masuk ke dalam kendaraan |
| `cars_allowed` | Enum | Opsional | Apakah kendaraan bermotor/mobil diperbolehkan masuk ke dalam kendaraan |

---

**`wheelchair_accessible`**

| Nilai | Deskripsi |
|:---:|---|
| `0` atau kosong | Tidak ada informasi |
| `1` | Bisa diakses kursi roda | 
| `2` | Tidak bisa diakses kursi roda | 

**`bikes_allowed`** & **`cars_allowed`**

| Nilai | Deskripsi | 
|:---:|---|
| `0` atau kosong | Tidak ada informasi | 
| `1` | Diperbolehkan |
| `2` | Tidak diperbolehkan |

### Panduan Pengisian `trips.txt` — LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

---

Berdasarkan informasi yang didapatkan dari [akun instagram resmi LRT Jabodebek](https://www.instagram.com/p/DUSKhaZjzpn/?igsh=ajMyZmV2N2d1ejZr),  terdapat dua jenis operasi LRT Jabodebek berdasarkan hari yaitu **Weekday (Senin–Jumat)** dan **Weekend (Sabtu–Minggu)**. Berikut rinciannya:

`Weekday (Senin–Jumat)`

`Lin Bekasi`

<table>
  <thead>
    <tr>
      <th>No</th>
      <th>Asal</th>
      <th>Tujuan</th>
      <th>Jumlah Trip</th>
      <th>Lin</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="center">1</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td>Jati Mulya</td>
      <td align="center">107</td>
      <td rowspan="3" align="center">
        <img src="assets/LRT_Jabodebek_BK_Line_Icon.png" width="30" height="30"><br>
        <img src="https://img.shields.io/badge/BEKASI_LINE-0E6938?style=flat-square&logoColor=white">
      </td>
    </tr>
    <tr>
      <td align="center">2</td>
      <td>Jati Mulya</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td align="center">107</td>
    </tr>
    <tr>
      <td align="center"><strong>Total</strong></td>
      <td></td>
      <td></td>
      <td align="center"><strong>214</strong></td>
    </tr>
  </tbody>
</table>


`Lin Cibubur`

<table>
  <thead>
    <tr>
      <th>No</th>
      <th>Asal</th>
      <th>Tujuan</th>
      <th>Jumlah Trip</th>
      <th>Lin</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="center">1</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td>Harjamukti</td>
      <td align="center">108</td>
      <td rowspan="3" align="center">
        <img src="assets/LRT_Jabodebek_CB_Line_Icon.png" width="30" height="30"><br>
        <img src="https://img.shields.io/badge/CIBUBUR_LINE-20409A?style=flat-square&logoColor=white">
      </td>
    </tr>
    <tr>
      <td align="center">2</td>
      <td>Harjamutkti</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td align="center">108</td>
    </tr>
    <tr>
      <td align="center"><strong>Total</strong></td>
      <td></td>
      <td></td>
      <td align="center"><strong>216</strong></td>
    </tr>
  </tbody>
</table>


`Weekend (Sabtu–Minggu)`


`Lin Bekasi`

<table>
  <thead>
    <tr>
      <th>No</th>
      <th>Asal</th>
      <th>Tujuan</th>
      <th>Jumlah Trip</th>
      <th>Lin</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="center">1</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td>Jati Mulya</td>
      <td align="center">67</td>
      <td rowspan="3" align="center">
        <img src="assets/LRT_Jabodebek_BK_Line_Icon.png" width="30" height="30"><br>
        <img src="https://img.shields.io/badge/BEKASI_LINE-0E6938?style=flat-square&logoColor=white">
      </td>
    </tr>
    <tr>
      <td align="center">2</td>
      <td>Jati Mulya</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td align="center">69</td>
    </tr>
    <tr>
      <td align="center"><strong>Total</strong></td>
      <td></td>
      <td></td>
      <td align="center"><strong>136</strong></td>
    </tr>
  </tbody>
</table>


`Lin Cibubur`

<table>
  <thead>
    <tr>
      <th>No</th>
      <th>Asal</th>
      <th>Tujuan</th>
      <th>Jumlah Trip</th>
      <th>Lin</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td align="center">1</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td>Harjamukti</td>
      <td align="center">67</td>
      <td rowspan="3" align="center">
        <img src="assets/LRT_Jabodebek_CB_Line_Icon.png" width="30" height="30"><br>
        <img src="https://img.shields.io/badge/CIBUBUR_LINE-20409A?style=flat-square&logoColor=white">
      </td>
    </tr>
    <tr>
      <td align="center">2</td>
      <td>Harjamutkti</td>
      <td>Dukuh Atas Bank Syariah Indonesia</td>
      <td align="center">67</td>
    </tr>
    <tr>
      <td align="center"><strong>Total</strong></td>
      <td></td>
      <td></td>
      <td align="center"><strong>134</strong></td>
    </tr>
  </tbody>
</table>

---

**`route_id`**

Diisi sesuai dengan `route_id` yang ada di `routes.txt` yaitu `BK` dan `CB`.

---

**`service_id`**

Direferensikan dari `calendar.txt`. Untuk LRT Jabodebek terdapat dua nilai `service_id` yang menunjukkan hari operasi trip:

| `service_id` | Hari Operasi |
|---|---|
| `WD` | Weekday — Senin s/d Jumat |
| `WD` | Weekend — Sabtu s/d Minggu|

---

**`trip_id`**


Penamaan `trip_id` terdiri dari **5 komponen** yang dipisahkan tanda hubung (`-`):

```
BK/CB -  WD -  DKA  -  JTM  -  001
 │       │      │       │       │
 │       │      │       │       └── Nomor urut keberangkatan (3 digit, mulai 001)
 │       │      │       └────────── Kode stasiun tujuan
 │       │      └────────────────── Kode stasiun asal
 │       └───────────────────────── Jenis hari operasi (WD/WE)
 └──────────────────────────────── Kode Lin
```

| Komponen | Nilai | Keterangan |
|---|---|---|
| Kode lin | `BK`/`CB` | Nama Lin (Lin Bekasi/Lin Cibubur) |
| Jenis hari | `WD`/`WE` | Weekday (Senin-Jumat) / Weekend (Sabtu-Minggu) |
| Kode asal | `DKA` / `JTM` | Kode stasiun keberangkatan |
| Kode tujuan | `JTM` / `DKA` | Kode stasiun tujuan |
| Nomor urut | `001` s/d `108` | Urutan keberangkatan dalam satu hari |

**Daftar trip:**

| `trip_id` | Jenis | Hari | Asal | Tujuan | Jumlah |
|---|---|---|---|---|:---:|
| `BK-WD-DKA-JTM-001` s/d `107` | Full trip | Weekday | Dukuh Atas Bank Syariah Indonesia | Jati Mulya | 107 |
| `BK-WD-JTM-DKA-001` s/d `107` | Full trip | Weekday | Jati Mulya | Dukuh Atas Bank Syariah Indonesia | 107 |
| `BK-WE-DKA-JTM-001` s/d `067` | Full trip | Weekday | Dukuh Atas Bank Syariah Indonesia | Jati Mulya | 67 |
| `BK-WE-JTM-DKA-001` s/d `069` | Full trip | Weekday | Jati Mulya | Dukuh Atas Bank Syariah Indonesia | 69 |
| `CB-WD-DKA-HAR-001` s/d `108` | Full trip | Weekday | Dukuh Atas Bank Syariah Indonesia | Harjamukti | 108 |
| `CB-WD-HAR-DKA-001` s/d `108` | Full trip | Weekday | Harjamukti | Dukuh Atas Bank Syariah Indonesia | 108 |
| `CB-WE-DKA-HAR-001` s/d `067` | Full trip | Weekday | Dukuh Atas Bank Syariah Indonesia | Harjamukti | 67 |
| `CB-WE-HAR-DKA-001` s/d `067` | Full trip | Weekday | Harjamukti | Dukuh Atas Bank Syariah Indonesia | 67 |

**Catatan:**
- Nomor urut menggunakan 3 digit dengan leading zero (`001`, `002`, dst) agar urutan alfanumerik tetap konsisten.

---

**`trip_headsign`**

Diisi berdasarkan stasiun tujuan akhir aktual dari setiap trip yaitu stasiun terakhir yang dilayani kereta. Teks ini yang ditampilkan di papan depan kereta maupun aplikasi perjalanan.

| Lin | Kondisi | `trip_headsign` |
|---|---|---|
| Lin Bekasi| Full trip ke arah Timur | `Jati Mulya` |
| Lin Bekasi| Full trip ke arah Barat | `Dukuh Atas` |
| Lin Cibubur| Full trip ke arah Selatan | `Harjamukti` |
| Lin Cibubur| Full trip ke arah Utara | `Dukuh Atas` |

---

**`direction_id`**

Diisi berdasarkan arah operasi kereta. Tidak digunakan untuk routing oleh OTP, hanya untuk membedakan arah saat menampilkan jadwal.

| Lin | Nilai | Arah | 
|:---:|:---:|---|
| Lin Bekasi | `0` | Ke Timur (Semua trip menuju Jati Mulya) |
| Lin Bekasi | `1` | Ke Barat (Semua trip menuju Dukuh Atas) |
| Lin Cibubur | `0` | Ke Selatan (Semua trip menuju Harjamukti) |
| Lin Cibubur | `1` | Ke Utara (Semua trip menuju Dukuh Atas) |

---

**`shape_id`**

Merujuk ke `shape_id` di `shapes.txt` yang berisi koordinat jalur kereta di peta. Diisi sesuai arah trip karena memiliki urutan koordinat yang berlawanan.

| Trip | `shape_id` |
|---|---|
| Full trip DKA → JTM | `shape_DKA_JTM` |
| Full trip JTM → DKA | `shape_JTM_DKA` |
| Full trip DKA → HAR | `shape_DKA_HAR` |
| Full trip HAR → DKA | `shape_HAR_DKA` |

---

**`wheelchair_accessible`**

Diisi `1` karena seluruh rangkaian kereta LRT Jabodebek dilengkapi fasilitas aksesibel kursi roda termasuk ruang khusus di dalam kereta.

| Nilai | Deskripsi |
|:---:|---|
| `1` | Semua trip LRT Jabodebek dapat diakses kursi roda |

---

**`bikes_allowed`**

Diisi `1` karena LRT Jabodebek mengizinkan penumpang membawa sepeda ke dalam kereta dengan ketentuan yang berlaku.

| Nilai | Deskripsi |
|:---:|---|
| `1` | Sepeda diperbolehkan masuk ke dalam kereta LRT Jabodebek |

In [12]:
import pandas as pd
import csv

TRIP_CONFIG = [
    # (prefix trip_id, service_id, direction, headsign, shape_id, count)
    ('BK-WD-DKA-JTM', 'WD', 0, 'Jati Mulya', 'shape_DKA_JTM', 107),
    ('BK-WD-JTM-DKA', 'WD', 1, 'Dukuh Atas', 'shape_JTM_DKA', 107),
    ('BK-WE-DKA-JTM', 'WE', 0, 'Jati Mulya', 'shape_DKA_JTM',  67),
    ('BK-WE-JTM-DKA', 'WE', 1, 'Dukuh Atas', 'shape_JTM_DKA',  69),
    ('CB-WD-DKA-HAR', 'WD', 0, 'Harjamukti', 'shape_DKA_HAR', 108),
    ('CB-WD-HAR-DKA', 'WD', 1, 'Dukuh Atas', 'shape_HAR_DKA', 108),
    ('CB-WE-DKA-HAR', 'WE', 0, 'Harjamukti', 'shape_DKA_HAR',  67),
    ('CB-WE-HAR-DKA', 'WE', 1, 'Dukuh Atas', 'shape_HAR_DKA',  67),
]

# Mapping prefix lin → route_id
ROUTE_ID_MAP = {
    'BK': 'BK',   # Bekasi Line
    'CB': 'CB',   # Cibubur Line
}

rows = []
for prefix, service_id, direction_id, headsign, shape_id, count in TRIP_CONFIG:
    # Ambil kode lin dari 2 karakter pertama prefix trip_id
    line_code = prefix.split('-')[0]
    route_id  = ROUTE_ID_MAP.get(line_code, "")

    for i in range(1, count + 1):
        rows.append({
            'trip_id'              : f"{prefix}-{str(i).zfill(3)}",
            'route_id'             : route_id,
            'service_id'           : service_id,
            'trip_headsign'        : headsign,
            'direction_id'         : direction_id,
            'shape_id'             : shape_id,
            'wheelchair_accessible': 1,
            'bikes_allowed'        : 1,
        })

df_trips = pd.DataFrame(rows)
df_trips.to_csv(
    'trips.txt',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
    quoting=csv.QUOTE_MINIMAL
)

print(f"✅ trips.txt berhasil dibuat — {len(df_trips)} trip total\n")
print("=== Ringkasan per prefix ===")
print(df_trips.groupby(['route_id', 'service_id', 'direction_id', 'trip_headsign'])
      .size()
      .reset_index(name='jumlah_trip')
      .to_string(index=False))
print(f"\nTotal trip: {len(df_trips)}")
df_trips

✅ trips.txt berhasil dibuat — 700 trip total

=== Ringkasan per prefix ===
route_id service_id  direction_id trip_headsign  jumlah_trip
      BK         WD             0    Jati Mulya          107
      BK         WD             1    Dukuh Atas          107
      BK         WE             0    Jati Mulya           67
      BK         WE             1    Dukuh Atas           69
      CB         WD             0    Harjamukti          108
      CB         WD             1    Dukuh Atas          108
      CB         WE             0    Harjamukti           67
      CB         WE             1    Dukuh Atas           67

Total trip: 700


,trip_id,route_id,service_id,trip_headsign,direction_id,shape_id,wheelchair_accessible,bikes_allowed
0,BK-WD-DKA-JTM-001,BK,WD,Jati Mulya,0,shape_DKA_JTM,1,1
1,BK-WD-DKA-JTM-002,BK,WD,Jati Mulya,0,shape_DKA_JTM,1,1
2,BK-WD-DKA-JTM-003,BK,WD,Jati Mulya,0,shape_DKA_JTM,1,1
3,BK-WD-DKA-JTM-004,BK,WD,Jati Mulya,0,shape_DKA_JTM,1,1
4,BK-WD-DKA-JTM-005,BK,WD,Jati Mulya,0,shape_DKA_JTM,1,1
...,...,...,...,...,...,...,...,...
695,CB-WE-HAR-DKA-063,CB,WE,Dukuh Atas,1,shape_HAR_DKA,1,1
696,CB-WE-HAR-DKA-064,CB,WE,Dukuh Atas,1,shape_HAR_DKA,1,1
697,CB-WE-HAR-DKA-065,CB,WE,Dukuh Atas,1,shape_HAR_DKA,1,1
698,CB-WE-HAR-DKA-066,CB,WE,Dukuh Atas,1,shape_HAR_DKA,1,1


## calendar.txt (Conditionally Required)

File `calendar.txt` mendefinisikan **jadwal hari operasi** layanan transportasi umum, menentukan hari apa saja dalam seminggu suatu layanan beroperasi, beserta rentang tanggal berlakunya. File ini menjadi acuan bagi `trips.txt` melalui field `service_id` untuk menentukan kapan sebuah trip dijalankan.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `service_id` | `trips.txt` | Foreign ID — setiap trip merujuk ke jadwal operasi |
| `service_id` | `calendar_dates.txt` | Foreign ID — pengecualian jadwal per tanggal tertentu |

---

**Skema kolom calendar.txt**

| Field | Tipe Data | Keharusan | Deskripsi |
|---|---|---|---|
| `service_id` | Unique ID | **Wajib** | ID unik pengenal jadwal operasi. Direferensikan dari `trips.txt`.|
| `monday` | Enum | **Wajib** | Beroperasi pada hari Senin. `1` = beroperasi, `0` = tidak beroperasi. |
| `tuesday` | Enum | **Wajib** | Beroperasi pada hari Selasa. `1` = beroperasi, `0` = tidak beroperasi. |
| `wednesday` | Enum | **Wajib** | Beroperasi pada hari Rabu. `1` = beroperasi, `0` = tidak beroperasi. |
| `thursday` | Enum | **Wajib** | Beroperasi pada hari Kamis. `1` = beroperasi, `0` = tidak beroperasi. |
| `friday` | Enum | **Wajib** | Beroperasi pada hari Jumat. `1` = beroperasi, `0` = tidak beroperasi. |
| `saturday` | Enum | **Wajib** | Beroperasi pada hari Sabtu. `1` = beroperasi, `0` = tidak beroperasi. |
| `sunday` | Enum | **Wajib** | Beroperasi pada hari Minggu. `1` = beroperasi, `0` = tidak beroperasi. |
| `start_date` | Date | **Wajib** | Tanggal mulai berlakunya jadwal dalam format `YYYYMMDD`. |
| `end_date` | Date | **Wajib** | Tanggal akhir berlakunya jadwal dalam format `YYYYMMDD`. Tanggal ini termasuk dalam rentang yang berlaku. |

### Panduan Pengisian `calendar.txt` — LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

---

**`service_id`**

ID unik pengenal jadwal operasi yang direferensikan dari `trips.txt`. Untuk LRT Jabodebek terdapat dua nilai:

| `service_id` | Hari Operasi |
|---|---|
| `WD` | Weekday — Senin s/d Jumat |
| `WE` | Weekend — Sabtu & Minggu |

---

**`monday` `tuesday` `wednesday` `thursday` `friday` `saturday` `sunday`**

Diisi dengan nilai `1` (beroperasi) atau `0` (tidak beroperasi) untuk setiap hari dalam seminggu. Disesuaikan dengan jenis `service_id`:

| `service_id` | `monday` | `tuesday` | `wednesday` | `thursday` | `friday` | `saturday` | `sunday` |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| `WD` | `1` | `1` | `1` | `1` | `1` | `0` | `0` |
| `WE` | `0` | `0` | `0` | `0` | `0` | `1` | `1` |

---

**`start_date`**

Diisi `20230828` karena LRT Jabodebek mulai beroperasi secara komersial pada **28 Agustus 2023**.

---

**`end_date`**

Diisi `20301231` sebagai tanggal akhir berlakunya jadwal yaitu **31 Desember 2030**. Tanggal ini bersifat sementara dan perlu diperbarui secara berkala sesuai kebijakan operasional LRT Jabodebek.

---

**Catatan Penting**

- **Hari libur nasional** tidak ditangani di `calendar.txt`, pengecualian jadwal seperti libur nasional didefinisikan secara terpisah di `calendar_dates.txt`.
- Jika pada hari libur nasional LRT Jabodebek beroperasi dengan **jadwal weekend**, perlu ditambahkan entri di `calendar_dates.txt` untuk menonaktifkan `WD` dan mengaktifkan `WE` pada tanggal tersebut.
- Jika pada hari libur nasional LRT Jabodebek **tidak beroperasi sama sekali**, cukup tambahkan entri di `calendar_dates.txt` dengan `exception_type=2` untuk menonaktifkan service pada tanggal tersebut.

In [13]:
import pandas as pd
import csv

# Data calendar LRT Jabodebek
calendar_data = {
    'service_id' : ['WD',        'WE'        ],
    'monday'     : [1,            0          ],
    'tuesday'    : [1,            0          ],
    'wednesday'  : [1,            0          ],
    'thursday'   : [1,            0          ],
    'friday'     : [1,            0          ],
    'saturday'   : [0,            1          ],
    'sunday'     : [0,            1          ],
    'start_date' : ['20230828',  '20230828'  ],
    'end_date'   : ['20301231',  '20301231'  ],
}

df_calendar = pd.DataFrame(calendar_data)
df_calendar.to_csv(
    'calendar.txt',
    index=False,
    encoding='utf-8',
    lineterminator='\n',
    quoting=csv.QUOTE_MINIMAL
)

print("✅ calendar.txt berhasil dibuat!")
print("\n📄 Preview isi file:")
df_calendar

✅ calendar.txt berhasil dibuat!

📄 Preview isi file:


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,WD,1,1,1,1,1,0,0,20230828,20301231
1,WE,0,0,0,0,0,1,1,20230828,20301231


## shapes.txt (Optional)

File `shapes.txt` mendefinisikan **jalur geometri** yang dilalui kendaraan melayani sebuah rute di peta. Berupa rangkaian titik koordinat yang membentuk garis rute secara visual. File ini bersifat opsional namun **sangat direkomendasikan** untuk semua layanan berbasis rute karena digunakan oleh aplikasi trip planner seperti OTP dan Google Maps untuk menampilkan jalur rute yang akurat di peta. Shape tidak harus melewati tepat di atas titik halte/stasiun, namun semua halte/stasiun dalam sebuah trip harus berada dalam jarak yang dekat dengan garis shape tersebut.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan di | Sebagai |
|---|---|---|
| `shape_id` | `trips.txt` | Foreign ID — setiap trip merujuk ke shape jalurnya |
| `shape_dist_traveled` | `stop_times.txt` | Konsistensi satuan jarak antar file |

---

**Skema kolom shapes.txt**

| Field | Tipe Data | Keharusan | Deskripsi |
|---|---|---|---|
| `shape_id` | ID | **Wajib** | ID pengenal shape. Satu shape terdiri dari banyak titik koordinat yang semuanya berbagi `shape_id` yang sama. |
| `shape_pt_lat` | Latitude | **Wajib** | Koordinat lintang titik shape dalam format desimal WGS84. |
| `shape_pt_lon` | Longitude | **Wajib** | Koordinat bujur titik shape dalam format desimal WGS84. |
| `shape_pt_sequence` | Non-negative integer | **Wajib** | Urutan titik dalam shape. Nilai harus selalu meningkat sepanjang perjalanan namun tidak harus berurutan berurutan (boleh `0, 5, 10` dst). |
| `shape_dist_traveled` | Non-negative float | Opsional | Jarak aktual yang ditempuh dari titik pertama shape hingga titik ini. Digunakan trip planner untuk menampilkan porsi jalur yang benar di peta. Nilai harus selalu meningkat sesuai `shape_pt_sequence` dan satuan harus konsisten dengan `stop_times.txt`. |

### Panduan Pengisian `shapes.txt` — LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

---

**`shape_id`**

UntukLRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1 diperlukan **4 shape** sesuai variasi trip yang ada:

| `shape_id` | Digunakan untuk | Keterangan |
|---|---|---|
| `shape_DKA_JTM` | Lin Bekasi Full trip arah Timur | Seluruh jalur DKA → JTM |
| `shape_JTM_DKA` | Lin Bekasi Full trip arah Barat | Seluruh jalur JTM → DKA |
| `shape_DKA_HAR` | Lin Cibubur Full trip arah Selatan | Seluruh jalur DKA → HAR |
| `shape_HAR_DKA` | Lin Cibubur Full trip arah Utara | Seluruh jalur HAR → DKA |

---

**`shape_pt_lat`** & **`shape_pt_lon`**

Koordinat titik-titik jalur diambil dari **Relation OpenStreetMap** yang tersedia secara terbuka. Setiap relation OSM berisi urutan titik koordinat yang membentuk jalur LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1 secara lengkap.

| `shape_id` | Sumber Relation OSM |
|---|---|
| `shape_DKA_JTM` | `https://www.openstreetmap.org/relation/16079479` |
| `shape_JTM_DKA` | `https://www.openstreetmap.org/relation/16079478` |
| `shape_DKA_HAR` | `https://www.openstreetmap.org/relation/16036440` |
| `shape_HAR_DKA` | `https://www.openstreetmap.org/relation/16036441` |

Untuk mengambil data koordinat dari OSM digunakan dua cara yaitu secara online **Overpass API** dengan beberapa mirror sebagai fallback atau secara offline yaitu ambil relation dari https://overpass-turbo.eu/ sebagai GeoJSON dengan query sebagai berikut:

**Online**
```python
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
```

**Offline**
```
1. Buka overpass-turbo.eu
2. Jalankan query:
    
    [out:json][timeout:90];
    relation(relation_id);
    out body;
    >;
    out skel qt;
    
3. Klik Export → GeoJSON
4. Simpan sebagai: relation_{relation_id}.geojson
```
---

**`shape_pt_sequence`**

Diisi dengan angka urut yang **selalu meningkat** untuk setiap titik dalam satu shape dimulai dari `0` untuk titik pertama. Nilai tidak harus berurutan berurutan (boleh `0, 1, 2` atau `0, 10, 20` dst) selama nilainya selalu bertambah.

| Titik | `shape_pt_sequence` |
|---|---|
| Titik pertama (stasiun asal) | `0` |
| Titik berikutnya | `1`, `2`, `3`, ... |
| Titik terakhir (stasiun tujuan) | `n` |

---

Berikut versi yang sudah dirapikan, konsisten, dan siap untuk dokumentasi markdown:

---

**`shape_dist_traveled` pada `shapes.txt`**

`shape_dist_traveled` merepresentasikan **jarak kumulatif aktual** yang ditempuh sepanjang shape, dihitung dari titik pertama hingga setiap titik berikutnya.

Nilai ini dihitung berdasarkan jarak antar pasangan titik koordinat:

* `shape_pt_lat`
* `shape_pt_lon`

yang disusun sesuai `shape_pt_sequence`.

**Pendekatan Perhitungan**

Terdapat **dua pendekatan umum** dalam menghitung `shape_dist_traveled`:


**1. Menggunakan Rumus Haversine (Manual)**

Pendekatan ini menghitung jarak antar dua titik di permukaan bumi menggunakan model bola bumi.

```python
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam meter (Haversine)."""
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))
```

Perhitungan dilakukan secara **kumulatif**:

* Titik 0 → `0`
* Titik 1 → jarak(0 → 1)
* Titik 2 → jarak(0 → 1) + jarak(1 → 2)
* dan seterusnya

**Cocok untuk:**

* Implementasi manual (pandas / numpy)
* Tanpa dependency GIS


**2. Menggunakan `gtfs_kit.append_dist_to_shapes()`**

Fungsi dari `gtfs_kit` menghitung jarak menggunakan pendekatan berbasis proyeksi:

**Proses:**

1. Mengubah koordinat dari **WGS84 (lat/lon)** ke sistem proyeksi berbasis meter (**UTM**)
2. Menghitung jarak antar titik menggunakan **Euclidean distance (planar)**
3. Menjumlahkannya secara **kumulatif** sepanjang shape

Sehingga metode yang digunakan adalah:

**`Projected distance (UTM) + Euclidean distance`**

**Keunggulan**

* Lebih cepat dibanding Haversine
* Akurat untuk jarak pendek (seperti shape GTFS)
* Konsisten dengan pendekatan GIS umum

**Catatan**

* Kedua metode **valid dalam spesifikasi GTFS**
* Hasil perhitungan dapat sedikit berbeda (umumnya sangat kecil)
* Hal yang paling penting:

  * Nilai harus **monoton meningkat**
  * Menggunakan **satuan yang konsisten** (meter atau kilometer)

---

**Catatan Penting**

- Satuan `shape_dist_traveled` harus **konsisten** dengan satuan yang digunakan di `stop_times.txt`.
- `shape_dist_traveled` **sangat disarankan diisi** untuk LRT Jabodebek agar OTP dapat menampilkan posisi kereta dan progres perjalanan yang akurat di peta.

### **Build `shape_DKA_JTM`, `shape_JTM_DKA`, `shape_DKA_HAR`, `shape_HAR_DKA`**

#### Validasi relation OSM `shape_DKA_JTM`, `shape_JTM_DKA`, `shape_DKA_HAR`, `shape_HAR_DKA`

In [14]:
# relation_16079479 = Dukuh Atas Bank Syariah Indonesia - Jatimulya
import json
import folium

# Load geojson
with open("data/shapes/relation_16079479.geojson") as f:
    data = json.load(f)

# Ambil koordinat
coords = data["features"][0]["geometry"]["coordinates"]

# GeoJSON: (lon, lat) → Folium: (lat, lon)
coords_latlon = [(lat, lon) for lon, lat in coords]

# Center map
center = coords_latlon[len(coords_latlon)//2]

m = folium.Map(location=center, zoom_start=13)

# Tambahkan line
folium.PolyLine(
    coords_latlon,
    color="blue",
    weight=4,
    opacity=0.8,
    tooltip="Route"
).add_to(m)

# Start marker
folium.Marker(
    coords_latlon[0],
    popup="START",
    icon=folium.Icon(color="green")
).add_to(m)

# End marker
folium.Marker(
    coords_latlon[-1],
    popup="END",
    icon=folium.Icon(color="red")
).add_to(m)

m

In [15]:
# relation_16079478 = Jatimulya - Dukuh Atas Bank Syariah Indonesia 
import json
import folium

# Load geojson
with open("data/shapes/relation_16079478.geojson") as f:
    data = json.load(f)

# Ambil koordinat
coords = data["features"][0]["geometry"]["coordinates"]

# GeoJSON: (lon, lat) → Folium: (lat, lon)
coords_latlon = [(lat, lon) for lon, lat in coords]

# Center map
center = coords_latlon[len(coords_latlon)//2]

m = folium.Map(location=center, zoom_start=13)

# Tambahkan line
folium.PolyLine(
    coords_latlon,
    color="blue",
    weight=4,
    opacity=0.8,
    tooltip="Route"
).add_to(m)

# Start marker
folium.Marker(
    coords_latlon[0],
    popup="START",
    icon=folium.Icon(color="green")
).add_to(m)

# End marker
folium.Marker(
    coords_latlon[-1],
    popup="END",
    icon=folium.Icon(color="red")
).add_to(m)

m

In [16]:
# relation_16036440 = Dukuh Atas Bank Syariah Indonesia - Harjamukti
import json
import folium

# Load geojson
with open("data/shapes/relation_16036440.geojson") as f:
    data = json.load(f)

# Ambil koordinat
coords = data["features"][0]["geometry"]["coordinates"]

# GeoJSON: (lon, lat) → Folium: (lat, lon)
coords_latlon = [(lat, lon) for lon, lat in coords]

# Center map
center = coords_latlon[len(coords_latlon)//2]

m = folium.Map(location=center, zoom_start=13)

# Tambahkan line
folium.PolyLine(
    coords_latlon,
    color="blue",
    weight=4,
    opacity=0.8,
    tooltip="Route"
).add_to(m)

# Start marker
folium.Marker(
    coords_latlon[0],
    popup="START",
    icon=folium.Icon(color="green")
).add_to(m)

# End marker
folium.Marker(
    coords_latlon[-1],
    popup="END",
    icon=folium.Icon(color="red")
).add_to(m)

m

In [17]:
# relation_16036441 = Harjamukti - Dukuh Atas Bank Syariah Indonesia
import json
import folium

# Load geojson
with open("data/shapes/relation_16036441.geojson") as f:
    data = json.load(f)

# Ambil koordinat
coords = data["features"][0]["geometry"]["coordinates"]

# GeoJSON: (lon, lat) → Folium: (lat, lon)
coords_latlon = [(lat, lon) for lon, lat in coords]

# Center map
center = coords_latlon[len(coords_latlon)//2]

m = folium.Map(location=center, zoom_start=13)

# Tambahkan line
folium.PolyLine(
    coords_latlon,
    color="blue",
    weight=4,
    opacity=0.8,
    tooltip="Route"
).add_to(m)

# Start marker
folium.Marker(
    coords_latlon[0],
    popup="START",
    icon=folium.Icon(color="green")
).add_to(m)

# End marker
folium.Marker(
    coords_latlon[-1],
    popup="END",
    icon=folium.Icon(color="red")
).add_to(m)

m

**`Hasil Validasi Relation OSM`**

Berdasarkan hasil validasi melalui visualisasi *shape relations*, diperoleh bahwa:

- Rute **`DKA-JTM`** dan **`JTM-DKA`**  
  ✅ Sudah sesuai dan tidak ditemukan kesalahan jalur

- Rute **`DKA-HAR`** dan **`HAR-DKA`**  
  ⚠️ Masih terdapat kesalahan pada segmen masuk **Stasiun Cawang**

---

**`Permasalahan`**

Pada kondisi saat ini, jalur yang digunakan belum sesuai dengan operasional sebenarnya:

- Dari arah **Dukuh Atas → Cawang**  untuk rute **`DKA-HAR`**

  ❌ Masuk ke **jalur 1** stasiun Cawang

  ✅ Seharusnya masuk ke **jalur 3** stasiun Cawang

- Dari arah **Harjamukti → Cawang** untuk rute **`HAR-DKA`**

  ❌ Masuk ke **jalur 2**  stasiun Cawang

  ✅ Seharusnya masuk ke **jalur 4** stasiun Cawang

---

**`Solusi Perbaikan`**

Untuk memperbaiki kesalahan tersebut, perlu dilakukan penggantian segmen *way* pada area Stasiun Cawang dengan *way* yang sesuai.

**` ✅ Way yang benar (pengganti) `**

- **Jalur 3 Stasiun Cawang**  
  https://www.openstreetmap.org/way/747460263

- **Jalur 4 Stasiun Cawang**  
  https://www.openstreetmap.org/way/747460262

---

**` ❌ Way yang salah (diganti) `**

- **Jalur 1 Stasiun Cawang**  
  https://www.openstreetmap.org/way/1364419054

- **Jalur 2 Stasiun Cawang**  
  https://www.openstreetmap.org/way/1364419053

---

**` Implementasi `**

Kedua *way* yang benar akan:

- Menggantikan segmen *way* lama pada *shape relations*
- Diterapkan pada rute:
  - `DKA-HAR`
  - `HAR-DKA`
- Dilakukan pada level **geometry (koordinat)** agar sesuai dengan kebutuhan GTFS

In [18]:
"""
Mengganti segmen way yang salah dalam GeoJSON rute LRT Jabodebek rute Dukuh Atas Bank Syariah Indonesia - Harjamukti dengan way yang benar dari OpenStreetMap.

Masalah:
  - Way 1364419054 : jalur SALAH (masuk stasiun Cawang via jalur 1)
  - Way 747460263  : jalur BENAR (jalur yang seharusnya, masuk stasiun Cawang via jalur 3)

Cara kerja:
  1. Fetch node lat/lon dari way SALAH (1364419054) via Overpass API
  2. Fetch node lat/lon dari way BENAR (747460263) via Overpass API
  3. Temukan segmen koordinat way SALAH di dalam LineString GeoJSON
  4. Ganti segmen tersebut dengan koordinat way BENAR
  5. Simpan GeoJSON hasil perbaikan

Catatan:
  - Way di Overpass menggunakan [lon, lat] untuk GeoJSON
  - Pencocokan segmen menggunakan toleransi jarak (snap threshold)
"""

import json
import math
import time
import requests
from datetime import datetime

# Path
INPUT_GEOJSON  = "data/shapes/relation_16036440.geojson"
OUTPUT_GEOJSON = "data/shapes/relation_16036440_fixed.geojson"

WAY_WRONG   = 1364419054   # way yang salah — akan didrop
WAY_CORRECT = 747460263    # way yang benar — pengganti

# Toleransi jarak (meter) untuk mencocokkan titik koordinat
SNAP_THRESHOLD_M = 10.0

OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

REQUEST_HEADERS = {
    "User-Agent"  : "GTFS-Builder/1.0 (LRT Jabodebek GTFS)",
    "Accept"      : "application/json",
    "Content-Type": "application/x-www-form-urlencoded",
}
# =====================================


# Haversine Formula 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik dalam METER."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Fetch way dari Overpass 
def try_request(mirror, query):
    """POST raw dulu, fallback ke GET params."""
    try:
        resp = requests.post(mirror, data=query, headers=REQUEST_HEADERS, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_post:
        print(f"    ⚠️  POST gagal: {e_post}")

    try:
        resp = requests.get(mirror, params={"data": query}, headers=REQUEST_HEADERS, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_get:
        print(f"    ⚠️  GET gagal: {e_get}")
        raise e_get


def fetch_way_nodes(way_id):
    """
    Fetch semua node (lat, lon) dari satu way via Overpass API.
    Return: list of (lat, lon) sesuai urutan way.
    """
    query = (
        f"[out:json][timeout:60];"
        f"way({way_id});"
        f"out body;"
        f">;"
        f"out skel qt;"
    )
    last_err = None
    for mirror in OVERPASS_MIRRORS:
        print(f"  → Mencoba {mirror} ...")
        try:
            data = try_request(mirror, query)
            # Parse nodes
            nodes = {
                e["id"]: (e["lat"], e["lon"])
                for e in data["elements"] if e["type"] == "node"
            }
            # Ambil urutan node dari way
            way = next((e for e in data["elements"] if e["type"] == "way"), None)
            if not way:
                raise ValueError(f"Way {way_id} tidak ditemukan!")
            coords = [nodes[nid] for nid in way["nodes"] if nid in nodes]
            print(f"  ✅ Way {way_id}: {len(coords)} node")
            return coords
        except Exception as e:
            print(f"  ✗  Gagal: {e}")
            last_err = e
            time.sleep(2)

    raise RuntimeError(f"Semua mirror gagal untuk way {way_id}: {last_err}")


# Cari segmen di LineString
def find_segment_indices(line_coords, way_coords, threshold=SNAP_THRESHOLD_M):
    """
    Cari indeks awal dan akhir segmen way di dalam LineString.

    Strategi:
      1. Cari titik di LineString yang paling dekat ke titik PERTAMA way
      2. Cari titik di LineString yang paling dekat ke titik TERAKHIR way
      3. Verifikasi bahwa titik-titik di antaranya cocok dengan way

    line_coords : list of [lon, lat]  (format GeoJSON)
    way_coords  : list of (lat, lon)  (format Overpass)

    Return: (idx_start, idx_end, dist_start, dist_end)
    """
    # Titik pertama dan terakhir way (dalam format lat, lon)
    way_first = way_coords[0]
    way_last  = way_coords[-1]

    best_start      = None
    best_start_dist = float("inf")
    best_end        = None
    best_end_dist   = float("inf")

    for i, (lon, lat) in enumerate(line_coords):
        d_first = haversine_m(lat, lon, way_first[0], way_first[1])
        d_last  = haversine_m(lat, lon, way_last[0],  way_last[1])

        if d_first < best_start_dist:
            best_start_dist = d_first
            best_start      = i
        if d_last < best_end_dist:
            best_end_dist = d_last
            best_end      = i

    # Pastikan urutan benar
    if best_start > best_end:
        best_start, best_end = best_end, best_start
        way_coords = list(reversed(way_coords))

    return best_start, best_end, best_start_dist, best_end_dist, way_coords


# Replace segmen
def replace_segment(line_coords, idx_start, idx_end, new_way_coords):
    """
    Ganti segmen line_coords[idx_start:idx_end+1] dengan koordinat way baru.

    line_coords    : list of [lon, lat]  (GeoJSON format)
    new_way_coords : list of (lat, lon)  (Overpass format)

    Return: list of [lon, lat] yang sudah diperbaiki
    """
    # Konversi way baru dari (lat, lon) ke [lon, lat] (GeoJSON format)
    new_segment = [[lon, lat] for lat, lon in new_way_coords]

    # Gabungkan: bagian sebelum + segmen baru + bagian sesudah
    result = (
        line_coords[:idx_start]     # sebelum segmen salah
        + new_segment               # segmen baru (benar)
        + line_coords[idx_end + 1:] # sesudah segmen salah
    )
    return result


def main():
    print("=" * 60)
    print(" Replace Way in GeoJSON  |  LRT Jabodebek Rute Dukuh Atas Bank Syariah Indonesia - Harjamukti")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Input  : {INPUT_GEOJSON}")
    print(f" Output : {OUTPUT_GEOJSON}")
    print(f" Drop   : way/{WAY_WRONG}  (jalur salah)")
    print(f" Insert : way/{WAY_CORRECT} (jalur benar)")
    print("=" * 60)

    # 1. Load GeoJSON 
    print(f"\n📂 Loading {INPUT_GEOJSON} ...")
    with open(INPUT_GEOJSON, "r", encoding="utf-8") as f:
        geojson = json.load(f)

    # Cari feature LineString (rute utuh)
    line_feature = next(
        (f for f in geojson["features"]
         if f.get("geometry", {}).get("type") == "LineString"),
        None
    )
    if not line_feature:
        print("❌ Tidak ada LineString di GeoJSON!")
        return

    line_coords = line_feature["geometry"]["coordinates"]  # [[lon, lat], ...]
    print(f"   LineString ditemukan: {len(line_coords)} titik")
    print(f"   Titik pertama : {line_coords[0]}")
    print(f"   Titik terakhir: {line_coords[-1]}")

    # 2. Fetch koordinat way SALAH dari Overpass
    print(f"\n{'─'*55}")
    print(f"🌐 Fetch way SALAH: way/{WAY_WRONG}")
    way_wrong_coords = fetch_way_nodes(WAY_WRONG)
    print(f"   Titik pertama : {way_wrong_coords[0]}")
    print(f"   Titik terakhir: {way_wrong_coords[-1]}")

    time.sleep(2)

    # 3. Fetch koordinat way BENAR dari Overpass
    print(f"\n{'─'*55}")
    print(f"🌐 Fetch way BENAR: way/{WAY_CORRECT}")
    way_correct_coords = fetch_way_nodes(WAY_CORRECT)
    print(f"   Titik pertama : {way_correct_coords[0]}")
    print(f"   Titik terakhir: {way_correct_coords[-1]}")

    # 4. Cari posisi segmen SALAH di LineString
    print(f"\n{'─'*55}")
    print(f"🔍 Mencari segmen way/{WAY_WRONG} di LineString ...")
    idx_start, idx_end, d_start, d_end, way_wrong_coords = find_segment_indices(
        line_coords, way_wrong_coords
    )

    print(f"   Indeks awal   : {idx_start}  (jarak ke titik pertama way: {d_start:.1f} m)")
    print(f"   Indeks akhir  : {idx_end}    (jarak ke titik terakhir way: {d_end:.1f} m)")
    print(f"   Jumlah titik yang akan didrop: {idx_end - idx_start + 1}")

    if d_start > SNAP_THRESHOLD_M * 10 or d_end > SNAP_THRESHOLD_M * 10:
        print(f"\n  ⚠️  Jarak ke segmen cukup besar!")
        print(f"     Pastikan way {WAY_WRONG} memang bagian dari rute ini.")
        print(f"     Lanjutkan? (script tetap berjalan)")

    # 5. Sesuaikan arah way BENAR
    print(f"\n{'─'*55}")
    print(f"🔄 Menyesuaikan arah way/{WAY_CORRECT} ...")

    # Titik sebelum segmen yang akan diganti (konteks arah)
    context_before = line_coords[idx_start - 1] if idx_start > 0 else line_coords[0]
    context_lat    = context_before[1]
    context_lon    = context_before[0]

    # Cek apakah way benar perlu di-reverse
    d_to_first = haversine_m(context_lat, context_lon,
                             way_correct_coords[0][0], way_correct_coords[0][1])
    d_to_last  = haversine_m(context_lat, context_lon,
                             way_correct_coords[-1][0], way_correct_coords[-1][1])

    if d_to_last < d_to_first:
        way_correct_coords = list(reversed(way_correct_coords))
        print(f"   🔄 Way benar di-reverse agar arahnya sesuai")
    else:
        print(f"   ✅ Arah way benar sudah sesuai — tidak perlu reverse")

    print(f"   Titik pertama (final): {way_correct_coords[0]}")
    print(f"   Titik terakhir (final): {way_correct_coords[-1]}")

    # 6. Replace segmen
    print(f"\n{'─'*55}")
    print(f"✂️  Mengganti segmen ...")
    n_before = len(line_coords)

    new_line_coords = replace_segment(
        line_coords, idx_start, idx_end, way_correct_coords
    )

    n_after = len(new_line_coords)
    print(f"   Titik sebelum : {n_before}")
    print(f"   Titik didrop  : {idx_end - idx_start + 1} (way {WAY_WRONG})")
    print(f"   Titik ditambah: {len(way_correct_coords)} (way {WAY_CORRECT})")
    print(f"   Titik sesudah : {n_after}")

    # 7. Update GeoJSON & simpan
    print(f"\n{'─'*55}")
    print(f"💾 Menyimpan {OUTPUT_GEOJSON} ...")
    line_feature["geometry"]["coordinates"] = new_line_coords

    # Tambahkan catatan di properties
    props = line_feature.get("properties", {})
    props["_gtfs_fix"] = (
        f"Replaced way/{WAY_WRONG} with way/{WAY_CORRECT} "
        f"on {datetime.now().strftime('%Y-%m-%d')}"
    )
    line_feature["properties"] = props

    with open(OUTPUT_GEOJSON, "w", encoding="utf-8") as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)

    print(f"   ✅ Tersimpan!")
    
    print(f"\n{'='*55}")
    print(f"✅ Selesai!")
    print(f"   Input  : {INPUT_GEOJSON} ({n_before} titik)")
    print(f"   Output : {OUTPUT_GEOJSON} ({n_after} titik)")
    print(f"   Drop   : way/{WAY_WRONG} ({idx_end - idx_start + 1} titik)")
    print(f"   Insert : way/{WAY_CORRECT} ({len(way_correct_coords)} titik)")

if __name__ == "__main__":
    main()

 Replace Way in GeoJSON  |  LRT Jabodebek Rute Dukuh Atas Bank Syariah Indonesia - Harjamukti
 2026-05-03 22:31:42
 Input  : data/shapes/relation_16036440.geojson
 Output : data/shapes/relation_16036440_fixed.geojson
 Drop   : way/1364419054  (jalur salah)
 Insert : way/747460263 (jalur benar)

📂 Loading data/shapes/relation_16036440.geojson ...
   LineString ditemukan: 392 titik
   Titik pertama : [106.8255247, -6.2048511]
   Titik terakhir: [106.8956519, -6.3738957]

───────────────────────────────────────────────────────
🌐 Fetch way SALAH: way/1364419054
  → Mencoba https://overpass-api.de/api/interpreter ...
  ✅ Way 1364419054: 9 node
   Titik pertama : (-6.2463535, 106.872859)
   Titik terakhir: (-6.2454609, 106.8697709)

───────────────────────────────────────────────────────
🌐 Fetch way BENAR: way/747460263
  → Mencoba https://overpass-api.de/api/interpreter ...
  ✅ Way 747460263: 12 node
   Titik pertama : (-6.2454609, 106.8697709)
   Titik terakhir: (-6.2463535, 106.872859)

─

In [19]:
# relation_16036440 = Dukuh Atas Bank Syariah Indonesia - Harjamukti
import json
import folium

# Load geojson
with open("data/shapes/relation_16036440_fixed.geojson") as f:
    data = json.load(f)

# Ambil koordinat
coords = data["features"][0]["geometry"]["coordinates"]

# GeoJSON: (lon, lat) → Folium: (lat, lon)
coords_latlon = [(lat, lon) for lon, lat in coords]

# Center map
center = coords_latlon[len(coords_latlon)//2]

m = folium.Map(location=center, zoom_start=13)

# Tambahkan line
folium.PolyLine(
    coords_latlon,
    color="blue",
    weight=4,
    opacity=0.8,
    tooltip="Route"
).add_to(m)

# Start marker
folium.Marker(
    coords_latlon[0],
    popup="START",
    icon=folium.Icon(color="green")
).add_to(m)

# End marker
folium.Marker(
    coords_latlon[-1],
    popup="END",
    icon=folium.Icon(color="red")
).add_to(m)

m

In [20]:
"""
Mengganti segmen way yang salah dalam GeoJSON rute LRT Jabodebek rute Harjamukti - Dukuh Atas Bank Syariah Indonesia dengan way yang benar dari OpenStreetMap.

Masalah:
  - Way 1364419053 : jalur SALAH (masuk stasiun Cawang via jalur 2)
  - Way 747460262  : jalur BENAR (jalur yang seharusnya, masuk stasiun Cawang via jalur 4)

Cara kerja:
  1. Fetch node lat/lon dari way SALAH (1364419053) via Overpass API
  2. Fetch node lat/lon dari way BENAR (747460262) via Overpass API
  3. Temukan segmen koordinat way SALAH di dalam LineString GeoJSON
  4. Ganti segmen tersebut dengan koordinat way BENAR
  5. Simpan GeoJSON hasil perbaikan

Catatan:
  - Way di Overpass menggunakan [lon, lat] untuk GeoJSON
  - Pencocokan segmen menggunakan toleransi jarak (snap threshold)
"""

import json
import math
import time
import requests
from datetime import datetime

# Path
INPUT_GEOJSON  = "data/shapes/relation_16036441.geojson"
OUTPUT_GEOJSON = "data/shapes/relation_16036441_fixed.geojson"

WAY_WRONG   = 1364419053   # way yang salah — akan didrop
WAY_CORRECT = 747460262    # way yang benar — pengganti

# Toleransi jarak (meter) untuk mencocokkan titik koordinat
SNAP_THRESHOLD_M = 10.0

OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

REQUEST_HEADERS = {
    "User-Agent"  : "GTFS-Builder/1.0 (LRT Jabodebek GTFS)",
    "Accept"      : "application/json",
    "Content-Type": "application/x-www-form-urlencoded",
}
# =====================================


# Haversine Formula 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik dalam METER."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Fetch way dari Overpass 
def try_request(mirror, query):
    """POST raw dulu, fallback ke GET params."""
    try:
        resp = requests.post(mirror, data=query, headers=REQUEST_HEADERS, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_post:
        print(f"    ⚠️  POST gagal: {e_post}")

    try:
        resp = requests.get(mirror, params={"data": query}, headers=REQUEST_HEADERS, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_get:
        print(f"    ⚠️  GET gagal: {e_get}")
        raise e_get


def fetch_way_nodes(way_id):
    """
    Fetch semua node (lat, lon) dari satu way via Overpass API.
    Return: list of (lat, lon) sesuai urutan way.
    """
    query = (
        f"[out:json][timeout:60];"
        f"way({way_id});"
        f"out body;"
        f">;"
        f"out skel qt;"
    )
    last_err = None
    for mirror in OVERPASS_MIRRORS:
        print(f"  → Mencoba {mirror} ...")
        try:
            data = try_request(mirror, query)
            # Parse nodes
            nodes = {
                e["id"]: (e["lat"], e["lon"])
                for e in data["elements"] if e["type"] == "node"
            }
            # Ambil urutan node dari way
            way = next((e for e in data["elements"] if e["type"] == "way"), None)
            if not way:
                raise ValueError(f"Way {way_id} tidak ditemukan!")
            coords = [nodes[nid] for nid in way["nodes"] if nid in nodes]
            print(f"  ✅ Way {way_id}: {len(coords)} node")
            return coords
        except Exception as e:
            print(f"  ✗  Gagal: {e}")
            last_err = e
            time.sleep(2)

    raise RuntimeError(f"Semua mirror gagal untuk way {way_id}: {last_err}")


# Cari segmen di LineString
def find_segment_indices(line_coords, way_coords, threshold=SNAP_THRESHOLD_M):
    """
    Cari indeks awal dan akhir segmen way di dalam LineString.

    Strategi:
      1. Cari titik di LineString yang paling dekat ke titik PERTAMA way
      2. Cari titik di LineString yang paling dekat ke titik TERAKHIR way
      3. Verifikasi bahwa titik-titik di antaranya cocok dengan way

    line_coords : list of [lon, lat]  (format GeoJSON)
    way_coords  : list of (lat, lon)  (format Overpass)

    Return: (idx_start, idx_end, dist_start, dist_end)
    """
    # Titik pertama dan terakhir way (dalam format lat, lon)
    way_first = way_coords[0]
    way_last  = way_coords[-1]

    best_start      = None
    best_start_dist = float("inf")
    best_end        = None
    best_end_dist   = float("inf")

    for i, (lon, lat) in enumerate(line_coords):
        d_first = haversine_m(lat, lon, way_first[0], way_first[1])
        d_last  = haversine_m(lat, lon, way_last[0],  way_last[1])

        if d_first < best_start_dist:
            best_start_dist = d_first
            best_start      = i
        if d_last < best_end_dist:
            best_end_dist = d_last
            best_end      = i

    # Pastikan urutan benar
    if best_start > best_end:
        best_start, best_end = best_end, best_start
        way_coords = list(reversed(way_coords))

    return best_start, best_end, best_start_dist, best_end_dist, way_coords


# Replace segmen
def replace_segment(line_coords, idx_start, idx_end, new_way_coords):
    """
    Ganti segmen line_coords[idx_start:idx_end+1] dengan koordinat way baru.

    line_coords    : list of [lon, lat]  (GeoJSON format)
    new_way_coords : list of (lat, lon)  (Overpass format)

    Return: list of [lon, lat] yang sudah diperbaiki
    """
    # Konversi way baru dari (lat, lon) ke [lon, lat] (GeoJSON format)
    new_segment = [[lon, lat] for lat, lon in new_way_coords]

    # Gabungkan: bagian sebelum + segmen baru + bagian sesudah
    result = (
        line_coords[:idx_start]     # sebelum segmen salah
        + new_segment               # segmen baru (benar)
        + line_coords[idx_end + 1:] # sesudah segmen salah
    )
    return result


def main():
    print("=" * 60)
    print(" Replace Way in GeoJSON  |  LRT Jabodebek Rute Harjamukti - Dukuh Atas Bank Syariah Indonesia")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Input  : {INPUT_GEOJSON}")
    print(f" Output : {OUTPUT_GEOJSON}")
    print(f" Drop   : way/{WAY_WRONG}  (jalur salah)")
    print(f" Insert : way/{WAY_CORRECT} (jalur benar)")
    print("=" * 60)

    # 1. Load GeoJSON 
    print(f"\n📂 Loading {INPUT_GEOJSON} ...")
    with open(INPUT_GEOJSON, "r", encoding="utf-8") as f:
        geojson = json.load(f)

    # Cari feature LineString (rute utuh)
    line_feature = next(
        (f for f in geojson["features"]
         if f.get("geometry", {}).get("type") == "LineString"),
        None
    )
    if not line_feature:
        print("❌ Tidak ada LineString di GeoJSON!")
        return

    line_coords = line_feature["geometry"]["coordinates"]  # [[lon, lat], ...]
    print(f"   LineString ditemukan: {len(line_coords)} titik")
    print(f"   Titik pertama : {line_coords[0]}")
    print(f"   Titik terakhir: {line_coords[-1]}")

    # 2. Fetch koordinat way SALAH dari Overpass
    print(f"\n{'─'*55}")
    print(f"🌐 Fetch way SALAH: way/{WAY_WRONG}")
    way_wrong_coords = fetch_way_nodes(WAY_WRONG)
    print(f"   Titik pertama : {way_wrong_coords[0]}")
    print(f"   Titik terakhir: {way_wrong_coords[-1]}")

    time.sleep(2)

    # 3. Fetch koordinat way BENAR dari Overpass
    print(f"\n{'─'*55}")
    print(f"🌐 Fetch way BENAR: way/{WAY_CORRECT}")
    way_correct_coords = fetch_way_nodes(WAY_CORRECT)
    print(f"   Titik pertama : {way_correct_coords[0]}")
    print(f"   Titik terakhir: {way_correct_coords[-1]}")

    # 4. Cari posisi segmen SALAH di LineString
    print(f"\n{'─'*55}")
    print(f"🔍 Mencari segmen way/{WAY_WRONG} di LineString ...")
    idx_start, idx_end, d_start, d_end, way_wrong_coords = find_segment_indices(
        line_coords, way_wrong_coords
    )

    print(f"   Indeks awal   : {idx_start}  (jarak ke titik pertama way: {d_start:.1f} m)")
    print(f"   Indeks akhir  : {idx_end}    (jarak ke titik terakhir way: {d_end:.1f} m)")
    print(f"   Jumlah titik yang akan didrop: {idx_end - idx_start + 1}")

    if d_start > SNAP_THRESHOLD_M * 10 or d_end > SNAP_THRESHOLD_M * 10:
        print(f"\n  ⚠️  Jarak ke segmen cukup besar!")
        print(f"     Pastikan way {WAY_WRONG} memang bagian dari rute ini.")
        print(f"     Lanjutkan? (script tetap berjalan)")

    # 5. Sesuaikan arah way BENAR
    print(f"\n{'─'*55}")
    print(f"🔄 Menyesuaikan arah way/{WAY_CORRECT} ...")

    # Titik sebelum segmen yang akan diganti (konteks arah)
    context_before = line_coords[idx_start - 1] if idx_start > 0 else line_coords[0]
    context_lat    = context_before[1]
    context_lon    = context_before[0]

    # Cek apakah way benar perlu di-reverse
    d_to_first = haversine_m(context_lat, context_lon,
                             way_correct_coords[0][0], way_correct_coords[0][1])
    d_to_last  = haversine_m(context_lat, context_lon,
                             way_correct_coords[-1][0], way_correct_coords[-1][1])

    if d_to_last < d_to_first:
        way_correct_coords = list(reversed(way_correct_coords))
        print(f"   🔄 Way benar di-reverse agar arahnya sesuai")
    else:
        print(f"   ✅ Arah way benar sudah sesuai — tidak perlu reverse")

    print(f"   Titik pertama (final): {way_correct_coords[0]}")
    print(f"   Titik terakhir (final): {way_correct_coords[-1]}")

    # 6. Replace segmen
    print(f"\n{'─'*55}")
    print(f"✂️  Mengganti segmen ...")
    n_before = len(line_coords)

    new_line_coords = replace_segment(
        line_coords, idx_start, idx_end, way_correct_coords
    )

    n_after = len(new_line_coords)
    print(f"   Titik sebelum : {n_before}")
    print(f"   Titik didrop  : {idx_end - idx_start + 1} (way {WAY_WRONG})")
    print(f"   Titik ditambah: {len(way_correct_coords)} (way {WAY_CORRECT})")
    print(f"   Titik sesudah : {n_after}")

    # 7. Update GeoJSON & simpan
    print(f"\n{'─'*55}")
    print(f"💾 Menyimpan {OUTPUT_GEOJSON} ...")
    line_feature["geometry"]["coordinates"] = new_line_coords

    # Tambahkan catatan di properties
    props = line_feature.get("properties", {})
    props["_gtfs_fix"] = (
        f"Replaced way/{WAY_WRONG} with way/{WAY_CORRECT} "
        f"on {datetime.now().strftime('%Y-%m-%d')}"
    )
    line_feature["properties"] = props

    with open(OUTPUT_GEOJSON, "w", encoding="utf-8") as f:
        json.dump(geojson, f, ensure_ascii=False, indent=2)

    print(f"   ✅ Tersimpan!")
    
    print(f"\n{'='*55}")
    print(f"✅ Selesai!")
    print(f"   Input  : {INPUT_GEOJSON} ({n_before} titik)")
    print(f"   Output : {OUTPUT_GEOJSON} ({n_after} titik)")
    print(f"   Drop   : way/{WAY_WRONG} ({idx_end - idx_start + 1} titik)")
    print(f"   Insert : way/{WAY_CORRECT} ({len(way_correct_coords)} titik)")

if __name__ == "__main__":
    main()

 Replace Way in GeoJSON  |  LRT Jabodebek Rute Harjamukti - Dukuh Atas Bank Syariah Indonesia
 2026-05-03 22:31:45
 Input  : data/shapes/relation_16036441.geojson
 Output : data/shapes/relation_16036441_fixed.geojson
 Drop   : way/1364419053  (jalur salah)
 Insert : way/747460262 (jalur benar)

📂 Loading data/shapes/relation_16036441.geojson ...
   LineString ditemukan: 390 titik
   Titik pertama : [106.8956864, -6.3738897]
   Titik terakhir: [106.8255355, -6.2048049]

───────────────────────────────────────────────────────
🌐 Fetch way SALAH: way/1364419053
  → Mencoba https://overpass-api.de/api/interpreter ...
  ✅ Way 1364419053: 9 node
   Titik pertama : (-6.2463331, 106.8729055)
   Titik terakhir: (-6.2454314, 106.8697843)

───────────────────────────────────────────────────────
🌐 Fetch way BENAR: way/747460262
  → Mencoba https://overpass-api.de/api/interpreter ...
  ✅ Way 747460262: 12 node
   Titik pertama : (-6.2463331, 106.8729055)
   Titik terakhir: (-6.2454314, 106.8697843)


In [21]:
# relation_16036441 = Harjamukti - Dukuh Atas Bank Syariah Indonesia
import json
import folium

# Load geojson
with open("data/shapes/relation_16036441_fixed.geojson") as f:
    data = json.load(f)

# Ambil koordinat
coords = data["features"][0]["geometry"]["coordinates"]

# GeoJSON: (lon, lat) → Folium: (lat, lon)
coords_latlon = [(lat, lon) for lon, lat in coords]

# Center map
center = coords_latlon[len(coords_latlon)//2]

m = folium.Map(location=center, zoom_start=13)

# Tambahkan line
folium.PolyLine(
    coords_latlon,
    color="blue",
    weight=4,
    opacity=0.8,
    tooltip="Route"
).add_to(m)

# Start marker
folium.Marker(
    coords_latlon[0],
    popup="START",
    icon=folium.Icon(color="green")
).add_to(m)

# End marker
folium.Marker(
    coords_latlon[-1],
    popup="END",
    icon=folium.Icon(color="red")
).add_to(m)

m

#### Build `shape_DKA_JTM`, `shape_JTM_DKA`, `shape_DKA_HAR`, `shape_HAR_DKA` with gtfs_kit for `shape_dist_traveled`

In [22]:
"""
Build file shapes.txt menggunakan data geometri jalur rel LRT Jabodebek yang diperoleh dari GeoJSON lokal atau melalui API Overpass (OpenStreetMap).

Shape yang dihasilkan:
  - shape_DKA_JTM : Dukuh Atas Bank Syariah Indonesia → Jati Mulya
  - shape_JTM_DKA : Jati Mulya → Dukuh Atas Bank Syariah Indonesia
  - shape_DKA_HAR : Dukuh Atas Bank Syariah Indonesia → Harjamukti
  - shape_HAR_DKA : Harjamukti → Dukuh Atas Bank Syariah Indonesia

Catatan:
  - File GeoJSON bisa didapatkan dari https://overpass-turbo.eu/ lakukan query untuk relation_id jalur tertentu kemudian download sebagai GeoJSON.
  - Terdapat proses trimming untuk memotong segmen pada stasiun terminus (awal & akhir) agar jalur rel tepat dimulai dari stop_id pertama.
  - Jika koordinat hasil parsing terbalik (arah OSM berlawanan dengan arah trip), aktifkan reverse=True pada konfigurasi RELATIONS.
  - shape_dist_traveled tidak dihitung di sini, akan dihitung menggunakan library gtfs_kit.
  - Urutan proses per shape: trim terminus → reverse (jika aktif) → tulis shapes.txt
  - Perhitungan shape_dist_traveled menggunakan gtfs_kit.append_dist_to_shapes(feed) setelah file ini dibuat.

MODE OPERASI (otomatis dipilih berdasarkan file yang tersedia):
  1. GeoJSON lokal  → jika file .geojson tersedia (dari overpass-turbo.eu)
  2. Online         → fetch dari Overpass API
"""

import requests
import csv
import math
import json
import os
import time
from datetime import datetime

# File path
OUTPUT_SHAPES = "shapes.txt"
STOPS_FILE    = "stops.txt"

# Terminus stop_id per shape
# first_stop_id : stop pertama yang dilayani trip (platform)
# last_stop_id  : stop terakhir yang dilayani trip (platform)
# reverse       : True jika koordinat OSM berlawanan arah dengan trip
RELATIONS = [
    {
        "relation_id"  : 16079479, # relation_id OSM untuk jalur LRT Jabodebek DKA-JTM
        "shape_id"     : "shape_DKA_JTM", # shape_id untuk jalur LRT Jabodebek DKA-JTM
        "label"        : "DKA → JTM", 
        "geojson_file" : "data/shapes/relation_16079479.geojson", # File GeoJSON untuk jalur LRT Jabodebek DKA-JTM
        "first_stop_id": "GDKA01", # platform DKA arah JTM
        "last_stop_id" : "GJTM01", # platform JTM arah dari DKA
        "reverse"      : False,    # koordinat OSM sudah benar
    },
    {
        "relation_id"  : 16079478, # relation_id OSM untuk jalur LRT Jabodebek JTM-DKA
        "shape_id"     : "shape_JTM_DKA", # shape_id untuk jalur LRT Jabodebek JTM-DKA
        "label"        : "JTM → DKA",
        "geojson_file" : "data/shapes/relation_16079478.geojson", # File GeoJSON untuk jalur LRT Jabodebek JTM-DKA
        "first_stop_id": "GJTM02", # platform JTM arah DKA
        "last_stop_id" : "GDKA02", # platform DKA arah dari JTM
        "reverse"      : False,   # koordinat OSM sudah benar
    },
    {
        "relation_id"  : 16036440, # relation_id OSM untuk jalur LRT Jabodebek DKA-HAR
        "shape_id"     : "shape_DKA_HAR", # shape_id untuk jalur LRT Jabodebek DKA-HAR
        "label"        : "DKA → HAR",
        "geojson_file" : "data/shapes/relation_16036440_fixed.geojson", # File GeoJSON untuk jalur LRT Jabodebek DKA-HAR
        "first_stop_id": "GDKA01", # platform DKA arah HAR
        "last_stop_id" : "GHAR01", # platform HAR arah dari DKA
        "reverse"      : False,   # koordinat OSM sudah benar
    },
    {
        "relation_id"  : 16036441, # relation_id OSM untuk jalur LRT Jabodebek HAR-DKA
        "shape_id"     : "shape_HAR_DKA", # shape_id untuk jalur LRT Jabodebek HAR-DKA
        "label"        : "HAR → DKA",
        "geojson_file" : "data/shapes/relation_16036441_fixed.geojson", # File GeoJSON untuk jalur LRT Jabodebek HAR-DKA
        "first_stop_id": "GHAR02", # platform HAR arah DKA
        "last_stop_id" : "GDKA02", # platform DKA arah dari HAR
        "reverse"      : False,   # koordinat OSM sudah benar
    },
]

OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

REQUEST_HEADERS = {
    "User-Agent"  : "GTFS-Builder/1.0 (LRT Jabodebek GTFS)",
    "Accept"      : "application/json",
    "Content-Type": "application/x-www-form-urlencoded",
}


# Rumus Haversine 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam METER — dipakai untuk chaining."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Load koordinat stop 
def load_stop_coords(stops_file, stop_id):
    """
    Baca (lat, lon, stop_name) dari stops.txt berdasarkan stop_id.
    Raise ValueError jika stop_id tidak ditemukan.
    """
    with open(stops_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["stop_id"] == stop_id:
                return float(row["stop_lat"]), float(row["stop_lon"]), row.get("stop_name", stop_id)
    raise ValueError(f"stop_id '{stop_id}' tidak ditemukan di {stops_file}")


# Trim terminus 
def find_nearest_index(coords, target_lat, target_lon):
    """Cari indeks titik di coords yang paling dekat dengan (target_lat, target_lon)."""
    best_idx  = 0
    best_dist = float("inf")
    for i, (lat, lon) in enumerate(coords):
        d = haversine_m(lat, lon, target_lat, target_lon)
        if d < best_dist:
            best_dist = d
            best_idx  = i
    return best_idx, best_dist


def trim_shape_to_terminus(coords, first_lat, first_lon, last_lat, last_lon):
    """
    Potong coords agar:
      - Dimulai dari titik terdekat ke stop pertama (first_stop)
      - Berakhir di titik terdekat ke stop terakhir (last_stop)

    Menghindari:
      - Garis jalur sebelum stasiun pertama (tampilan peta bersih)
      - trip_distance_exceeds_shape_distance warning di validator GTFS

    Return: (coords_trimmed, idx_first, idx_last, dist_first_m, dist_last_m)
    """
    idx_first, dist_first = find_nearest_index(coords, first_lat, first_lon)
    idx_last,  dist_last  = find_nearest_index(coords, last_lat,  last_lon)

    if idx_first > idx_last:
        idx_first, idx_last = idx_last, idx_first

    return coords[idx_first : idx_last + 1], idx_first, idx_last, dist_first, dist_last


# Reverse koordinat 
def reverse_coords(coords):
    """
    Balik urutan koordinat.

    Digunakan jika koordinat hasil parsing OSM berlawanan arah dengan trip.
    Proses ini dilakukan SETELAH trim terminus agar:
      1. Trim tetap bekerja dengan benar menggunakan indeks koordinat asli
      2. shape_pt_sequence di-reset dari 0 setelah reverse oleh write_shapes_txt
      3. gtfs_kit.append_dist_to_shapes() menghitung dist dari titik awal yang benar

    Contoh untuk shape_DKA_JTM:
      Sebelum reverse: JTM(0) → ... → DKA(n)  ← arah OSM
      Setelah reverse: DKA(0) → ... → JTM(n)  ← arah trip DKA→JTM ✅
    """
    return list(reversed(coords))


# GeoJSON Parser (mode offline)
def extract_coords_from_geojson(geojson_path):
    """
    Parse GeoJSON dari overpass-turbo.eu → list of (lat, lon).

    GeoJSON dari overpass-turbo berisi FeatureCollection dengan:
      - LineString       : satu segmen way
      - MultiLineString  : beberapa segmen way
      - Point            : node (diabaikan)
    """
    print(f"  → Membaca: {geojson_path}")
    with open(geojson_path, "r", encoding="utf-8") as f:
        geojson = json.load(f)

    segments = []
    n_point  = 0

    for feature in geojson.get("features", []):
        geom  = feature.get("geometry", {})
        gtype = geom.get("type", "")

        if gtype == "LineString":
            segments.append([(c[1], c[0]) for c in geom["coordinates"]])
        elif gtype == "MultiLineString":
            for line in geom["coordinates"]:
                segments.append([(c[1], c[0]) for c in line])
        elif gtype == "Point":
            n_point += 1

    if not segments:
        raise ValueError("Tidak ada LineString/MultiLineString di GeoJSON!")

    print(f"  ✅ {len(segments)} segmen ditemukan, {n_point} point diabaikan")
    ordered = _chain_segments(segments)
    print(f"  ✅ Chain selesai → {len(ordered):,} titik koordinat")
    return ordered


def _chain_segments(segments):
    """
    Sambungkan list segmen menjadi satu jalur berurutan.
    Algoritma greedy berdasarkan jarak ujung-ke-ujung terdekat.
    Gap > 50m diberi warning tapi tetap disambung.
    """
    SNAP_THRESHOLD_M = 50.0
    remaining = [list(seg) for seg in segments]
    result    = remaining.pop(0)

    while remaining:
        tail      = result[-1]
        best_idx  = None
        best_dist = float("inf")
        best_rev  = False

        for i, seg in enumerate(remaining):
            d_head = haversine_m(tail[0], tail[1], seg[0][0],  seg[0][1])
            d_tail = haversine_m(tail[0], tail[1], seg[-1][0], seg[-1][1])
            if d_head < best_dist:
                best_dist, best_idx, best_rev = d_head, i, False
            if d_tail < best_dist:
                best_dist, best_idx, best_rev = d_tail, i, True

        seg = remaining.pop(best_idx)
        if best_rev:
            seg = list(reversed(seg))
        if best_dist > SNAP_THRESHOLD_M:
            print(f"  ⚠️  Gap {best_dist:.1f}m — tetap disambung")

        skip = 1 if haversine_m(tail[0], tail[1], seg[0][0], seg[0][1]) < 1.0 else 0
        result.extend(seg[skip:])

    return result


# Overpass API (mode online)
def try_request(mirror, query):
    """POST raw dulu, fallback ke GET params."""
    try:
        print(f"    [POST raw] ...")
        resp = requests.post(mirror, data=query, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_post:
        print(f"    ⚠️  POST gagal: {e_post}")

    try:
        print(f"    [GET params] ...")
        resp = requests.get(mirror, params={"data": query}, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_get:
        print(f"    ⚠️  GET gagal: {e_get}")
        raise e_get


def fetch_relation_online(relation_id):
    """Fetch relation dari Overpass API, coba semua mirror secara berurutan."""
    query = (
        f"[out:json][timeout:90];"
        f"relation({relation_id});"
        f"out body;>;"
        f"out skel qt;"
    )
    last_err = None
    for mirror in OVERPASS_MIRRORS:
        print(f"  → Mencoba {mirror} ...")
        try:
            data = try_request(mirror, query)
            print(f"  ✅ Berhasil! {len(data['elements'])} elements")
            return data
        except Exception as e:
            print(f"  ✗  Mirror gagal\n")
            last_err = e
            time.sleep(3)
    raise RuntimeError(f"Semua mirror gagal: {last_err}")


def extract_coords_from_overpass(data):
    """Parse Overpass JSON → list of (lat, lon)."""
    nodes = {e["id"]: (e["lat"], e["lon"])
             for e in data["elements"] if e["type"] == "node"}
    ways  = {e["id"]: e["nodes"]
             for e in data["elements"] if e["type"] == "way"}
    rel   = next((e for e in data["elements"] if e["type"] == "relation"), None)

    if not rel:
        raise ValueError("Relation tidak ditemukan!")

    tags        = rel.get("tags", {})
    way_members = [m for m in rel["members"] if m["type"] == "way"]
    print(f"\n  📋 Relation: {tags.get('name','N/A')} | {len(way_members)} ways")

    ordered_coords = []
    last_node_id   = None
    skipped        = 0

    for member in way_members:
        way_id = member["ref"]
        role   = member.get("role", "")

        if way_id not in ways or not ways[way_id]:
            skipped += 1
            continue

        way_nodes  = ways[way_id]
        first_node = way_nodes[0]
        last_node  = way_nodes[-1]

        if last_node_id is None:
            reverse = (role == "backward")
        else:
            if last_node_id == first_node:
                reverse = False
            elif last_node_id == last_node:
                reverse = True
            else:
                if first_node in nodes and last_node in nodes and last_node_id in nodes:
                    c       = nodes[last_node_id]
                    d_first = haversine_m(c[0], c[1], nodes[first_node][0], nodes[first_node][1])
                    d_last  = haversine_m(c[0], c[1], nodes[last_node][0],  nodes[last_node][1])
                    reverse = d_last < d_first
                else:
                    reverse = (role == "backward")

        seq = list(reversed(way_nodes)) if reverse else way_nodes
        for j, node_id in enumerate(seq):
            if node_id not in nodes:
                continue
            coord = nodes[node_id]
            if ordered_coords and j == 0 and ordered_coords[-1] == coord:
                continue
            ordered_coords.append(coord)
        last_node_id = seq[-1]

    if skipped:
        print(f"  ⚠️  {skipped} ways dilewati")

    return ordered_coords


# Selector source data
def get_coords(rel_cfg):
    """
    Pilih sumber data secara otomatis:
      1. GeoJSON lokal → jika file .geojson ada
      2. Online        → fetch dari Overpass API
    """
    geojson_file = rel_cfg.get("geojson_file", "")
    if geojson_file and os.path.exists(geojson_file):
        print(f"  📂 Mode: GeoJSON lokal")
        return extract_coords_from_geojson(geojson_file)

    print(f"  🌐 Mode: Online Overpass API")
    data = fetch_relation_online(rel_cfg["relation_id"])
    return extract_coords_from_overpass(data)


# Output
def write_shapes_txt(all_shape_data, output_file):
    """
    Tulis shapes.txt tanpa kolom shape_dist_traveled.
    shape_dist_traveled akan dihitung oleh gtfs_kit.append_dist_to_shapes().
    shape_pt_sequence di-reset dari 0 untuk setiap shape.
    """
    fieldnames = [
        "shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"
    ]
    rows = []
    for shape_id, coords in all_shape_data.items():
        for seq, (lat, lon) in enumerate(coords):
            rows.append({
                "shape_id"         : shape_id,
                "shape_pt_lat"     : f"{lat:.7f}",
                "shape_pt_lon"     : f"{lon:.7f}",
                "shape_pt_sequence": seq,
            })

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return rows


def save_json_ref(shape_id, relation_id, coords):
    """Simpan JSON referensi koordinat per shape (untuk keperluan debug/cek)."""
    total_m = sum(
        haversine_m(coords[i-1][0], coords[i-1][1],
                    coords[i][0],   coords[i][1])
        for i in range(1, len(coords))
    )
    json_path = f"shape_ref_{shape_id}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({
            "shape_id"     : shape_id,
            "relation_id"  : relation_id,
            "total_points" : len(coords),
            "total_dist_m" : round(total_m, 1),
            "total_dist_km": round(total_m / 1000, 3),
            "coords"       : [{"lat": lat, "lon": lon} for lat, lon in coords]
        }, f, indent=2)
    return json_path, total_m

def main():
    print("=" * 60)
    print(" OSM → GTFS shapes.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" shape_dist_traveled : tidak dihitung — gunakan gtfs_kit setelahnya")
    print(f" Reverse             : per konfigurasi RELATIONS")
    print("=" * 60)
    print()
    print(" Prioritas sumber data:")
    print("   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada")
    print("   2. Online Overpass API               — jika tidak ada")
    print()
    for rel in RELATIONS:
        ada     = os.path.exists(rel["geojson_file"])
        status  = "✅ ada → pakai GeoJSON" if ada else "⬜ tidak ada → fetch online"
        rev_str = "🔄 reverse=True" if rel.get("reverse") else "  reverse=False"
        print(f"   {rel['label']:<12} : {rel['geojson_file']:<40} {status}  |  {rev_str}")
    print()

    stops_available = os.path.exists(STOPS_FILE)
    if not stops_available:
        print(f"⚠️  {STOPS_FILE} tidak ditemukan — trim terminus dinonaktifkan")
    else:
        print(f"📂 stops.txt ditemukan — trim terminus aktif")
    print()

    all_shape_data = {}

    for rel_cfg in RELATIONS:
        shape_id   = rel_cfg["shape_id"]
        label      = rel_cfg["label"]
        do_reverse = rel_cfg.get("reverse", False)

        print(f"{'─'*55}")
        print(f"🔄  Memproses : {label}  {'(reverse aktif)' if do_reverse else ''}")
        print(f"{'─'*55}")

        try:
            coords      = get_coords(rel_cfg)
            n_raw       = len(coords)
            total_m_raw = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            # ── Step 1: Trim terminus ──────────────────────────────────────────
            trimmed = False
            if stops_available and rel_cfg.get("first_stop_id") and rel_cfg.get("last_stop_id"):
                try:
                    f_lat, f_lon, f_name = load_stop_coords(STOPS_FILE, rel_cfg["first_stop_id"])
                    l_lat, l_lon, l_name = load_stop_coords(STOPS_FILE, rel_cfg["last_stop_id"])

                    coords_trimmed, idx_f, idx_l, d_f, d_l = trim_shape_to_terminus(
                        coords, f_lat, f_lon, l_lat, l_lon
                    )
                    trimmed = True

                    print(f"\n  ✂️  Trim terminus:")
                    print(f"     Stop pertama : {rel_cfg['first_stop_id']} — {f_name}")
                    print(f"       indeks {idx_f} | jarak ke shape: {d_f:.1f} m")
                    print(f"     Stop terakhir: {rel_cfg['last_stop_id']} — {l_name}")
                    print(f"       indeks {idx_l} | jarak ke shape: {d_l:.1f} m")
                    print(f"     Titik dipotong: {idx_f} di awal, {n_raw - 1 - idx_l} di akhir")

                    coords = coords_trimmed

                except ValueError as e:
                    print(f"  ⚠️  Trim gagal: {e} — pakai koordinat penuh")

            # ── Step 2: Reverse (setelah trim, sebelum tulis) ──────────────────
            if do_reverse:
                coords = reverse_coords(coords)
                print(f"\n  🔄  Reverse selesai:")
                print(f"     Titik pertama (baru) : {coords[0]}")
                print(f"     Titik terakhir (baru): {coords[-1]}")

            if not coords:
                print(f"  ❌ Tidak ada koordinat!")
                continue

            all_shape_data[shape_id] = coords

            total_m = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            print(f"\n  📊 Hasil:")
            print(f"     Shape ID       : {shape_id}")
            print(f"     Titik (raw)    : {n_raw:,}")
            print(f"     Titik (final)  : {len(coords):,}  "
                  f"({'trim+reverse' if trimmed and do_reverse else 'trim' if trimmed else 'reverse' if do_reverse else 'tanpa trim/reverse'})")
            print(f"     Jarak raw      : {total_m_raw:,.1f} m  ({total_m_raw/1000:.3f} km)")
            print(f"     Jarak final    : {total_m:,.1f} m  ({total_m/1000:.3f} km)")
            print(f"     Titik pertama  : {coords[0]}")
            print(f"     Titik terakhir : {coords[-1]}")

            json_path, _ = save_json_ref(shape_id, rel_cfg["relation_id"], coords)
            print(f"     JSON referensi : {json_path}")

        except Exception as e:
            print(f"  ❌ Error: {e}")
            import traceback; traceback.print_exc()

        time.sleep(1)

    if not all_shape_data:
        print("\n⚠️  Tidak ada shape yang berhasil. shapes.txt tidak dibuat.")
        return

    rows = write_shapes_txt(all_shape_data, OUTPUT_SHAPES)

    print(f"\n{'='*55}")
    print(f"✅ shapes.txt berhasil dibuat!")
    print(f"   Path    : {OUTPUT_SHAPES}")
    print(f"   Kolom   : shape_id, shape_pt_lat, shape_pt_lon, shape_pt_sequence")
    print(f"   Rows    : {len(rows):,}")
    print()
    print(f"   {'Shape ID':<22} {'Titik':>8}  {'Jarak (m)':>12}  {'Jarak (km)':>12}  {'Reverse':>8}")
    print(f"   {'─'*65}")
    for rel_cfg in RELATIONS:
        sid    = rel_cfg["shape_id"]
        coords = all_shape_data.get(sid)
        if not coords:
            continue
        total_m = sum(
            haversine_m(coords[i-1][0], coords[i-1][1],
                        coords[i][0],   coords[i][1])
            for i in range(1, len(coords))
        )
        rev_str = "✅" if rel_cfg.get("reverse") else "—"
        print(f"   {sid:<22} {len(coords):>8,}  "
              f"{total_m:>12,.1f}  {total_m/1000:>12.3f}  {rev_str:>8}")

if __name__ == "__main__":
    main()

 OSM → GTFS shapes.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1
 2026-05-03 22:31:49
 shape_dist_traveled : tidak dihitung — gunakan gtfs_kit setelahnya
 Reverse             : per konfigurasi RELATIONS

 Prioritas sumber data:
   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada
   2. Online Overpass API               — jika tidak ada

   DKA → JTM    : data/shapes/relation_16079479.geojson    ✅ ada → pakai GeoJSON  |    reverse=False
   JTM → DKA    : data/shapes/relation_16079478.geojson    ✅ ada → pakai GeoJSON  |    reverse=False
   DKA → HAR    : data/shapes/relation_16036440_fixed.geojson ✅ ada → pakai GeoJSON  |    reverse=False
   HAR → DKA    : data/shapes/relation_16036441_fixed.geojson ✅ ada → pakai GeoJSON  |    reverse=False

📂 stops.txt ditemukan — trim terminus aktif

───────────────────────────────────────────────────────
🔄  Memproses : DKA → JTM  
───────────────────────────────────────────────────────
  📂 Mode: GeoJSON lokal
  → Membaca: data/shapes/rela

In [23]:
# Build field shape_dist_traveled with gtfs_kit (gtfs_kit.append_dist_to_shapes(feed))
import gtfs_kit as gk

feed = gk.read_feed('.', dist_units='m') # Pastikan satuan jarak dalam meter secara konsisten
feed = gk.append_dist_to_shapes(feed)
feed.shapes.to_csv(
    "shapes.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)

feed.shapes

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
733,shape_DKA_HAR,-6.204851,106.825525,0,0.000000
734,shape_DKA_HAR,-6.205027,106.826366,1,95.130649
735,shape_DKA_HAR,-6.205126,106.826840,2,148.761961
736,shape_DKA_HAR,-6.205191,106.827174,3,186.338210
737,shape_DKA_HAR,-6.205226,106.827389,4,210.520585
...,...,...,...,...,...
728,shape_JTM_DKA,-6.205179,106.827396,370,27062.977750
729,shape_JTM_DKA,-6.205144,106.827182,371,27087.014556
730,shape_JTM_DKA,-6.205080,106.826850,372,27124.467072
731,shape_JTM_DKA,-6.205061,106.826757,373,27134.960195


#### Cek shapes.txt untuk **`shape_DKA_JTM`** 

In [24]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_DKA_JTM
TARGET_TRIP_ID = 'BK-WD-DKA-JTM-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : BK-WD-DKA-JTM-001
✅ Shape : shape_DKA_JTM
✅ Jumlah titik shape: 358


In [25]:
m

#### Cek shapes.txt untuk **`shape_JTM_DKA`** 

In [26]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_JTM_DKA
TARGET_TRIP_ID = 'BK-WD-JTM-DKA-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : BK-WD-JTM-DKA-001
✅ Shape : shape_JTM_DKA
✅ Jumlah titik shape: 375


In [27]:
m

#### Cek shapes.txt untuk **`shape_DKA_HAR`** 

In [28]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_DKA_HAR
TARGET_TRIP_ID = 'CB-WD-DKA-HAR-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : CB-WD-DKA-HAR-001
✅ Shape : shape_DKA_HAR
✅ Jumlah titik shape: 395


In [29]:
m

#### Cek shapes.txt untuk **`shape_HAR_DKA`** 

In [30]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_HAR_DKA
TARGET_TRIP_ID = 'CB-WD-HAR-DKA-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : CB-WD-HAR-DKA-001
✅ Shape : shape_HAR_DKA
✅ Jumlah titik shape: 393


In [31]:
m

#### Build `shape_DKA_JTM`, `shape_JTM_DKA`, `shape_DKA_HAR`, `shape_HAR_DKA` with haversine formula for `shape_dist_traveled`

In [32]:
"""
Build file shapes.txt menggunakan data geometri jalur rel LRT Jabodebek yang diperoleh dari GeoJSON lokal atau melalui API Overpass (OpenStreetMap).

Shape yang dihasilkan:
  - shape_DKA_JTM : Dukuh Atas Bank Syariah Indonesia → Jati Mulya
  - shape_JTM_DKA : Jati Mulya → Dukuh Atas Bank Syariah Indonesia
  - shape_DKA_HAR : Dukuh Atas Bank Syariah Indonesia → Harjamukti
  - shape_HAR_DKA : Harjamukti → Dukuh Atas Bank Syariah Indonesia

Catatan:
  - File GeoJSON bisa didapatkan dari https://overpass-turbo.eu/ lakukan query untuk relation_id jalur tertentu kemudian download sebagai GeoJSON.
  - Terdapat proses trimming untuk memotong segmen pada stasiun terminus (awal & akhir) agar jalur rel tepat dimulai dari stop_id pertama.
  - Jika koordinat hasil parsing terbalik (arah OSM berlawanan dengan arah trip), aktifkan reverse=True pada konfigurasi RELATIONS.
  - shape_dist_traveled dihitung menggunakan rumus haversine dengan satuan Meter SETELAH proses reverse (jika aktif).

MODE OPERASI (otomatis dipilih berdasarkan file yang tersedia):
  1. GeoJSON lokal  → jika file .geojson tersedia (dari overpass-turbo.eu)
  2. Online         → fetch dari Overpass API
"""

import requests
import csv
import math
import json
import os
import time
from datetime import datetime

# File path
OUTPUT_SHAPES = "shapes.txt"
STOPS_FILE    = "stops.txt"

# Terminus stop_id per shape
# first_stop_id : stop pertama yang dilayani trip (platform)
# last_stop_id  : stop terakhir yang dilayani trip (platform)
# reverse       : True jika koordinat OSM berlawanan arah dengan trip
RELATIONS = [
    {
        "relation_id"  : 16079479, # relation_id OSM untuk jalur LRT Jabodebek DKA-JTM
        "shape_id"     : "shape_DKA_JTM", # shape_id untuk jalur LRT Jabodebek DKA-JTM
        "label"        : "DKA → JTM", 
        "geojson_file" : "data/shapes/relation_16079479.geojson", # File GeoJSON untuk jalur LRT Jabodebek DKA-JTM
        "first_stop_id": "GDKA01", # platform DKA arah JTM
        "last_stop_id" : "GJTM01", # platform JTM arah dari DKA
        "reverse"      : False,    # koordinat OSM sudah benar
    },
    {
        "relation_id"  : 16079478, # relation_id OSM untuk jalur LRT Jabodebek JTM-DKA
        "shape_id"     : "shape_JTM_DKA", # shape_id untuk jalur LRT Jabodebek JTM-DKA
        "label"        : "JTM → DKA",
        "geojson_file" : "data/shapes/relation_16079478.geojson", # File GeoJSON untuk jalur LRT Jabodebek JTM-DKA
        "first_stop_id": "GJTM02", # platform JTM arah DKA
        "last_stop_id" : "GDKA02", # platform DKA arah dari JTM
        "reverse"      : False,   # koordinat OSM sudah benar
    },
    {
        "relation_id"  : 16036440, # relation_id OSM untuk jalur LRT Jabodebek DKA-HAR
        "shape_id"     : "shape_DKA_HAR", # shape_id untuk jalur LRT Jabodebek DKA-HAR
        "label"        : "DKA → HAR",
        "geojson_file" : "data/shapes/relation_16036440_fixed.geojson", # File GeoJSON untuk jalur LRT Jabodebek DKA-HAR
        "first_stop_id": "GDKA01", # platform DKA arah HAR
        "last_stop_id" : "GHAR01", # platform HAR arah dari DKA
        "reverse"      : False,   # koordinat OSM sudah benar
    },
    {
        "relation_id"  : 16036441, # relation_id OSM untuk jalur LRT Jabodebek HAR-DKA
        "shape_id"     : "shape_HAR_DKA", # shape_id untuk jalur LRT Jabodebek HAR-DKA
        "label"        : "HAR → DKA",
        "geojson_file" : "data/shapes/relation_16036441_fixed.geojson", # File GeoJSON untuk jalur LRT Jabodebek HAR-DKA
        "first_stop_id": "GHAR02", # platform HAR arah DKA
        "last_stop_id" : "GDKA02", # platform DKA arah dari HAR
        "reverse"      : False,   # koordinat OSM sudah benar
    },
]

OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

REQUEST_HEADERS = {
    "User-Agent"  : "GTFS-Builder/1.0 (LRT Jabodebek GTFS)",
    "Accept"      : "application/json",
    "Content-Type": "application/x-www-form-urlencoded",
}

# Rumus Haversine 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam METER."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Load stop koordinat 
def load_stop_coords(stops_file, stop_id):
    """
    Baca (lat, lon, stop_name) dari stops.txt berdasarkan stop_id.
    Raise ValueError jika stop_id tidak ditemukan.
    """
    with open(stops_file, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["stop_id"] == stop_id:
                return (
                    float(row["stop_lat"]),
                    float(row["stop_lon"]),
                    row.get("stop_name", stop_id),
                )
    raise ValueError(f"stop_id '{stop_id}' tidak ditemukan di {stops_file}")


# Trim terminus 
def find_nearest_index(coords, target_lat, target_lon):
    """Cari indeks titik di coords yang paling dekat ke (target_lat, target_lon)."""
    best_idx  = 0
    best_dist = float("inf")
    for i, (lat, lon) in enumerate(coords):
        d = haversine_m(lat, lon, target_lat, target_lon)
        if d < best_dist:
            best_dist = d
            best_idx  = i
    return best_idx, best_dist


def trim_shape_to_terminus(coords, first_lat, first_lon, last_lat, last_lon):
    """
    Potong coords agar:
      - Dimulai dari titik terdekat ke stop pertama (first_stop)
      - Berakhir di titik terdekat ke stop terakhir (last_stop)

    Return: (coords_trimmed, idx_first, idx_last, dist_first_m, dist_last_m)
    """
    idx_first, dist_first = find_nearest_index(coords, first_lat, first_lon)
    idx_last,  dist_last  = find_nearest_index(coords, last_lat,  last_lon)

    if idx_first > idx_last:
        idx_first, idx_last = idx_last, idx_first

    return (
        coords[idx_first : idx_last + 1],
        idx_first, idx_last,
        dist_first, dist_last,
    )


# Reverse koordinat 
def reverse_coords(coords):
    """
    Balik urutan koordinat dan reset shape_pt_sequence dari 0.

    Digunakan jika koordinat hasil parsing OSM berlawanan arah dengan trip.
    Proses ini dilakukan SETELAH trim terminus dan SEBELUM hitung shape_dist_traveled
    agar:
      1. shape_pt_sequence tetap ascending dari 0
      2. shape_dist_traveled dihitung dari titik awal yang benar
      3. Titik pertama = platform terminus awal trip

    Contoh:
      Sebelum reverse: JTM(0) → ... → DKA(n)  ← arah OSM
      Setelah reverse: DKA(0) → ... → JTM(n)  ← arah trip DKA→JTM
    """
    return list(reversed(coords))


# GeoJSON Parser (mode offline) 
def extract_coords_from_geojson(geojson_path):
    """
    Parse GeoJSON dari overpass-turbo.eu → list of (lat, lon).

    GeoJSON dari overpass-turbo berisi FeatureCollection dengan:
      - LineString       : satu segmen way
      - MultiLineString  : beberapa segmen way
      - Point            : node (diabaikan)
    """
    print(f"  → Membaca: {geojson_path}")
    with open(geojson_path, "r", encoding="utf-8") as f:
        geojson = json.load(f)

    segments = []
    n_point  = 0

    for feature in geojson.get("features", []):
        geom  = feature.get("geometry", {})
        gtype = geom.get("type", "")

        if gtype == "LineString":
            segments.append([(c[1], c[0]) for c in geom["coordinates"]])
        elif gtype == "MultiLineString":
            for line in geom["coordinates"]:
                segments.append([(c[1], c[0]) for c in line])
        elif gtype == "Point":
            n_point += 1

    if not segments:
        raise ValueError("Tidak ada LineString/MultiLineString di GeoJSON!")

    print(f"  ✅ {len(segments)} segmen ditemukan, {n_point} point diabaikan")
    ordered = _chain_segments(segments)
    print(f"  ✅ Chain selesai → {len(ordered):,} titik koordinat")
    return ordered


def _chain_segments(segments):
    """
    Sambungkan list segmen menjadi satu jalur berurutan.
    Algoritma greedy berdasarkan jarak ujung-ke-ujung terdekat.
    Gap > 50m diberi warning tapi tetap disambung.
    """
    SNAP_THRESHOLD_M = 50.0
    remaining = [list(seg) for seg in segments]
    result    = remaining.pop(0)

    while remaining:
        tail      = result[-1]
        best_idx  = None
        best_dist = float("inf")
        best_rev  = False

        for i, seg in enumerate(remaining):
            d_head = haversine_m(tail[0], tail[1], seg[0][0],  seg[0][1])
            d_tail = haversine_m(tail[0], tail[1], seg[-1][0], seg[-1][1])

            if d_head < best_dist:
                best_dist, best_idx, best_rev = d_head, i, False
            if d_tail < best_dist:
                best_dist, best_idx, best_rev = d_tail, i, True

        seg = remaining.pop(best_idx)
        if best_rev:
            seg = list(reversed(seg))
        if best_dist > SNAP_THRESHOLD_M:
            print(f"  ⚠️  Gap {best_dist:.1f}m — tetap disambung")

        skip = 1 if haversine_m(tail[0], tail[1], seg[0][0], seg[0][1]) < 1.0 else 0
        result.extend(seg[skip:])

    return result


#  Overpass API OSM (mode online)
def try_request(mirror, query):
    """POST raw dulu, fallback ke GET params."""
    try:
        print(f"    [POST raw] ...")
        resp = requests.post(mirror, data=query, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_post:
        print(f"    ⚠️  POST gagal: {e_post}")

    try:
        print(f"    [GET params] ...")
        resp = requests.get(mirror, params={"data": query}, headers=REQUEST_HEADERS, timeout=90)
        resp.raise_for_status()
        return resp.json()
    except Exception as e_get:
        print(f"    ⚠️  GET gagal: {e_get}")
        raise e_get


def fetch_relation_online(relation_id):
    """Fetch relation dari Overpass API, coba semua mirror secara berurutan."""
    query = (
        f"[out:json][timeout:90];"
        f"relation({relation_id});"
        f"out body;>;"
        f"out skel qt;"
    )
    last_err = None
    for mirror in OVERPASS_MIRRORS:
        print(f"  → Mencoba {mirror} ...")
        try:
            data = try_request(mirror, query)
            print(f"  ✅ Berhasil! {len(data['elements'])} elements")
            return data
        except Exception as e:
            print(f"  ✗  Mirror gagal\n")
            last_err = e
            time.sleep(3)
    raise RuntimeError(f"Semua mirror gagal: {last_err}")


def extract_coords_from_overpass(data):
    """Parse Overpass JSON → list of (lat, lon)."""
    nodes = {e["id"]: (e["lat"], e["lon"])
             for e in data["elements"] if e["type"] == "node"}
    ways  = {e["id"]: e["nodes"]
             for e in data["elements"] if e["type"] == "way"}
    rel   = next((e for e in data["elements"] if e["type"] == "relation"), None)

    if not rel:
        raise ValueError("Relation tidak ditemukan!")

    tags        = rel.get("tags", {})
    way_members = [m for m in rel["members"] if m["type"] == "way"]
    print(f"\n  📋 Relation: {tags.get('name','N/A')} | {len(way_members)} ways")

    ordered_coords = []
    last_node_id   = None
    skipped        = 0

    for member in way_members:
        way_id = member["ref"]
        role   = member.get("role", "")

        if way_id not in ways or not ways[way_id]:
            skipped += 1
            continue

        way_nodes  = ways[way_id]
        first_node = way_nodes[0]
        last_node  = way_nodes[-1]

        if last_node_id is None:
            reverse = (role == "backward")
        else:
            if last_node_id == first_node:
                reverse = False
            elif last_node_id == last_node:
                reverse = True
            else:
                if first_node in nodes and last_node in nodes and last_node_id in nodes:
                    c       = nodes[last_node_id]
                    d_first = haversine_m(c[0], c[1], nodes[first_node][0], nodes[first_node][1])
                    d_last  = haversine_m(c[0], c[1], nodes[last_node][0],  nodes[last_node][1])
                    reverse = d_last < d_first
                else:
                    reverse = (role == "backward")

        seq = list(reversed(way_nodes)) if reverse else way_nodes
        for j, node_id in enumerate(seq):
            if node_id not in nodes:
                continue
            coord = nodes[node_id]
            if ordered_coords and j == 0 and ordered_coords[-1] == coord:
                continue
            ordered_coords.append(coord)
        last_node_id = seq[-1]

    if skipped:
        print(f"  ⚠️  {skipped} ways dilewati")

    return ordered_coords


# Selector source data 
def get_coords(rel_cfg):
    """
    Pilih sumber data secara otomatis:
      1. GeoJSON lokal → jika file .geojson ada
      2. Online        → fetch dari Overpass API
    """
    geojson_file = rel_cfg.get("geojson_file", "")
    if geojson_file and os.path.exists(geojson_file):
        print(f"  📂 Mode: GeoJSON lokal")
        return extract_coords_from_geojson(geojson_file)

    print(f"  🌐 Mode: Online Overpass API")
    data = fetch_relation_online(rel_cfg["relation_id"])
    return extract_coords_from_overpass(data)


# Output 
def write_shapes_txt(all_shape_data, output_file):
    """
    Tulis shapes.txt dengan kolom shape_dist_traveled.

    Urutan proses per shape:
      1. Trim terminus (sudah dilakukan sebelumnya di main)
      2. Reverse jika dikonfigurasi (sudah dilakukan sebelumnya di main)
      3. Reset shape_pt_sequence dari 0
      4. Hitung shape_dist_traveled kumulatif dari titik pertama (dist=0)

    Satuan: meter, presisi penuh (tidak dibulatkan).
    """
    fieldnames = [
        "shape_id", "shape_pt_lat", "shape_pt_lon",
        "shape_pt_sequence", "shape_dist_traveled",
    ]
    rows = []
    for shape_id, coords in all_shape_data.items():
        dist = 0.0
        prev = None
        for seq, (lat, lon) in enumerate(coords):
            if prev:
                dist += haversine_m(prev[0], prev[1], lat, lon)
            rows.append({
                "shape_id"           : shape_id,
                "shape_pt_lat"       : f"{lat:.7f}",
                "shape_pt_lon"       : f"{lon:.7f}",
                "shape_pt_sequence"  : seq,
                "shape_dist_traveled": f"{dist}",
            })
            prev = (lat, lon)

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return rows


def save_json_ref(shape_id, relation_id, coords, total_m):
    """Simpan JSON referensi koordinat per shape."""
    json_path = f"shape_ref_{shape_id}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({
            "shape_id"     : shape_id,
            "relation_id"  : relation_id,
            "total_points" : len(coords),
            "total_dist_m" : round(total_m, 1),
            "total_dist_km": round(total_m / 1000, 3),
            "coords"       : [{"lat": lat, "lon": lon} for lat, lon in coords]
        }, f, indent=2)
    return json_path


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print(" OSM → GTFS shapes.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Satuan shape_dist_traveled : Meter (haversine)")
    print(f" Trim terminus              : aktif jika stops.txt tersedia")
    print(f" Reverse                    : per konfigurasi RELATIONS")
    print("=" * 60)
    print()
    print(" Prioritas sumber data:")
    print("   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada")
    print("   2. Online Overpass API               — jika tidak ada")
    print()
    for rel in RELATIONS:
        ada     = os.path.exists(rel["geojson_file"])
        status  = "✅ ada → pakai GeoJSON" if ada else "⬜ tidak ada → fetch online"
        rev_str = "🔄 reverse=True" if rel.get("reverse") else "  reverse=False"
        print(f"   {rel['label']:<12} : {rel['geojson_file']:<40} {status}  |  {rev_str}")
    print()

    stops_available = os.path.exists(STOPS_FILE)
    if stops_available:
        print(f"📂 {STOPS_FILE} ditemukan — trim terminus aktif")
    else:
        print(f"⚠️  {STOPS_FILE} tidak ditemukan — trim terminus dinonaktifkan")
    print()

    all_shape_data = {}

    for rel_cfg in RELATIONS:
        shape_id   = rel_cfg["shape_id"]
        label      = rel_cfg["label"]
        do_reverse = rel_cfg.get("reverse", False)

        print(f"{'─'*55}")
        print(f"🔄  Memproses : {label}  {'(reverse aktif)' if do_reverse else ''}")
        print(f"{'─'*55}")

        try:
            coords      = get_coords(rel_cfg)
            n_raw       = len(coords)
            total_m_raw = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            # Step 1: Trim terminus 
            trimmed = False
            if (stops_available
                    and rel_cfg.get("first_stop_id")
                    and rel_cfg.get("last_stop_id")):
                try:
                    f_lat, f_lon, f_name = load_stop_coords(
                        STOPS_FILE, rel_cfg["first_stop_id"]
                    )
                    l_lat, l_lon, l_name = load_stop_coords(
                        STOPS_FILE, rel_cfg["last_stop_id"]
                    )
                    coords, idx_f, idx_l, d_f, d_l = trim_shape_to_terminus(
                        coords, f_lat, f_lon, l_lat, l_lon
                    )
                    trimmed = True

                    print(f"\n  ✂️  Trim terminus:")
                    print(f"     Stop pertama  : {rel_cfg['first_stop_id']} — {f_name}")
                    print(f"       idx={idx_f} | jarak ke shape: {d_f:.1f} m")
                    print(f"     Stop terakhir : {rel_cfg['last_stop_id']} — {l_name}")
                    print(f"       idx={idx_l} | jarak ke shape: {d_l:.1f} m")
                    print(f"     Titik dipotong: {idx_f} di awal, "
                          f"{n_raw - 1 - idx_l} di akhir")

                except ValueError as e:
                    print(f"  ⚠️  Trim gagal: {e} — pakai koordinat penuh")

            # Step 2: Reverse (setelah trim, sebelum hitung dist)
            if do_reverse:
                coords = reverse_coords(coords)
                print(f"\n  🔄  Reverse selesai:")
                print(f"     Titik pertama (baru) : {coords[0]}")
                print(f"     Titik terakhir (baru): {coords[-1]}")

            # Step 3: Hitung total jarak (setelah reverse) 
            if not coords:
                print(f"  ❌ Tidak ada koordinat!")
                continue

            all_shape_data[shape_id] = coords

            total_m = sum(
                haversine_m(coords[i-1][0], coords[i-1][1],
                            coords[i][0],   coords[i][1])
                for i in range(1, len(coords))
            )

            print(f"\n  📊 Hasil:")
            print(f"     Shape ID        : {shape_id}")
            print(f"     Titik raw       : {n_raw:,}")
            print(f"     Titik final     : {len(coords):,}  "
                  f"({'trim+reverse' if trimmed and do_reverse else 'trim' if trimmed else 'reverse' if do_reverse else 'tanpa trim/reverse'})")
            print(f"     Jarak raw       : {total_m_raw:,.1f} m  ({total_m_raw/1000:.3f} km)")
            print(f"     Jarak final     : {total_m:,.1f} m  ({total_m/1000:.3f} km)")
            print(f"     Titik pertama   : {coords[0]}")
            print(f"     Titik terakhir  : {coords[-1]}")

            json_path = save_json_ref(shape_id, rel_cfg["relation_id"], coords, total_m)
            print(f"     JSON referensi  : {json_path}")

        except Exception as e:
            print(f"  ❌ Error: {e}")
            import traceback; traceback.print_exc()

        time.sleep(1)

    if not all_shape_data:
        print("\n⚠️  Tidak ada shape yang berhasil. shapes.txt tidak dibuat.")
        return

    rows = write_shapes_txt(all_shape_data, OUTPUT_SHAPES)

    print(f"\n{'='*55}")
    print(f"✅ shapes.txt berhasil dibuat!")
    print(f"   Path    : {OUTPUT_SHAPES}")
    print(f"   Kolom   : shape_id, shape_pt_lat, shape_pt_lon,")
    print(f"             shape_pt_sequence, shape_dist_traveled")
    print(f"   Rows    : {len(rows):,}")
    print()
    print(f"   {'Shape ID':<22} {'Titik':>8}  {'Jarak (m)':>12}  {'Jarak (km)':>12}  {'Reverse':>8}")
    print(f"   {'─'*65}")
    for rel_cfg in RELATIONS:
        sid    = rel_cfg["shape_id"]
        coords = all_shape_data.get(sid)
        if not coords:
            continue
        total_m = sum(
            haversine_m(coords[i-1][0], coords[i-1][1],
                        coords[i][0],   coords[i][1])
            for i in range(1, len(coords))
        )
        rev_str = "✅" if rel_cfg.get("reverse") else "—"
        print(f"   {sid:<22} {len(coords):>8,}  "
              f"{total_m:>12,.1f}  {total_m/1000:>12.3f}  {rev_str:>8}")


if __name__ == "__main__":
    main()

 OSM → GTFS shapes.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1
 2026-05-03 22:31:54
 Satuan shape_dist_traveled : Meter (haversine)
 Trim terminus              : aktif jika stops.txt tersedia
 Reverse                    : per konfigurasi RELATIONS

 Prioritas sumber data:
   1. GeoJSON lokal (overpass-turbo.eu) — jika file ada
   2. Online Overpass API               — jika tidak ada

   DKA → JTM    : data/shapes/relation_16079479.geojson    ✅ ada → pakai GeoJSON  |    reverse=False
   JTM → DKA    : data/shapes/relation_16079478.geojson    ✅ ada → pakai GeoJSON  |    reverse=False
   DKA → HAR    : data/shapes/relation_16036440_fixed.geojson ✅ ada → pakai GeoJSON  |    reverse=False
   HAR → DKA    : data/shapes/relation_16036441_fixed.geojson ✅ ada → pakai GeoJSON  |    reverse=False

📂 stops.txt ditemukan — trim terminus aktif

───────────────────────────────────────────────────────
🔄  Memproses : DKA → JTM  
───────────────────────────────────────────────────────
  📂 Mo

#### Cek shapes.txt untuk **`shape_DKA_JTM`** 

In [33]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_DKA_JTM
TARGET_TRIP_ID = 'BK-WD-DKA-JTM-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : BK-WD-DKA-JTM-001
✅ Shape : shape_DKA_JTM
✅ Jumlah titik shape: 358


In [34]:
m

#### Cek shapes.txt untuk **`shape_JTM_DKA`** 

In [35]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_JTM_DKA
TARGET_TRIP_ID = 'BK-WD-JTM-DKA-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : BK-WD-JTM-DKA-001
✅ Shape : shape_JTM_DKA
✅ Jumlah titik shape: 375


In [36]:
m

#### Cek shapes.txt untuk **`shape_DKA_HAR`** 

In [37]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_DKA_HAR
TARGET_TRIP_ID = 'CB-WD-DKA-HAR-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : CB-WD-DKA-HAR-001
✅ Shape : shape_DKA_HAR
✅ Jumlah titik shape: 395


In [38]:
m

#### Cek shapes.txt untuk **`shape_HAR_DKA`** 

In [39]:
import pandas as pd
import folium

#File path
SHAPES_FILE = 'shapes.txt'
TRIPS_FILE  = 'trips.txt'

# Ambil salah satu trip dengan shape_HAR_DKA
TARGET_TRIP_ID = 'CB-WD-HAR-DKA-001'

shapes = pd.read_csv(SHAPES_FILE)
trips  = pd.read_csv(TRIPS_FILE)

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == TARGET_TRIP_ID]

if len(trip_row) == 0:
    print(f"❌ Trip {TARGET_TRIP_ID} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip  : {TARGET_TRIP_ID}")
    print(f"✅ Shape : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))

        # Buat peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {TARGET_TRIP_ID}"
        ).add_to(m)

        # Titik awal shape
        folium.Marker(
            coords[0],
            popup=f"START – {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        # Titik akhir shape
        folium.Marker(
            coords[-1],
            popup=f"END – {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Simpan
        output_file = f'map_shape_{TARGET_TRIP_ID}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip  : CB-WD-HAR-DKA-001
✅ Shape : shape_HAR_DKA
✅ Jumlah titik shape: 393


In [40]:
m

## stop_times.txt (Required)

File `stop_times.txt` mendefinisikan **waktu kedatangan dan keberangkatan moda transportasi di setiap stasiun/stasiun/stops untuk setiap perjalanan (trip)**. File ini adalah file terbesar dan terpenting dalam GTFS karena menghubungkan tiga entitas utama yaitu trip, stop, dan waktu menjadi jadwal yang lengkap. Setiap baris mewakili satu pemberhentian dalam satu perjalanan.

---

**Field yang Direferensikan di File Lain**

| Field | Direferensikan dari | Keterangan |
|---|---|---|
| `trip_id` | `trips.txt` | Setiap baris merujuk ke perjalanan tertentu |
| `stop_id` | `stops.txt` | Hanya boleh merujuk ke `location_type=0` (platform) |
| `shape_dist_traveled` | `shapes.txt` | Satuan harus konsisten dengan `shapes.shape_dist_traveled` |

---

**Skema kolom stop_times.txt**

| Field | Tipe Data | Status | Deskripsi |
|---|---|---|---|
| `trip_id` | Foreign ID → `trips.trip_id` | **Wajib** | ID perjalanan yang dilayani pada baris ini. |
| `arrival_time` | Time | **Kondisional** | Waktu tiba di stasiun dalam format `HH:MM:SS`. Untuk waktu melewati tengah malam gunakan jam lebih dari 24 — contoh `25:35:00` untuk 01:35 dini hari. **Wajib** untuk stasiun pertama dan terakhir dalam trip, serta jika `timepoint=1`. |
| `departure_time` | Time | **Kondisional** | Waktu berangkat dari stasiun dalam format `HH:MM:SS`. Aturan sama dengan `arrival_time`. Jika waktu tiba dan berangkat sama, isi kedua field dengan nilai yang sama. |
| `stop_id` | Foreign ID → `stops.stop_id` | **Kondisional** | ID stasiun yang dilayani. Harus merujuk ke `location_type=0` (platform). **Wajib** jika `location_group_id` dan `location_id` tidak didefinisikan. |
| `location_group_id` | Foreign ID → `location_groups` | **Kondisional** | ID grup lokasi untuk layanan on-demand. **Dilarang** jika `stop_id` atau `location_id` sudah didefinisikan. |
| `location_id` | Foreign ID → `locations.geojson` | **Kondisional** | ID lokasi GeoJSON untuk layanan on-demand. **Dilarang** jika `stop_id` atau `location_group_id` sudah didefinisikan. |
| `stop_sequence` | Non-negative integer | **Wajib** | Urutan stasiun dalam perjalanan. Nilai harus selalu meningkat sepanjang trip namun tidak harus berurutan berurutan. |
| `stop_headsign` | Text | Opsional | Teks tujuan yang ditampilkan di papan kendaraan khusus untuk stasiun ini — menggantikan `trips.trip_headsign` jika berbeda di tengah perjalanan. |
| `pickup_type` | Enum | **Kondisional** | Metode naik penumpang. Lihat tabel nilai enum di bawah. |
| `drop_off_type` | Enum | **Kondisional** | Metode turun penumpang. Lihat tabel nilai enum di bawah. |
| `continuous_pickup` | Enum | **Kondisional** | Apakah penumpang bisa naik di sembarang titik dari stasiun ini ke stasiun berikutnya. Jika diisi, menggantikan nilai di `routes.txt`. |
| `continuous_drop_off` | Enum | **Kondisional** | Apakah penumpang bisa turun di sembarang titik dari stasiun ini ke stasiun berikutnya. Jika diisi, menggantikan nilai di `routes.txt`. |
| `shape_dist_traveled` | Non-negative float | Opsional | Jarak aktual yang ditempuh dari stasiun pertama hingga stasiun ini, dalam satuan yang sama dengan `shapes.txt`. Nilai harus selalu meningkat sesuai `stop_sequence`. |
| `timepoint` | Enum | Opsional | Tingkat keakuratan waktu. `1` = waktu pasti, `0` = waktu perkiraan/interpolasi. Jika tidak diisi semua waktu dianggap pasti. |
| `pickup_booking_rule_id` | Foreign ID → `booking_rules` | Opsional | ID aturan pemesanan untuk naik penumpang. Disarankan jika `pickup_type=2`. |
| `drop_off_booking_rule_id` | Foreign ID → `booking_rules` | Opsional | ID aturan pemesanan untuk turun penumpang. Disarankan jika `drop_off_type=2`. |

---

**`pickup_type`** & **`drop_off_type`**

| Nilai | Arti |
|:---:|---|
| `0` atau kosong | Naik/turun terjadwal reguler | 
| `1` | Tidak tersedia |
| `2` | Perlu telepon operator |
| `3` | Perlu koordinasi dengan pengemudi | 

**`timepoint`**

| Nilai | Arti |
|:---:|---|
| `1` atau kosong | Waktu **pasti** | 
| `0` | Waktu **perkiraan** / interpolasi |

### Panduan Pengisian `stop_times.txt` — LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

---

**`trip_id`**

Diisi sesuai dengan `trip_id` yang ada di `trips.txt`. Terdiri dari 4 jenis trip yaitu:

- `BK-WD/WE-DKA-JTM-xxx`
- `BK-WD/WE-JTM-DKA-xxx`
- `CB-WD/WE-DKA-HAR-xxx`
- `CB-WD/WE-HAR-DKA-xxx`

---

**`stop_id`**

Diisi dengan `stop_id` platform (`location_type=0`) dari setiap stasiun pemberhentian. LRT Jabodetabek memiliki empat arah perjalanan dan 4 jenis trip dengan mapping platform sebagai berikut:

**Full trip DKA → JTM** (`BK-WD/WE-DKA-JTM-xxx`)

| Stasiun | `stop_id` |
|---|---|
| DKA | `GDKA01` |
| SET | `GSET01` |
| RAS | `GRAS01` |
| KUA | `GKUA01` |
| PAN | `GPAN01` |
| CKK | `GCKK01` |
| CIL | `GCIL01` |
| CWG | `GCWG01` |
| HAL | `GHAL01` |
| JBU | `GJBU01` |
| CK1 | `GCK101` |
| CK2 | `GCK201` |
| BEK | `GBEK01` |
| JTM | `GJTM01` |

**Full trip JTM → DKA** (`BK-WD/WE-JTM-DKA-xxx`)

| Stasiun | `stop_id` |
|---|---|
| JTM | `GJTM02` |
| BEK | `GBEK02` |
| CK2 | `GCK202` |
| CK1 | `GCK102` |
| JBU | `GJBU02` |
| HAL | `GHAL02` |
| CWG | `GCWG02` |
| CIL | `GCIL02` |
| CKK | `GCKK02` |
| PAN | `GPAN02` |
| KUA | `GKUA02` |
| RAS | `GRAS02` |
| SET | `GSET02` |
| DKA | `GDKA02` |

**Full trip DKA → HAR** (`CB-WD/WE-DKA-HAR-xxx`)

| Stasiun | `stop_id` |
|---|---|
| DKA | `GDKA01` |
| SET | `GSET01` |
| RAS | `GRAS01` |
| KUA | `GKUA01` |
| PAN | `GPAN01` |
| CKK | `GCKK01` |
| CIL | `GCIL01` |
| CWG | `GCWG03` |
| TMI | `GTMI01` |
| KAM | `GKAM01` |
| CRC | `GCRC01` |
| HAR | `GHAR01` |


**Full trip HAR → DKA** (`CB-WD/WE-HAR-DKA-xxx`)

| Stasiun | `stop_id` |
|---|---|
| HAR | `GHAR02` |
| CRC | `GCRC02` |
| KAM | `GKAM02` |
| TMI | `GTMI02` |
| CWG | `GCWG04` |
| CIL | `GCIL02` |
| CKK | `GCKK02` |
| PAN | `GPAN02` |
| KUA | `GKUA02` |
| RAS | `GRAS02` |
| SET | `GSET02` |
| DKA | `GDKA02` |

---

**`arrival_time`** & **`departure_time`**

Referensi jadwal diambil dari [akun instagram resmi LRT Jabodebek](https://www.instagram.com/p/DUSKhaZjzpn/?igsh=ajMyZmV2N2d1ejZr) yang hanya menyediakan `departure_time`. Dwell time aktual LRT Jabodebek adalah **30 detik**, sehingga:

| Kondisi | `arrival_time` |
|---|---|
| Stasiun pertama (`stop_sequence=0`) | `= departure_time` |
| Stasiun tengah | `= departure_time - 30 detik` |
| Stasiun terakhir | `= departure_time` |

Format waktu mengikuti spesifikasi GTFS untuk waktu melewati tengah malam jam tetap bertambah dan tidak direset ke `00`, contoh `25:05:00` untuk pukul 01:05 dini hari.

---

**`stop_sequence`**

Diisi secara berurutan dimulai dari `0` untuk setiap trip, mengikuti urutan stasiun sesuai 4 jenis trip yang sudah didefinisikan di atas.

| Stop | `stop_sequence` |
|---|:---:|
| Stasiun pertama | `0` |
| Stasiun kedua | `1` |
| Stasiun ketiga | `2` |
| ... | ... |

---

**`pickup_type`** & **`drop_off_type`**

Diisi `0` untuk semua baris naik dan turun penumpang hanya di stasiun resmi sesuai jadwal.

---

**`continuous_pickup`** & **`continuous_drop_off`**

Dikosongkan nilai default `1` berlaku otomatis, artinya penumpang tidak bisa naik maupun turun di sembarang titik sepanjang jalur, hanya di stasiun resmi.

---

**`timepoint`**

Diisi `1` untuk semua baris karena LRT Jabodebek memiliki ketepatan waktu presisi dengan jadwal yang bersifat pasti bukan perkiraan.

---

**`stop_headsign`**

Diisi berdasarkan arah perjalanan trip:

|Lin| Arah | `stop_headsign` |
|---|---|---|
|Lin Bekasi| Ke Timur (DKA → JTM) | `Jati Mulya` |
|Lin Bekasi| Ke Barat (JTM → DKA) | `Dukuh Atas` |
|Lin Cibubur| Ke Selatan (DKA → HAR) | `Harjamukti` |
|Lin Cibubur| Ke Utara (HAR → DKA) | `Dukuh Atas` |

---

**`shape_dist_traveled`**

`shape_dist_traveled` merepresentasikan **jarak kumulatif aktual** yang ditempuh kendaraan dari titik awal shape hingga suatu `stop_id`.

Nilai ini dihitung secara **berurutan mengikuti stop_sequence**, sehingga setiap baris menunjukkan posisi relatif stop terhadap keseluruhan jalur. Nilai ini dihitung berdasarkan jarak antar `stop_id` ke `stop_id` yang berurutan.

**Pendekatan Perhitungan**

Terdapat **dua pendekatan umum** dalam menghitung `shape_dist_traveled`:

**1. Menggunakan Rumus Haversine (Manual)**

Pendekatan ini menghitung jarak antar dua titik koordinat di permukaan bumi berdasarkan model bola bumi.

```python
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam meter (Haversine)."""
    R = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))
```

Perhitungan dilakukan secara **kumulatif**:

* Stop 0 → `0`
* Stop 1 → jarak(0 → 1)
* Stop 2 → jarak(0 → 1) + jarak(1 → 2)
* dan seterusnya

**Kelebihan:**

* Sederhana dan mudah diimplementasikan (pandas / numpy)
* Tidak memerlukan library GIS
* Lebih stabil terhadap perbedaan presisi (minim mismatch dengan `shapes.txt`)


**2. Menggunakan `gtfs_kit.append_dist_to_stop_times()`**

Fungsi dari library `gtfs_kit` menghitung `shape_dist_traveled` dengan pendekatan spasial yang lebih akurat:

**Proses:**

* Koordinat diubah dari **WGS84 (lat/lon)** ke sistem proyeksi berbasis meter (UTM)
* Dibentuk:

  * LineString untuk shape
  * Point untuk setiap stop
    
* Jarak dihitung menggunakan:

  * `LineString.project(Point)`
  * menghasilkan posisi jarak kumulatif sepanjang shape

**Validasi hasil:**

* Jika jarak monotonik meningkat → digunakan langsung
* Jika tidak valid → diperbaiki dengan interpolasi berbasis waktu (fallback)

**Catatan & Kekurangan**

Pendekatan `gtfs_kit` dapat menghasilkan **perbedaan sangat kecil (floating point precision)** antara:

* `shape_dist_traveled` maksimum di `shapes.txt`
* `shape_dist_traveled` pada stop terakhir di `stop_times.txt`

Contoh:

```
14609.29264816948   (stop_times)
14609.292648169478  (shapes)
```

Walaupun selisihnya sangat kecil, hal ini dapat memicu warning:

**`trip_distance_exceeds_shape_distance_below_threshold`**

`Jarak antara titik terakhir shape dan stop terakhir lebih dari 0 tetapi kurang dari threshold (±11.1 meter)`

**Solusi yang Disarankan**

Untuk menghindari warning tersebut, lakukan **penyesuaian (force snap)**:

* Set nilai `shape_dist_traveled` pada **stop terakhir**
* Menjadi **persis sama** dengan nilai maksimum pada `shapes.txt`

Pendekatan ini memastikan:

* Konsistensi antar file GTFS
* Tidak ada mismatch akibat presisi floating point
* Validator GTFS tidak menghasilkan warning


**Catatan**

* Kedua metode diatas valid dalam GTFS
* Hasil bisa sedikit berbeda (biasanya sangat kecil)
* Yang penting:
  * nilai **monoton meningkat**
  * konsisten satuan (meter/km)

### Build `stop_times.txt` with gtfs_kit for shape_dist_traveled

In [41]:
"""
Build stop_times.txt berdasarkan jadwal keberangkatan LRT Jabodebek

departure_time  : langsung dari file jadwal keberangkatan
arrival_time    : departure_time - DWELL_TIME (detik), kecuali stasiun pertama & terakhir: arrival = departure

Format waktu arrival_time & departure_time pada stop_times.txt:
  - Waktu TIDAK direset ke 00:xx:xx saat melewati tengah malam
  - Melainkan terus bertambah: 24:xx:xx, 25:xx:xx, dst
  - Contoh: kereta berangkat 23:40 tiba 00:01 → ditulis 24:01:xx

Catatan:
  - shape_dist_traveled tidak dihitung di sini, akan dihitung menggunakan library gtfs_kit
  - Perhitungan shape_dist_traveled menggunakan gtfs_kit.append_dist_to_stop_times(feed) setelah file ini dibuat
  - Nilai shape_dist_traveled pada stop_times.txt perlu disesuaikan dengan shapes.txt, khususnya pada stop terakhir setiap trip agar sama dengan panjang 
    total shape, sehingga menghindari selisih kecil (floating point) dan warning pada validator GTFS (trip_distance_exceeds_shape_distance_below_threshold)
"""

import pandas as pd
import csv
import os
from datetime import datetime

# File path
OUTPUT_FILE = "stop_times.txt"

# Dwell time dalam detik
# arrival = departure - DWELL_TIME
# kecuali stasiun pertama & terakhir: arrival = departure
DWELL_TIME  = 30   # detik

# File jadwal keberangkatan lRT Jabodebek
SCHEDULE_FILES = {
    "DKA_JTM_WD": "data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekdays).csv",
    "DKA_JTM_WE": "data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekends).csv",
    "JTM_DKA_WD": "data/jadwal-keberangkatan/JatiMulya_DukuhAtas_(Weekdays).csv",
    "JTM_DKA_WE": "data/jadwal-keberangkatan/JatiMulya_DukuhAtas_(Weekends).csv",
    "DKA_HAR_WD": "data/jadwal-keberangkatan/DukuhAtas_Harjamukti_(Weekdays).csv",
    "DKA_HAR_WE": "data/jadwal-keberangkatan/DukuhAtas_Harjamukti_(Weekends).csv",
    "HAR_DKA_WD": "data/jadwal-keberangkatan/Harjamukti_DukuhAtas_(Weekdays).csv",
    "HAR_DKA_WE": "data/jadwal-keberangkatan/Harjamukti_DukuhAtas_(Weekends).csv",
}

# Mapping kode stasiun → stop_id platform
PLATFORM_MAP = {
    "DKA_JTM": {
        "DKA": "GDKA01", "SET": "GSET01", "RAS": "GRAS01",
        "KUA": "GKUA01", "PAN": "GPAN01", "CKK": "GCKK01",
        "CIL": "GCIL01", "CWG": "GCWG01", "HAL": "GHAL01",
        "JBU": "GJBU01", "CK1": "GCK101", "CK2": "GCK201",
        "BEK": "GBEK01", "JTM": "GJTM01",
    },
    "JTM_DKA": {
        "JTM": "GJTM02", "BEK": "GBEK02", "CK2": "GCK202",
        "CK1": "GCK102", "JBU": "GJBU02", "HAL": "GHAL02",
        "CWG": "GCWG02", "CIL": "GCIL02", "CKK": "GCKK02",
        "PAN": "GPAN02", "KUA": "GKUA02", "RAS": "GRAS02",
        "SET": "GSET02", "DKA": "GDKA02",
    },
    "DKA_HAR": {
        "DKA": "GDKA01", "SET": "GSET01", "RAS": "GRAS01",
        "KUA": "GKUA01", "PAN": "GPAN01", "CKK": "GCKK01",
        "CIL": "GCIL01", "CWG": "GCWG03", "TMI": "GTMI01",
        "KAM": "GKAM01", "CRC": "GCRC01", "HAR": "GHAR01",
    },
    "HAR_DKA": {
        "HAR": "GHAR02", "CRC": "GCRC02", "KAM": "GKAM02",
        "TMI": "GTMI02", "CWG": "GCWG04", "CIL": "GCIL02", 
        "CKK": "GCKK02", "PAN": "GPAN02", "KUA": "GKUA02",
        "RAS": "GRAS02", "SET": "GSET02", "DKA": "GDKA02",
    },
}


# Time utils
def parse_time(time_str):
    """
    Parse waktu HH:MM:SS ke total detik.
    Support jam > 24 untuk layanan melewati tengah malam (GTFS compliant).

    Contoh:
        "05:12:00" →  18720
        "23:40:30" →  85230
        "24:00:50" →  86450
    """
    parts = str(time_str).strip().split(":")
    h = int(parts[0])
    m = int(parts[1])
    s = int(parts[2]) if len(parts) > 2 else 0
    return h * 3600 + m * 60 + s


def format_time(total_seconds):
    """
    Format total detik ke HH:MM:SS sesuai spesifikasi GTFS.
    Jam TIDAK direset ke 00 saat melewati tengah malam melainkan terus bertambah (24:xx:xx, 25:xx:xx, dst).
    Contoh:
        85230 → "23:40:30"
        86450 → "24:00:50"  ✅ bukan "00:00:50"
        90000 → "25:00:00"
    """
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def fix_midnight_rollover(current_sec, prev_sec):
    """
    Koreksi midnight rollover sesuai spesifikasi GTFS.

    GTFS mengharuskan waktu selalu ASCENDING dalam satu trip.
    Jika current_sec < prev_sec, berarti CSV menyimpan waktu sebagai
    00:xx:xx setelah tengah malam — dikoreksi dengan +86400 detik.

    Contoh:
        prev    = 23:40:30 → 85230 detik
        current = 00:00:50 →    50 detik  ← lebih kecil → rollover!
        koreksi = 50 + 86400 = 86450      → "24:00:50"
    """
    while current_sec < prev_sec:
        current_sec += 86400
    return current_sec


def calc_arrival(departure_sec, stop_sequence, total_stops):
    """
    Hitung arrival_time dalam detik.

    Aturan:
      - Stasiun pertama  (seq=0)        : arrival = departure
      - Stasiun terakhir (seq=total-1)  : arrival = departure
      - Stasiun lainnya                 : arrival = departure - DWELL_TIME
    """
    if stop_sequence == 0 or stop_sequence == total_stops - 1:
        return departure_sec
    return departure_sec - DWELL_TIME


# trip
def get_trip_info(trip_id):
    """
    Tentukan platform_dir dan stop_headsign berdasarkan trip_id.
    Return: (platform_dir, stop_headsign)
    """
    if "DKA-JTM" in trip_id:
        return "DKA_JTM", "Jati Mulya"
    elif "JTM-DKA" in trip_id:
        return "JTM_DKA", "Dukuh Atas"
    elif "DKA-HAR" in trip_id:
        return "DKA_HAR", "Harjamukti"
    elif "HAR-DKA" in trip_id:
        return "HAR_DKA", "Dukuh Atas"
    else:
        raise ValueError(f"Tidak bisa mendeteksi jenis trip dari: {trip_id}")


# Generate stop_times
def generate_stop_times(schedule_df, platform_dir):
    """
    Generate baris stop_times dari satu DataFrame jadwal.

    Proses per trip:
      1. Iterasi setiap stasiun secara berurutan
      2. Parse departure_time dari file jadwal keberangkatan LRT Jabodebek
      3. Koreksi midnight rollover jika perlu (00:xx → 24:xx)
      4. Hitung arrival_time berdasarkan DWELL_TIME

    Return: (list of dict, int rollover_count)
    """
    rows                    = []
    plat_map                = PLATFORM_MAP[platform_dir]
    station_cols            = [c for c in schedule_df.columns
                               if c not in ("trip_id", "Full Trip")]
    midnight_rollover_count = 0

    for _, trip_row in schedule_df.iterrows():
        trip_id                  = trip_row["trip_id"]
        _, stop_headsign         = get_trip_info(trip_id)

        # Filter stasiun yang ada jadwalnya (tidak kosong / NaN)
        active_stations = [
            s for s in station_cols
            if pd.notna(trip_row[s]) and str(trip_row[s]).strip() != ""
        ]
        total_stops  = len(active_stations)
        prev_dep_sec = -1   # sentinel: belum ada stop sebelumnya

        for seq, stn in enumerate(active_stations):
            raw_dep_sec = parse_time(trip_row[stn])

            # Koreksi midnight rollover 
            if prev_dep_sec >= 0:
                corrected_dep_sec = fix_midnight_rollover(raw_dep_sec, prev_dep_sec)
                if corrected_dep_sec != raw_dep_sec:
                    midnight_rollover_count += 1
            else:
                corrected_dep_sec = raw_dep_sec

            arr_sec = calc_arrival(corrected_dep_sec, seq, total_stops)

            rows.append({
                "trip_id"             : trip_id, #id trip berdasarkan trips.txt
                "arrival_time"        : format_time(arr_sec), #waktu kedatangan berdasarkan 30 detik sebelum keberangkatan
                "departure_time"      : format_time(corrected_dep_sec), #waktu keberangkatan berdasarkan jadwal
                "stop_id"             : plat_map.get(stn, ""), #id setiap stasiun (platform) per trip
                "stop_sequence"       : seq, #urutan stasiun per trip
                "stop_headsign"       : stop_headsign, #sesuai arah trip
                "pickup_type"         : 0, #naik/turun di stasiun sesuai jadwal trip
                "drop_off_type"       : 0, #naik/turun di stasiun sesuai jadwal trip
                "continuous_pickup"   : 1, #naik/turun di stasiun resmi
                "continuous_drop_off" : 1, #naik/turun di stasiun resmi
                "timepoint"           : 1, #tepat Waktu (1)
            })

            prev_dep_sec = corrected_dep_sec

    return rows, midnight_rollover_count


# Validasi ascending per trip
def validate_ascending(df_stop_times):
    """
    Validasi bahwa departure_time selalu ascending dalam setiap trip.
    Return: list of dict issue (kosong = semua OK)
    """
    issues = []
    for trip_id, group in df_stop_times.groupby("trip_id"):
        group    = group.sort_values("stop_sequence")
        dep_secs = group["departure_time"].apply(parse_time).tolist()
        for i in range(1, len(dep_secs)):
            if dep_secs[i] < dep_secs[i - 1]:
                issues.append({
                    "trip_id"   : trip_id,
                    "stop_seq"  : group.iloc[i]["stop_sequence"],
                    "stop_id"   : group.iloc[i]["stop_id"],
                    "prev_time" : group.iloc[i - 1]["departure_time"],
                    "curr_time" : group.iloc[i]["departure_time"],
                })
    return issues

def main():
    print("=" * 60)
    print(" Build stop_times.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Dwell time   : {DWELL_TIME} detik")
    print(f" Format waktu : HH:MM:SS (>24 jam untuk lewat tengah malam)")
    print(f" shape_dist   : tidak dihitung — gunakan gtfs_kit setelahnya")
    print("=" * 60)

    # Proses setiap file jadwal
    all_rows             = []
    total_rollover_count = 0
    seen_trip_ids        = set()

    for key, filename in SCHEDULE_FILES.items():
        if not os.path.exists(filename):
            print(f"\n⚠️  File tidak ditemukan: {filename} — dilewati")
            continue

        # platform_dir = "DKA_JTM" if key.startswith("DKA") else "JTM_DKA"
        platform_dir = key.rsplit("_", 1)[0] 

        print(f"\n{'─'*55}")
        print(f"📋 Memproses : {filename}")
        line = "Lin Bekasi" if platform_dir.startswith(("DKA_JTM", "JTM_DKA")) else "Lin Cibubur"
        print(f"   Lin               : {line}")
        print(f"   Arah              : {platform_dir}")

        df = pd.read_csv(filename)
        if "Full Trip" in df.columns:
            df = df.drop(columns=["Full Trip"])

        # Guard duplikat trip_id antar file
        incoming_ids  = set(df["trip_id"].unique())
        duplicate_ids = incoming_ids & seen_trip_ids
        if duplicate_ids:
            print(f"   ⚠️  {len(duplicate_ids)} trip_id duplikat dilewati:")
            for tid in sorted(duplicate_ids)[:5]:
                print(f"      {tid}")
            df = df[~df["trip_id"].isin(duplicate_ids)]
        seen_trip_ids.update(set(df["trip_id"].unique()))

        rows, rollover_count = generate_stop_times(df, platform_dir)
        all_rows.extend(rows)
        total_rollover_count += rollover_count

        print(f"   Trip              : {df['trip_id'].nunique()}")
        print(f"   Baris             : {len(rows)}")
        print(f"   Midnight rollover : {rollover_count} koreksi")

    # Save stop_times.txt
    if not all_rows:
        print("\n⚠️  Tidak ada baris yang dihasilkan.")
        return

    fieldnames = [
        "trip_id", "arrival_time", "departure_time",
        "stop_id", "stop_sequence", "stop_headsign",
        "pickup_type", "drop_off_type",
        "continuous_pickup", "continuous_drop_off", "timepoint",
    ]

    df_out = pd.DataFrame(all_rows, columns=fieldnames)
    df_out.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    # Validasi ascending
    print(f"\n{'─'*55}")
    print("🔍 Validasi ascending time per trip ...")
    issues = validate_ascending(df_out)
    if issues:
        print(f"   ⚠️  {len(issues)} waktu tidak ascending:")
        for iss in issues[:10]:
            print(f"      trip={iss['trip_id']}  seq={iss['stop_seq']}"
                  f"  stop={iss['stop_id']}"
                  f"  {iss['prev_time']} → {iss['curr_time']}")
        if len(issues) > 10:
            print(f"      ... dan {len(issues) - 10} lainnya")
    else:
        print("   ✅ Semua trip ascending — tidak ada masalah waktu!")

    print(f"\n{'='*55}")
    print(f"✅ stop_times.txt berhasil dibuat!")
    print(f"   Path              : {OUTPUT_FILE}")
    print(f"   Total baris       : {len(df_out):,}")
    print(f"   Total trip        : {df_out['trip_id'].nunique():,}")
    print(f"   Midnight rollover : {total_rollover_count} koreksi total")

    print(f"\n📄 Preview 5 baris pertama:")
    print(df_out.head().to_string(index=False))

    print(f"\n📄 Preview 5 baris terakhir (cek format >24 jam):")
    print(df_out.tail().to_string(index=False))


if __name__ == "__main__":
    main()

 Build stop_times.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1
 2026-05-03 22:31:59
 Dwell time   : 30 detik
 Format waktu : HH:MM:SS (>24 jam untuk lewat tengah malam)
 shape_dist   : tidak dihitung — gunakan gtfs_kit setelahnya

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekdays).csv
   Lin               : Lin Bekasi
   Arah              : DKA_JTM
   Trip              : 107
   Baris             : 1498
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekends).csv
   Lin               : Lin Bekasi
   Arah              : DKA_JTM
   Trip              : 67
   Baris             : 938
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/JatiMulya_DukuhAtas_(Weekdays).csv
   Lin               : Lin Bekasi
   Arah              : JT

In [42]:
# Build field shape_dist_traveled with gtfs_kit (gtfs_kit.append_dist_to_stop_time(feed))

import gtfs_kit as gk
import pandas as pd
import numpy as np

feed = gk.read_feed(".", dist_units="m")
feed = gk.append_dist_to_stop_times(feed)

shape_max = (
    feed.shapes
    .groupby("shape_id")["shape_dist_traveled"]
    .max()
    .to_dict()
)

# Lookup: trip_id → shape_id
trip_to_shape = feed.trips.set_index("trip_id")["shape_id"].to_dict()

# Force snap: stop terakhir = max shape dist (nilai exact)
st = feed.stop_times.copy()

# Indeks baris stop terakhir per trip
last_idx = st.groupby("trip_id")["stop_sequence"].idxmax()

snapped = 0
for trip_id, idx in last_idx.items():
    shape_id = trip_to_shape.get(trip_id)
    if not shape_id or shape_id not in shape_max:
        continue
    exact_val                      = shape_max[shape_id]
    st.at[idx, "shape_dist_traveled"] = exact_val
    snapped += 1

feed.stop_times = st

print(f"✅ Force snap selesai: {snapped} trip dikoreksi")

# Verifikasi: pastikan selisih = 0
print("\n=== Verifikasi (5 trip pertama) ===")
issues = 0
for trip_id, idx in list(last_idx.items())[:5]:
    shape_id   = trip_to_shape.get(trip_id)
    val_stop   = st.at[idx, "shape_dist_traveled"]
    val_shape  = shape_max.get(shape_id)
    match      = val_stop == val_shape   # harus True (identik bit-for-bit)
    status     = "✅" if match else "❌"
    issues    += 0 if match else 1
    print(f"  {status} {trip_id}")
    print(f"     stop_times  : {val_stop:.15f}")
    print(f"     shapes      : {val_shape:.15f}")
    print(f"     selisih     : {abs(val_stop - val_shape):.2e}")

if issues == 0:
    print("\n✅ Semua stop terakhir identik dengan shapes")
else:
    print(f"\n⚠️  {issues} trip masih ada selisih")

# Save
feed.stop_times.to_csv(
    "stop_times.txt",
    index=False,
    encoding='utf-8',        
    lineterminator='\n',         
    quoting=csv.QUOTE_MINIMAL   
)
feed.stop_times

✅ Force snap selesai: 700 trip dikoreksi

=== Verifikasi (5 trip pertama) ===
  ✅ BK-WD-DKA-JTM-001
     stop_times  : 27282.705044830803672
     shapes      : 27282.705044830803672
     selisih     : 0.00e+00
  ✅ BK-WD-DKA-JTM-002
     stop_times  : 27282.705044830803672
     shapes      : 27282.705044830803672
     selisih     : 0.00e+00
  ✅ BK-WD-DKA-JTM-003
     stop_times  : 27282.705044830803672
     shapes      : 27282.705044830803672
     selisih     : 0.00e+00
  ✅ BK-WD-DKA-JTM-004
     stop_times  : 27282.705044830803672
     shapes      : 27282.705044830803672
     selisih     : 0.00e+00
  ✅ BK-WD-DKA-JTM-005
     stop_times  : 27282.705044830803672
     shapes      : 27282.705044830803672
     selisih     : 0.00e+00

✅ Semua stop terakhir identik dengan shapes


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,continuous_pickup,continuous_drop_off,timepoint,shape_dist_traveled
0,BK-WD-DKA-JTM-001,06:05:00,06:05:00,GDKA01,0,Jati Mulya,0,0,1,1,1,0.000000
1,BK-WD-DKA-JTM-001,06:07:30,06:08:00,GSET01,1,Jati Mulya,0,0,1,1,1,800.352317
2,BK-WD-DKA-JTM-001,06:10:30,06:11:00,GRAS01,2,Jati Mulya,0,0,1,1,1,2189.124644
3,BK-WD-DKA-JTM-001,06:12:30,06:13:00,GKUA01,3,Jati Mulya,0,0,1,1,1,2989.361208
4,BK-WD-DKA-JTM-001,06:17:30,06:18:00,GPAN01,4,Jati Mulya,0,0,1,1,1,5569.156695
...,...,...,...,...,...,...,...,...,...,...,...,...
9095,CB-WE-HAR-DKA-067,22:20:30,22:21:00,GPAN02,7,Dukuh Atas,0,0,1,1,1,18813.807303
9096,CB-WE-HAR-DKA-067,22:25:30,22:26:00,GKUA02,8,Dukuh Atas,0,0,1,1,1,21385.983197
9097,CB-WE-HAR-DKA-067,22:28:30,22:29:00,GRAS02,9,Dukuh Atas,0,0,1,1,1,22187.021627
9098,CB-WE-HAR-DKA-067,22:31:30,22:32:00,GSET02,10,Dukuh Atas,0,0,1,1,1,23575.176028


### Validate shape_dist_traveled Consistency (Stop Times vs Shapes) result gtfs_kit

In [43]:
import pandas as pd

SHAPES_FILE     = "shapes.txt"
STOP_TIMES_FILE = "stop_times.txt"
TRIPS_FILE      = "trips.txt"

print("="*60)
print(" VALIDASI shape_dist_traveled (FULL PRECISION)")
print("="*60)

# Load data
shapes = pd.read_csv(SHAPES_FILE)
stop_times = pd.read_csv(STOP_TIMES_FILE)
trips = pd.read_csv(TRIPS_FILE)

# shape max
shape_max = (
    shapes
    .groupby("shape_id")["shape_dist_traveled"]
    .max()
    .to_dict()
)

# TRIP → SHAPE
trip_to_shape = trips.set_index("trip_id")["shape_id"].to_dict()

# LAST STOP PER TRIP
last_stops = stop_times.loc[
    stop_times.groupby("trip_id")["stop_sequence"].idxmax()
].copy()

# VALIDATION
results = []
mismatch_count = 0

for _, row in last_stops.iterrows():
    trip_id  = row["trip_id"]
    shape_id = trip_to_shape.get(trip_id)

    if not shape_id or shape_id not in shape_max:
        continue

    stop_val  = row["shape_dist_traveled"]
    shape_val = shape_max[shape_id]

    # STRICT COMPARISON (tanpa tolerance)
    match = stop_val == shape_val

    if not match:
        mismatch_count += 1

    results.append({
        "trip_id": trip_id,
        "shape_id": shape_id,
        "stop_times_last": stop_val,
        "shapes_max": shape_val,
        "diff": stop_val - shape_val,
        "status": "OK" if match else "MISMATCH"
    })

df_result = pd.DataFrame(results)

# ================= SUMMARY =================
print(f"\n📊 Total trip dicek : {len(df_result):,}")
print(f"❌ Mismatch         : {mismatch_count:,}")
print(f"✅ Match            : {len(df_result) - mismatch_count:,}")

def print_full_precision(df, n=10):
    print("\n📄 Sample hasil (FULL PRECISION):")
    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26} {'status'}"
    )

    for _, r in df.head(n).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff']):<26} "
            f"{r['status']}"
        )

print_full_precision(df_result)

# DETAIL MISMATCH
if mismatch_count > 0:
    print("\n⚠️ Detail mismatch (FULL PRECISION):")

    df_mismatch = (
        df_result[df_result["status"] == "MISMATCH"]
        .copy()
    )

    # sort by absolute diff (tanpa ubah nilai)
    df_mismatch["abs_diff"] = df_mismatch["diff"].abs()
    df_mismatch = df_mismatch.sort_values("abs_diff", ascending=False)

    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26}"
    )

    for _, r in df_mismatch.head(20).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff'])}"
        )
else:
    print("\n✅ Semua trip IDENTIK")

 VALIDASI shape_dist_traveled (FULL PRECISION)

📊 Total trip dicek : 700
❌ Mismatch         : 0
✅ Match            : 700

📄 Sample hasil (FULL PRECISION):
trip_id              shape_id           stop_times_last            shapes_max                 diff                       status
BK-WD-DKA-JTM-001    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-002    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-003    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-004    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-005    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-006    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0

#### Cek shapes.txt dan stop_times.txt untuk **`shape_DKA_JTM`** 

In [44]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'BK-WD-DKA-JTM-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : BK-WD-DKA-JTM-001
✅ Shape  : shape_DKA_JTM
✅ Jumlah titik shape: 358
✅ Jumlah stop digambar: 14


In [45]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_JTM_DKA`** 

In [46]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'BK-WD-JTM-DKA-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : BK-WD-JTM-DKA-001
✅ Shape  : shape_JTM_DKA
✅ Jumlah titik shape: 375
✅ Jumlah stop digambar: 14


In [47]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_DKA_HAR`** 

In [48]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'CB-WD-DKA-HAR-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : CB-WD-DKA-HAR-001
✅ Shape  : shape_DKA_HAR
✅ Jumlah titik shape: 395
✅ Jumlah stop digambar: 12


In [49]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_HAR_DKA`** 

In [50]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'CB-WD-HAR-DKA-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : CB-WD-HAR-DKA-001
✅ Shape  : shape_HAR_DKA
✅ Jumlah titik shape: 393
✅ Jumlah stop digambar: 12


In [51]:
m

### Build `stop_times.txt` with formula haversine for shape_dist_traveled

In [52]:
"""
Build stop_times.txt berdasarkan jadwal keberangkatan LRT Jabodebek

departure_time  : langsung dari file jadwal keberangkatan
arrival_time    : departure_time - DWELL_TIME (detik), kecuali stasiun pertama & terakhir: arrival = departure

Satuan shape_dist_traveled : METER (konsisten dengan shapes.txt)

Format waktu GTFS:
  - Waktu TIDAK direset ke 00:xx:xx saat melewati tengah malam
  - Melainkan terus bertambah: 24:xx:xx, 25:xx:xx, dst
  - Contoh: kereta berangkat 23:40 tiba 00:01 → ditulis 24:01:xx

Catatan:
  - shape_dist_traveled dihitung menggunakan rumus haversine dengan satuan Meter
"""

import pandas as pd
import csv
import math
import os
from datetime import datetime

# File path 
SHAPES_FILE     = "shapes.txt"
STOPS_FILE      = "stops.txt"
OUTPUT_FILE     = "stop_times.txt"

# Dwell time dalam detik
# arrival = departure - DWELL_TIME
# kecuali stasiun pertama & terakhir: arrival = departure
DWELL_TIME      = 30   # detik

# File jadwal keberangkatan lRT Jabodebek
SCHEDULE_FILES = {
    "DKA_JTM_WD": "data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekdays).csv",
    "DKA_JTM_WE": "data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekends).csv",
    "JTM_DKA_WD": "data/jadwal-keberangkatan/JatiMulya_DukuhAtas_(Weekdays).csv",
    "JTM_DKA_WE": "data/jadwal-keberangkatan/JatiMulya_DukuhAtas_(Weekends).csv",
    "DKA_HAR_WD": "data/jadwal-keberangkatan/DukuhAtas_Harjamukti_(Weekdays).csv",
    "DKA_HAR_WE": "data/jadwal-keberangkatan/DukuhAtas_Harjamukti_(Weekends).csv",
    "HAR_DKA_WD": "data/jadwal-keberangkatan/Harjamukti_DukuhAtas_(Weekdays).csv",
    "HAR_DKA_WE": "data/jadwal-keberangkatan/Harjamukti_DukuhAtas_(Weekends).csv",
}

# Mapping kode stasiun → stop_id platform
PLATFORM_MAP = {
    "DKA_JTM": {
        "DKA": "GDKA01", "SET": "GSET01", "RAS": "GRAS01",
        "KUA": "GKUA01", "PAN": "GPAN01", "CKK": "GCKK01",
        "CIL": "GCIL01", "CWG": "GCWG01", "HAL": "GHAL01",
        "JBU": "GJBU01", "CK1": "GCK101", "CK2": "GCK201",
        "BEK": "GBEK01", "JTM": "GJTM01",
    },
    "JTM_DKA": {
        "JTM": "GJTM02", "BEK": "GBEK02", "CK2": "GCK202",
        "CK1": "GCK102", "JBU": "GJBU02", "HAL": "GHAL02",
        "CWG": "GCWG02", "CIL": "GCIL02", "CKK": "GCKK02",
        "PAN": "GPAN02", "KUA": "GKUA02", "RAS": "GRAS02",
        "SET": "GSET02", "DKA": "GDKA02",
    },
    "DKA_HAR": {
        "DKA": "GDKA01", "SET": "GSET01", "RAS": "GRAS01",
        "KUA": "GKUA01", "PAN": "GPAN01", "CKK": "GCKK01",
        "CIL": "GCIL01", "CWG": "GCWG03", "TMI": "GTMI01",
        "KAM": "GKAM01", "CRC": "GCRC01", "HAR": "GHAR01",
    },
    "HAR_DKA": {
        "HAR": "GHAR02", "CRC": "GCRC02", "KAM": "GKAM02",
        "TMI": "GTMI02", "CWG": "GCWG04", "CIL": "GCIL02", 
        "CKK": "GCKK02", "PAN": "GPAN02", "KUA": "GKUA02",
        "RAS": "GRAS02", "SET": "GSET02", "DKA": "GDKA02",
    },
}

# shape_id per jenis trip
SHAPE_MAP = {
    "DKA_JTM"   : "shape_DKA_JTM",
    "JTM_DKA"   : "shape_JTM_DKA",
    "DKA_HAR"   : "shape_DKA_HAR",
    "HAR_DKA"   : "shape_HAR_DKA"
}


# Formula Haversine 
def haversine_m(lat1, lon1, lat2, lon2):
    """Jarak antara dua titik koordinat dalam METER."""
    R  = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))


# Load shapes
def load_shapes(shapes_file):
    """
    Load shapes.txt → dict shape_id → DataFrame (lat, lon, shape_dist_traveled)
    diurutkan berdasarkan shape_pt_sequence.
    """
    df = pd.read_csv(shapes_file)
    shapes = {}
    for sid in df["shape_id"].unique():
        shapes[sid] = (
            df[df["shape_id"] == sid]
            .sort_values("shape_pt_sequence")
            .reset_index(drop=True)
        )
    return shapes


def get_shape_dist(shape_df, stop_lat, stop_lon):
    """
    Cari shape_dist_traveled pada titik shape terdekat ke (stop_lat, stop_lon).
    Return nilai shape_dist_traveled dalam meter.
    """
    dists = shape_df.apply(
        lambda r: haversine_m(stop_lat, stop_lon,
                              r["shape_pt_lat"], r["shape_pt_lon"]),
        axis=1
    )
    nearest_idx = dists.idxmin()
    return float(shape_df.loc[nearest_idx, "shape_dist_traveled"])


# Load stops
def load_stops(stops_file):
    """Load stops.txt → dict stop_id → (stop_lat, stop_lon)."""
    df = pd.read_csv(stops_file, dtype={"stop_id": str})
    return {
        row["stop_id"]: (float(row["stop_lat"]), float(row["stop_lon"]))
        for _, row in df.iterrows()
        if pd.notna(row.get("stop_lat")) and str(row.get("stop_lat", "")).strip() != ""
    }


# Time utils
def parse_time(time_str):
    """
    Parse waktu HH:MM:SS ke total detik.
    Support jam > 24 untuk layanan melewati tengah malam (GTFS compliant).

    Contoh:
        "05:12:00" →  18720
        "23:40:30" →  85230
        "24:00:50" →  86450  (sudah >24, langsung parse tanpa modifikasi)
    """
    parts = str(time_str).strip().split(":")
    h = int(parts[0])
    m = int(parts[1])
    s = int(parts[2]) if len(parts) > 2 else 0
    return h * 3600 + m * 60 + s


def format_time(total_seconds):
    """
    Format total detik ke HH:MM:SS sesuai spesifikasi GTFS.

    Jam TIDAK direset ke 00 saat melewati tengah malam —
    melainkan terus bertambah (24:xx:xx, 25:xx:xx, dst).

    Contoh:
        85230  → "23:40:30"
        86450  → "24:00:50"  ✅ bukan "00:00:50"
        90000  → "25:00:00"
    """
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def fix_midnight_rollover(current_sec, prev_sec):
    """
    Koreksi midnight rollover sesuai spesifikasi GTFS.

    GTFS mengharuskan waktu selalu ASCENDING dalam satu trip.
    Kalau current_sec < prev_sec, artinya CSV menyimpan waktu
    sebagai 00:xx:xx setelah melewati tengah malam — harus
    dikoreksi dengan menambah 86400 detik (24 jam).

    Menggunakan while loop untuk mengakomodasi edge case trip
    yang melewati tengah malam lebih dari satu kali.

    Contoh:
        prev    = 23:40:30 → 85230 detik
        current = 00:00:50 →    50 detik  ← lebih kecil → rollover!
        koreksi = 50 + 86400 = 86450      → format_time → "24:00:50" ✅

    Args:
        current_sec : waktu stop saat ini dalam detik
        prev_sec    : waktu stop sebelumnya dalam detik

    Returns:
        current_sec yang sudah dikoreksi (int)
    """
    while current_sec < prev_sec:
        current_sec += 86400  # +24 jam
    return current_sec


def calc_arrival(departure_sec, stop_sequence, total_stops):
    """
    Hitung arrival_time dalam detik.

    Aturan:
      - Stasiun pertama  (seq=0)        : arrival = departure (no dwell)
      - Stasiun terakhir (seq=total-1)  : arrival = departure (no dwell)
      - Stasiun lainnya                 : arrival = departure - DWELL_TIME

    Catatan: arrival tidak perlu rollover correction karena selalu
    dihitung SETELAH departure sudah dikoreksi.
    """
    if stop_sequence == 0 or stop_sequence == total_stops - 1:
        return departure_sec
    return departure_sec - DWELL_TIME


# Deteksi jenis trip
def get_trip_type(trip_id):
    """
    Tentukan jenis trip berdasarkan trip_id.
    Return: (trip_type, platform_dir)
    """
   
    if "DKA-JTM" in trip_id:
        return "DKA_JTM", "DKA_JTM", "Jati Mulya"
    elif "JTM-DKA" in trip_id:
        return "JTM_DKA", "JTM_DKA", "Dukuh Atas"
    elif "DKA-HAR" in trip_id:
        return "DKA_HAR", "DKA_HAR", "Harjamukti"
    elif "HAR-DKA" in trip_id:
        return "HAR_DKA", "HAR_DKA", "Dukuh Atas"
        
    else:
        raise ValueError(f"Tidak bisa mendeteksi jenis trip dari: {trip_id}")


# Generate stop_times 
def generate_stop_times(schedule_df, platform_dir, shapes, stops):
    """
    Generate baris stop_times dari satu DataFrame jadwal.

    Proses per trip:
      1. Iterasi setiap stasiun secara berurutan
      2. Parse departure_time dari CSV
      3. Koreksi midnight rollover jika perlu (00:xx → 24:xx) 
      4. Hitung arrival_time dengan DWELL_TIME (setelah koreksi)
      5. Map stop_id dari PLATFORM_MAP
      6. Hitung shape_dist_traveled dari shapes.txt

    Returns:
        (list of dict, int rollover_count)
    """
    rows                    = []
    plat_map                = PLATFORM_MAP[platform_dir]
    station_cols            = [c for c in schedule_df.columns
                               if c not in ("trip_id", "Full Trip")]
    midnight_rollover_count = 0

    for _, trip_row in schedule_df.iterrows():
        trip_id      = trip_row["trip_id"]
        trip_type, _, stop_headsign = get_trip_type(trip_id)
        shape_id     = SHAPE_MAP[trip_type]
        shape_df     = shapes.get(shape_id)

        # Filter stasiun yang ada jadwalnya (tidak kosong / NaN)
        active_stations = [
            s for s in station_cols
            if pd.notna(trip_row[s]) and str(trip_row[s]).strip() != ""
        ]
        total_stops = len(active_stations)

        # ── Hitung shape_dist_traveled per stasiun ────────────────
        shape_dists_raw = {}
        if shape_df is not None:
            for stn in active_stations:
                stop_id = plat_map.get(stn)
                if stop_id and stop_id in stops:
                    lat, lon = stops[stop_id]
                    shape_dists_raw[stn] = get_shape_dist(shape_df, lat, lon)
                else:
                    shape_dists_raw[stn] = None
                    
        shape_dists = {
                s: (shape_dists_raw[s]
                    if shape_dists_raw.get(s) is not None else "")
                for s in active_stations
            }

        # Iterasi per stasiun dengan midnight rollover 
        prev_dep_sec = -1  # sentinel: belum ada stop sebelumnya

        for seq, stn in enumerate(active_stations):
            raw_dep_sec = parse_time(trip_row[stn])

            # ── Koreksi midnight rollover ─────────────────────────
            # Harus dilakukan SEBELUM calc_arrival agar arrival
            # juga menggunakan waktu yang sudah dikoreksi.
            #
            # Contoh tanpa koreksi (SALAH):
            #   IST  dep=23:40:30 → 85230 detik
            #   DKA  dep=00:00:50 →    50 detik  ← lebih kecil!
            #   arr  = 50 - 30 = 20 detik → "00:00:20" ← SALAH
            #
            # Contoh dengan koreksi (BENAR):
            #   IST  dep=23:40:30 → 85230 detik
            #   DKA  dep=00:00:50 → 86450 detik  ← +86400 ✅
            #   arr  = 86450 - 0  = 86450 detik  → "24:00:50" ✅
            if prev_dep_sec >= 0:
                corrected_dep_sec = fix_midnight_rollover(
                    raw_dep_sec, prev_dep_sec
                )
                if corrected_dep_sec != raw_dep_sec:
                    midnight_rollover_count += 1
            else:
                corrected_dep_sec = raw_dep_sec

            arr_sec       = calc_arrival(corrected_dep_sec, seq, total_stops)
            dist_traveled = shape_dists.get(stn, "")

            rows.append({
                "trip_id"        : trip_id,
                "arrival_time"   : format_time(arr_sec),
                "departure_time" : format_time(corrected_dep_sec),
                "stop_id"        : plat_map.get(stn, ""),
                "stop_sequence"  : seq,
                "stop_headsign"  : stop_headsign,
                "pickup_type"    : 0,
                "drop_off_type"  : 0,
                "continuous_pickup": 1,
                "continuous_drop_off": 1,
                "timepoint": 1,
                "shape_dist_traveled": dist_traveled,
            })

            prev_dep_sec = corrected_dep_sec  # update untuk iterasi berikutnya

    return rows, midnight_rollover_count


# Validasi ascending per trip
def validate_ascending(df_stop_times):
    """
    Validasi bahwa departure_time selalu ascending dalam setiap trip.
    Return: list of dict issue (kosong = semua OK)
    """
    issues = []
    for trip_id, group in df_stop_times.groupby("trip_id"):
        group    = group.sort_values("stop_sequence")
        dep_secs = group["departure_time"].apply(parse_time).tolist()
        for i in range(1, len(dep_secs)):
            if dep_secs[i] < dep_secs[i - 1]:
                issues.append({
                    "trip_id"   : trip_id,
                    "stop_seq"  : group.iloc[i]["stop_sequence"],
                    "stop_id"   : group.iloc[i]["stop_id"],
                    "prev_time" : group.iloc[i - 1]["departure_time"],
                    "curr_time" : group.iloc[i]["departure_time"],
                })
    return issues


def main():
    print("=" * 60)
    print(" Build stop_times.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Dwell time  : {DWELL_TIME} detik")
    print(f" Satuan dist : METER")
    print(f" Format waktu: HH:MM:SS (>24 jam untuk lewat tengah malam)")
    print("=" * 60)

    # Validasi file wajib
    for f in [SHAPES_FILE, STOPS_FILE]:
        if not os.path.exists(f):
            print(f"\n❌ File tidak ditemukan: {f}")
            return

    # Load shapes & stops
    print("\n📂 Loading shapes.txt ...")
    shapes = load_shapes(SHAPES_FILE)
    print(f"   Shape tersedia : {list(shapes.keys())}")

    print("📂 Loading stops.txt ...")
    stops = load_stops(STOPS_FILE)
    print(f"   Stop tersedia  : {len(stops)} stop")

    # Proses setiap file jadwal
    all_rows             = []
    total_rollover_count = 0
    seen_trip_ids        = set()  # guard duplikat trip_id antar file

    for key, filename in SCHEDULE_FILES.items():
        if not os.path.exists(filename):
            print(f"\n⚠️  File tidak ditemukan: {filename} — dilewati")
            continue

        # platform_dir = "DKA_JTM" if key.startswith("DKA") else "JTM_DKA"
        platform_dir = key.rsplit("_", 1)[0] 

        print(f"\n{'─'*55}")
        print(f"📋 Memproses : {filename}")
        line = "Lin Bekasi" if platform_dir.startswith(("DKA_JTM", "JTM_DKA")) else "Lin Cibubur"
        print(f"   Lin               : {line}")
        print(f"   Arah              : {platform_dir}")

        df = pd.read_csv(filename)
        if "Full Trip" in df.columns:
            df = df.drop(columns=["Full Trip"])

        # Guard duplikat trip_id antar file
        incoming_ids  = set(df["trip_id"].unique())
        duplicate_ids = incoming_ids & seen_trip_ids
        if duplicate_ids:
            print(f"   ⚠️  {len(duplicate_ids)} trip_id duplikat dilewati:")
            for tid in sorted(duplicate_ids)[:5]:
                print(f"      {tid}")
            df = df[~df["trip_id"].isin(duplicate_ids)]
        seen_trip_ids.update(set(df["trip_id"].unique()))

        rows, rollover_count = generate_stop_times(
            df, platform_dir, shapes, stops
        )
        all_rows.extend(rows)
        total_rollover_count += rollover_count

        print(f"   Trip              : {df['trip_id'].nunique()}")
        print(f"   Baris             : {len(rows)}")
        print(f"   Midnight rollover : {rollover_count} koreksi")

    # Save stop_times.txt
    if not all_rows:
        print("\n⚠️  Tidak ada baris yang dihasilkan.")
        return

    fieldnames = [
        "trip_id", "arrival_time", "departure_time",
        "stop_id", "stop_sequence", "stop_headsign", "pickup_type", "drop_off_type",
        "continuous_pickup", "continuous_drop_off", "timepoint", "shape_dist_traveled"
    ]

    df_out = pd.DataFrame(all_rows, columns=fieldnames)
    df_out.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    # Validasi ascending
    print(f"\n{'─'*55}")
    print("🔍 Validasi ascending time per trip ...")
    issues = validate_ascending(df_out)
    if issues:
        print(f"   ⚠️  {len(issues)} waktu tidak ascending:")
        for iss in issues[:10]:
            print(f"      trip={iss['trip_id']}  seq={iss['stop_seq']}"
                  f"  stop={iss['stop_id']}"
                  f"  {iss['prev_time']} → {iss['curr_time']}")
        if len(issues) > 10:
            print(f"      ... dan {len(issues) - 10} lainnya")
    else:
        print("   ✅ Semua trip ascending — tidak ada masalah waktu!")

    # Summary
    print(f"\n{'='*55}")
    print(f"✅ stop_times.txt berhasil dibuat!")
    print(f"   Path              : {OUTPUT_FILE}")
    print(f"   Total baris       : {len(df_out):,}")
    print(f"   Total trip        : {df_out['trip_id'].nunique():,}")
    print(f"   Midnight rollover : {total_rollover_count} koreksi total")

    print(f"\n📄 Preview 5 baris pertama:")
    print(df_out.head().to_string(index=False))

    print(f"\n📄 Preview 5 baris terakhir (cek format >24 jam):")
    print(df_out.tail().to_string(index=False))


if __name__ == "__main__":
    main()

 Build stop_times.txt  |  LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1
 2026-05-03 22:32:01
 Dwell time  : 30 detik
 Satuan dist : METER
 Format waktu: HH:MM:SS (>24 jam untuk lewat tengah malam)

📂 Loading shapes.txt ...
   Shape tersedia : ['shape_DKA_JTM', 'shape_JTM_DKA', 'shape_DKA_HAR', 'shape_HAR_DKA']
📂 Loading stops.txt ...
   Stop tersedia  : 124 stop

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekdays).csv
   Lin               : Lin Bekasi
   Arah              : DKA_JTM
   Trip              : 107
   Baris             : 1498
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
📋 Memproses : data/jadwal-keberangkatan/DukuhAtas_JatiMulya_(Weekends).csv
   Lin               : Lin Bekasi
   Arah              : DKA_JTM
   Trip              : 67
   Baris             : 938
   Midnight rollover : 0 koreksi

───────────────────────────────────────────────────────
📋 Mempro

### Validate shape_dist_traveled Consistency (Stop Times vs Shapes) result formula haversine

In [53]:
import pandas as pd

SHAPES_FILE     = "shapes.txt"
STOP_TIMES_FILE = "stop_times.txt"
TRIPS_FILE      = "trips.txt"

print("="*60)
print(" VALIDASI shape_dist_traveled (FULL PRECISION)")
print("="*60)

# Load data
shapes = pd.read_csv(SHAPES_FILE)
stop_times = pd.read_csv(STOP_TIMES_FILE)
trips = pd.read_csv(TRIPS_FILE)

# shape max
shape_max = (
    shapes
    .groupby("shape_id")["shape_dist_traveled"]
    .max()
    .to_dict()
)

# TRIP → SHAPE
trip_to_shape = trips.set_index("trip_id")["shape_id"].to_dict()

# LAST STOP PER TRIP
last_stops = stop_times.loc[
    stop_times.groupby("trip_id")["stop_sequence"].idxmax()
].copy()

# VALIDATION
results = []
mismatch_count = 0

for _, row in last_stops.iterrows():
    trip_id  = row["trip_id"]
    shape_id = trip_to_shape.get(trip_id)

    if not shape_id or shape_id not in shape_max:
        continue

    stop_val  = row["shape_dist_traveled"]
    shape_val = shape_max[shape_id]

    # STRICT COMPARISON (tanpa tolerance)
    match = stop_val == shape_val

    if not match:
        mismatch_count += 1

    results.append({
        "trip_id": trip_id,
        "shape_id": shape_id,
        "stop_times_last": stop_val,
        "shapes_max": shape_val,
        "diff": stop_val - shape_val,
        "status": "OK" if match else "MISMATCH"
    })

df_result = pd.DataFrame(results)

# ================= SUMMARY =================
print(f"\n📊 Total trip dicek : {len(df_result):,}")
print(f"❌ Mismatch         : {mismatch_count:,}")
print(f"✅ Match            : {len(df_result) - mismatch_count:,}")

def print_full_precision(df, n=10):
    print("\n📄 Sample hasil (FULL PRECISION):")
    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26} {'status'}"
    )

    for _, r in df.head(n).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff']):<26} "
            f"{r['status']}"
        )

print_full_precision(df_result)

# DETAIL MISMATCH
if mismatch_count > 0:
    print("\n⚠️ Detail mismatch (FULL PRECISION):")

    df_mismatch = (
        df_result[df_result["status"] == "MISMATCH"]
        .copy()
    )

    # sort by absolute diff (tanpa ubah nilai)
    df_mismatch["abs_diff"] = df_mismatch["diff"].abs()
    df_mismatch = df_mismatch.sort_values("abs_diff", ascending=False)

    print(
        f"{'trip_id':<20} {'shape_id':<18} "
        f"{'stop_times_last':<26} {'shapes_max':<26} "
        f"{'diff':<26}"
    )

    for _, r in df_mismatch.head(20).iterrows():
        print(
            f"{r['trip_id']:<20} "
            f"{r['shape_id']:<18} "
            f"{repr(r['stop_times_last']):<26} "
            f"{repr(r['shapes_max']):<26} "
            f"{repr(r['diff'])}"
        )
else:
    print("\n✅ Semua trip IDENTIK")

 VALIDASI shape_dist_traveled (FULL PRECISION)

📊 Total trip dicek : 700
❌ Mismatch         : 0
✅ Match            : 700

📄 Sample hasil (FULL PRECISION):
trip_id              shape_id           stop_times_last            shapes_max                 diff                       status
BK-WD-DKA-JTM-001    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-002    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-003    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-004    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-005    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0                        OK
BK-WD-DKA-JTM-006    shape_DKA_JTM      27282.705044830804         27282.705044830804         0.0

#### Cek shapes.txt dan stop_times.txt untuk **`shape_DKA_JTM`** 

In [54]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'BK-WD-DKA-JTM-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : BK-WD-DKA-JTM-001
✅ Shape  : shape_DKA_JTM
✅ Jumlah titik shape: 358
✅ Jumlah stop digambar: 14


In [55]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_JTM_DKA`** 

In [56]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'BK-WD-JTM-DKA-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : BK-WD-JTM-DKA-001
✅ Shape  : shape_JTM_DKA
✅ Jumlah titik shape: 375
✅ Jumlah stop digambar: 14


In [57]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_DKA_HAR`** 

In [58]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'CB-WD-DKA-HAR-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : CB-WD-DKA-HAR-001
✅ Shape  : shape_DKA_HAR
✅ Jumlah titik shape: 395
✅ Jumlah stop digambar: 12


In [59]:
m

#### Cek shapes.txt dan stop_times.txt untuk **`shape_HAR_DKA`** 

In [60]:
import pandas as pd
import folium

shapes     = pd.read_csv('shapes.txt')
trips      = pd.read_csv('trips.txt')
stops      = pd.read_csv('stops.txt')
stop_times = pd.read_csv('stop_times.txt')

target_trip_id = 'CB-WD-HAR-DKA-001'

# Ambil shape_id dari trips.txt
trip_row = trips[trips['trip_id'] == target_trip_id]

if len(trip_row) == 0:
    print(f"❌ Trip {target_trip_id} tidak ditemukan di trips.txt")
else:
    shape_id = trip_row['shape_id'].values[0]
    print(f"✅ Trip   : {target_trip_id}")
    print(f"✅ Shape  : {shape_id}")

    # Ambil titik shape
    shape_data = shapes[shapes['shape_id'] == shape_id].sort_values('shape_pt_sequence')

    if len(shape_data) == 0:
        print(f"❌ Shape {shape_id} tidak ditemukan di shapes.txt")
    else:
        print(f"✅ Jumlah titik shape: {len(shape_data)}")

        # Titik tengah peta
        center_lat = shape_data['shape_pt_lat'].mean()
        center_lon = shape_data['shape_pt_lon'].mean()

        # Buat peta
        m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

        # Gambar jalur shape
        coords = list(zip(shape_data['shape_pt_lat'], shape_data['shape_pt_lon']))
        folium.PolyLine(
            coords,
            color='blue',
            weight=4,
            opacity=0.8,
            tooltip=f"Shape: {shape_id} | Trip: {target_trip_id}"
        ).add_to(m)

        # Titik awal dan akhir shape
        folium.Marker(
            coords[0],
            popup=f"START shape {shape_id}",
            icon=folium.Icon(color='green', icon='play')
        ).add_to(m)

        folium.Marker(
            coords[-1],
            popup=f"END shape {shape_id}",
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(m)

        # Gambar stop jika ada di stop_times
        st = stop_times[stop_times['trip_id'] == target_trip_id]

        if len(st) == 0:
            print(f"⚠️  Trip {target_trip_id} tidak ada di stop_times.txt")
            print(f"   Jalur shape tetap digambar tanpa titik stop")
        else:
            st = st.sort_values('stop_sequence').merge(
                stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
                on='stop_id', how='left'
            )
            for _, stop in st.iterrows():
                folium.CircleMarker(
                    location=[stop['stop_lat'], stop['stop_lon']],
                    radius=6,
                    color='orange',
                    fill=True,
                    fill_color='orange',
                    fill_opacity=0.9,
                    popup=f"seq:{stop['stop_sequence']} | {stop['stop_id']} | {stop['stop_name']}"
                ).add_to(m)
            print(f"✅ Jumlah stop digambar: {len(st)}")

        # Simpan
        output_file = f'map_lrt_jabodebek_{target_trip_id}.html'
        # m.save(output_file)
        # print(f"\n✅ Peta disimpan: {output_file}")
        # print(f"   Buka di browser untuk melihat rute")

✅ Trip   : CB-WD-HAR-DKA-001
✅ Shape  : shape_HAR_DKA
✅ Jumlah titik shape: 393
✅ Jumlah stop digambar: 12


In [61]:
m

## Fare (Optional)

### Panduan Pengisian File Tarif GTFS LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1

Dokumentasi ini menjelaskan struktur, field, dan panduan pengisian untuk **7 file tarif GTFS** LRT Jabodebek Lin Bekasi dan Lin Cibubur.

**`Daftar File`**

| File | Versi | Fungsi |
|---|---|---|
| `fare_attributes.txt` | Fares V1 | Definisi kelas tarif dan nominal harga |
| `fare_rules.txt` | Fares V1 | Aturan penerapan tarif per pasangan O-D |
| `timeframes.txt` | Fares V2 | Definisi periode waktu (jam sibuk, luar jam sibuk, libur) |
| `areas.txt` | Fares V2 | Definisi zona area per stasiun |
| `stop_areas.txt` | Fares V2 | Mapping platform stop_id ke area_id |
| `fare_products.txt` | Fares V2 | Produk tarif per periode dan nilai |
| `fare_leg_rules.txt` | Fares V2 | Aturan tarif lengkap per O-D dan periode |

> **Catatan:** Fares V1 digunakan oleh **OpenTripPlanner (OTP)**. Fares V2 digunakan oleh platform yang mendukung spesifikasi GTFS terbaru seperti Google Maps dan MobilityData validator. Keduanya boleh ada dalam satu feed secara bersamaan.

---

**`Tarif LRT Jabodebek`**

LRT Jabodebek menerapkan sistem tarif berbasis jarak dengan tiga periode waktu:

| Periode | Waktu Berlaku | Keterangan |
|---|---|---|
| **Jam Sibuk (Peak)** | Senin–Jumat, 06:00–08:59 dan 16:00–19:59 | Tarif tertinggi |
| **Luar Jam Sibuk (Offpeak)** | Senin–Jumat, di luar jam sibuk | Tarif standar |
| **Hari Libur (Holiday)** | Sabtu, Minggu & Hari Libur Nasional | Tarif jam sibuk tidak berlaku |

---

**`Fares V1`**

**`fare_attributes.txt`**

File ini mendefinisikan **kelas tarif dan nominal harga** yang digunakan oleh OTP untuk menghitung ongkos perjalanan. Menggunakan tarif **offpeak sebagai tarif default** karena Fares V1 tidak mendukung diferensiasi waktu.

`Field`

| Field | Tipe | Keharusan | Deskripsi |
|---|---|---|---|
| `fare_id` | Unique ID | **Wajib** | ID unik kelas tarif. Format: `LRTJAB_{nilai_tarif}` |
| `price` | Non-negative float | **Wajib** | Nominal tarif dalam satuan mata uang `currency_type` |
| `currency_type` | Currency code | **Wajib** | Kode mata uang ISO 4217 |
| `payment_method` | Enum | **Wajib** | `0` = bayar di kendaraan, `1` = bayar sebelum boarding |
| `transfers` | Enum | **Wajib** | `0` = tidak ada transfer gratis |
| `agency_id` | Foreign ID | Kondisional | ID operator merujuk ke `agency.txt` |

`Panduan untuk LRT Jabodebek`

- `fare_id` diformat sebagai `LRTJAB_{nominal}` — contoh: `LRTJAB_5000`, `LRTJAB_7500`
- `price` diisi nominal tarif offpeak dalam Rupiah — contoh: `5000`
- `currency_type` diisi `IDR`
- `payment_method` diisi `1` — penumpang tap kartu di gate sebelum masuk peron
- `transfers` diisi `0` — tidak ada transfer gratis antar lin
- `agency_id` diisi `KAI`

Contoh

```
fare_id,price,currency_type,payment_method,transfers,agency_id
LRTJAB_5000,5000,IDR,1,0,KAI
LRTJAB_5700,5700,IDR,1,0,KAI
LRTJAB_7100,7100,IDR,1,0,KAI
...
LRTJAB_10000,10000,IDR,1,0,KAI
```

---

**`fare_rules.txt`**

File ini mendefinisikan **aturan penerapan tarif** menghubungkan `fare_id` dengan pasangan zona asal (`origin_id`) dan zona tujuan (`destination_id`). Menggunakan tarif offpeak sebagai default untuk OTP.

`Field`

| Field | Tipe | Keharusan | Deskripsi |
|---|---|---|---|
| `fare_id` | Foreign ID → `fare_attributes.fare_id` | **Wajib** | ID kelas tarif yang diterapkan |
| `route_id` | Foreign ID → `routes.route_id` | Opsional | ID rute  dikosongkan agar berlaku untuk semua lin |
| `origin_id` | Foreign ID → `stops.zone_id` | Opsional | Kode zona stasiun asal |
| `destination_id` | Foreign ID → `stops.zone_id` | Opsional | Kode zona stasiun tujuan |
| `contains_id` | Foreign ID → `stops.zone_id` | Opsional | Zona yang harus dilalui, tidak digunakan |

`Panduan untuk LRT Jabodebek`

- `fare_id` merujuk ke `fare_attributes.txt` dengan tarif offpeak
- `route_id` dikosongkan agar tarif berlaku untuk **Lin Bekasi dan Lin Cibubur** sekaligus
- `origin_id` dan `destination_id` diisi kode stasiun (`DKA`, `SET`, `RAS`, dst) yang harus sama dengan `zone_id` di `stops.txt`
- Total **306 pasangan O-D** (18 stasiun × 17 tujuan)
- `contains_id` dikosongkan tidak digunakan untuk LRT Jabodebek

Contoh

```
fare_id,route_id,origin_id,destination_id,contains_id
LRTJAB_5000,,DKA,SET,
LRTJAB_5700,,DKA,RAS,
LRTJAB_7100,,DKA,KUA,
...
LRTJAB_5000,,SET,DKA,
```

---

**`Fares V2`**

**`timeframes.txt`**

File ini mendefinisikan **periode waktu** yang digunakan untuk membedakan tarif berdasarkan waktu keberangkatan penumpang.

`Field`

| Field | Tipe | Keharusan | Deskripsi |
|---|---|---|---|
| `timeframe_group_id` | ID | **Wajib** | ID grup periode waktu |
| `start_time` | Time | **Wajib** | Waktu mulai periode dalam format `HH:MM:SS` |
| `end_time` | Time | **Wajib** | Waktu akhir periode dalam format `HH:MM:SS` |
| `service_id` | Foreign ID → `calendar.service_id` | **Wajib** | ID jadwal operasi merujuk ke `calendar.txt` |

`Panduan untuk LRT Jabodebek`

Tiga `timeframe_group_id` yang digunakan:

| `timeframe_group_id` | `service_id` | Rentang Waktu |
|---|---|---|
| `PEAK` | `WD` | 06:00–09:00 (pagi) dan 16:00–20:00 (sore) |
| `OFFPEAK` | `WD` | 05:00–06:00, 09:00–16:00, dan 20:00–24:00 |
| `HOLIDAY` | `WE` | 05:00–24:00 (seluruh hari operasi) |

> **Catatan:** `end_time` menggunakan format eksklusif — `09:00:00` berarti periode berakhir tepat sebelum pukul 09:00, sehingga `06:00–09:00` mencakup 06:00 s/d 08:59:59.

Contoh

```
timeframe_group_id,start_time,end_time,service_id
PEAK,06:00:00,09:00:00,WD
PEAK,16:00:00,20:00:00,WD
OFFPEAK,05:00:00,06:00:00,WD
OFFPEAK,09:00:00,16:00:00,WD
OFFPEAK,20:00:00,24:00:00,WD
HOLIDAY,05:00:00,24:00:00,WE
```

---

**`areas.txt`**

File ini mendefinisikan **zona area** yang digunakan sebagai referensi asal dan tujuan perjalanan dalam Fares V2. Setiap stasiun LRT Jabodebek menjadi satu area tersendiri.

`Field`

| Field | Tipe | Keharusan | Deskripsi |
|---|---|---|---|
| `area_id` | Unique ID | **Wajib** | ID unik area digunakan di `stop_areas.txt` dan `fare_leg_rules.txt` |
| `area_name` | Text | Opsional | Nama area yang ditampilkan ke pengguna |

`Panduan untuk LRT Jabodebek`

- `area_id` menggunakan kode stasiun 3 huruf yang sama dengan `zone_id` di `stops.txt` — contoh: `DKA`, `SET`, `CWG`
- `area_name` diisi nama lengkap stasiun

Contoh

```
area_id,area_name
DKA,Dukuh Atas
SET,Setiabudi
RAS,Rasuna Said
CWG,Cawang
...
HAR,Harjamukti
JTM,Jati Mulya
```

---

**`stop_areas.txt`**

File ini mendefinisikan **mapping** antara `stop_id` platform di `stops.txt` dengan `area_id` di `areas.txt`. Memungkinkan Fares V2 mengenali stasiun mana yang termasuk zona mana.

`Field`

| Field | Tipe | Keharusan | Deskripsi |
|---|---|---|---|
| `area_id` | Foreign ID → `areas.area_id` | **Wajib** | ID area stasiun |
| `stop_id` | Foreign ID → `stops.stop_id` | **Wajib** | ID platform yang termasuk dalam area ini |

`Panduan untuk LRT Jabodebek`

- Setiap stasiun memiliki **2 platform** (`01` dan `02`) yang keduanya dipetakan ke satu `area_id`
- Stasiun **Cawang** memiliki **4 platform** (`GCWG01`–`GCWG04`) karena melayani dua lin (Bekasi dan Cibubur) semuanya dipetakan ke area `CWG`
- Hanya platform (`location_type=0`) yang dimasukkan parent station dan entrance tidak perlu

Contoh

```
area_id,stop_id
DKA,GDKA01
DKA,GDKA02
SET,GSET01
SET,GSET02
CWG,GCWG01
CWG,GCWG02
CWG,GCWG03
CWG,GCWG04
...
```

---

**`fare_products.txt`**

File ini mendefinisikan **produk tarif** — setiap kombinasi periode waktu dan nilai nominal menjadi satu produk tarif tersendiri.

`Field`

| Field | Tipe | Keharusan | Deskripsi |
|---|---|---|---|
| `fare_product_id` | Unique ID | **Wajib** | ID unik produk tarif |
| `fare_product_name` | Text | Opsional | Nama produk yang ditampilkan ke pengguna |
| `amount` | Currency amount | **Wajib** | Nominal tarif |
| `currency` | Currency code | **Wajib** | Kode mata uang ISO 4217 |

`Panduan untuk LRT Jabodebek`

- `fare_product_id` diformat sebagai `LRTJAB_{PERIODE}_{nominal}` — contoh: `LRTJAB_PEAK_7500`, `LRTJAB_OFFPEAK_5000`, `LRTJAB_HOLIDAY_5000`
- `fare_product_name` mendeskripsikan produk secara lengkap — contoh: `LRT Jabodebek Jam Sibuk Rp7.500`
- `amount` diisi nominal tarif sesuai periode
- `currency` diisi `IDR`

Contoh

```
fare_product_id,fare_product_name,amount,currency
LRTJAB_OFFPEAK_5000,LRT Jabodebek Luar Jam Sibuk Rp5.000,5000,IDR
LRTJAB_OFFPEAK_5700,LRT Jabodebek Luar Jam Sibuk Rp5.700,5700,IDR
LRTJAB_PEAK_7500,LRT Jabodebek Jam Sibuk Rp7.500,7500,IDR
LRTJAB_HOLIDAY_5000,LRT Jabodebek Hari Libur Rp5.000,5000,IDR
...
```

---

**`fare_leg_rules.txt`**

File ini adalah **inti dari Fares V2** — mendefinisikan aturan tarif lengkap per kombinasi zona asal, zona tujuan, dan periode waktu. Setiap baris merepresentasikan satu aturan tarif yang spesifik.

`Field`

| Field | Tipe | Keharusan | Deskripsi |
|---|---|---|---|
| `leg_group_id` | ID | Opsional | ID grup aturan — untuk mengelompokkan aturan terkait |
| `network_id` | ID | Opsional | ID jaringan transportasi — merujuk ke `networks.txt` atau `routes.network_id` |
| `from_area_id` | Foreign ID → `areas.area_id` | Opsional | Area zona asal |
| `to_area_id` | Foreign ID → `areas.area_id` | Opsional | Area zona tujuan |
| `fare_product_id` | Foreign ID → `fare_products.fare_product_id` | **Wajib** | Produk tarif yang diterapkan |
| `from_timeframe_group_id` | Foreign ID → `timeframes.timeframe_group_id` | Opsional | Periode waktu keberangkatan |
| `to_timeframe_group_id` | Foreign ID → `timeframes.timeframe_group_id` | Opsional | Periode waktu kedatangan — dikosongkan |

`Panduan untuk LRT Jabodebek`

- `leg_group_id` diformat sebagai `{ORIGIN}_{DEST}_{PERIODE}` — contoh: `DKA_SET_PEAK`
- `network_id` diisi `LRTJAB` agar aturan hanya berlaku untuk jaringan LRT Jabodebek
- `from_area_id` dan `to_area_id` menggunakan kode stasiun 3 huruf yang sama dengan `area_id` di `areas.txt`
- `fare_product_id` merujuk ke produk tarif di `fare_products.txt` sesuai periode
- `from_timeframe_group_id` diisi `PEAK`, `OFFPEAK`, atau `HOLIDAY`
- `to_timeframe_group_id` dikosongkan — LRT Jabodebek tidak membedakan tarif berdasarkan waktu tiba
- Total **918 baris** = 306 pasangan O-D × 3 periode waktu

Contoh

```
leg_group_id,network_id,from_area_id,to_area_id,fare_product_id,from_timeframe_group_id,to_timeframe_group_id
DKA_SET_OFFPEAK,LRTJAB,DKA,SET,LRTJAB_OFFPEAK_5000,OFFPEAK,
DKA_SET_PEAK,LRTJAB,DKA,SET,LRTJAB_PEAK_6000,PEAK,
DKA_SET_HOLIDAY,LRTJAB,DKA,SET,LRTJAB_HOLIDAY_5000,HOLIDAY,
DKA_RAS_OFFPEAK,LRTJAB,DKA,RAS,LRTJAB_OFFPEAK_5700,OFFPEAK,
...
```

---

**`Hubungan Antar File`**

```
stops.txt
  zone_id = "DKA"
       │
       ▼
stop_areas.txt          areas.txt
  stop_id = GDKA01  →   area_id = DKA
  stop_id = GDKA02  →   area_id = DKA
                              │
                              ▼
                    fare_leg_rules.txt
                      from_area_id = DKA
                      to_area_id   = SET
                      from_timeframe_group_id = PEAK
                              │
                              ▼
                    fare_products.txt          timeframes.txt
                      fare_product_id =    ←   timeframe_group_id = PEAK
                      LRTJAB_PEAK_6000         service_id = WD
                              │
                              ▼
                    amount = 6000, currency = IDR
```

---

**`Ringkasan Jumlah Baris per File`**

| File | Jumlah Baris | Keterangan |
|---|---|---|
| `fare_attributes.txt` | ~9 | Satu baris per nilai tarif unik offpeak |
| `fare_rules.txt` | 306 | 18 × 17 pasangan O-D |
| `timeframes.txt` | 6 | 2 peak + 3 offpeak + 1 holiday |
| `areas.txt` | 18 | Satu per stasiun |
| `stop_areas.txt` | ~38 | Semua platform per stasiun |
| `fare_products.txt` | ~27 | Nilai tarif unik × 3 periode |
| `fare_leg_rules.txt` | 918 | 306 O-D × 3 periode |

In [62]:
"""
Generate file tarif GTFS (Fares V1 & V2) untuk LRT Jabodebek Lin Bekasi dan Lin Cibubur Fase 1.

Input:
  - fare_offpeak.csv  : tabel tarif luar jam sibuk (matrix O-D)
  - fare_peak.csv     : tabel tarif jam sibuk (matrix O-D)
  - fare_holiday.csv  : tabel tarif hari libur (matrix O-D)

Output Fares V1 (untuk OTP):
  - fare_attributes.txt
  - fare_rules.txt

Output Fares V2 (untuk platform yang support):
  - fare_products.txt
  - fare_leg_rules.txt
  - timeframes.txt
  - areas.txt
  - stop_areas.txt

Catatan:
  - Fares V1 menggunakan tarif offpeak sebagai tarif default
  - Fares V2 mendukung differensiasi peak, offpeak, dan holiday
"""

import pandas as pd
import csv
import os
from datetime import datetime

# File input tabel tarif (matrix O-D)
FARE_FILES = {
    "offpeak" : "data/fare/fare_offpeak.csv",   # Tarif luar jam sibuk
    "peak"    : "data/fare/fare_peak.csv",      # Tarif jam sibuk
    "holiday" : "data/fare/fare_holiday.csv",   # Tarif hari libur
}

# Output directory
OUTPUT_DIR = "."

# Informasi operator
AGENCY_ID  = "KAI"
NETWORK_ID = "LRTJAB"
CURRENCY   = "IDR"

# Mapping kode stasiun → stop_id platform (semua platform per stasiun)
STOP_AREA_MAP = {
    "DKA": ["GDKA01", "GDKA02"],
    "SET": ["GSET01", "GSET02"],
    "RAS": ["GRAS01", "GRAS02"],
    "KUA": ["GKUA01", "GKUA02"],
    "PAN": ["GPAN01", "GPAN02"],
    "CKK": ["GCKK01", "GCKK02"],
    "CIL": ["GCIL01", "GCIL02"],
    "CWG": ["GCWG01", "GCWG02", "GCWG03", "GCWG04"],  # 4 platform (shared station)
    "HAL": ["GHAL01", "GHAL02"],
    "JBU": ["GJBU01", "GJBU02"],
    "CK1": ["GCK101", "GCK102"],
    "CK2": ["GCK201", "GCK202"],
    "BEK": ["GBEK01", "GBEK02"],
    "JTM": ["GJTM01", "GJTM02"],
    "HAR": ["GHAR01", "GHAR02"],
    "KAM": ["GKAM01", "GKAM02"],
    "CRC": ["GCRC01", "GCRC02"],
    "TMI": ["GTMI01", "GTMI02"],
}

# Nama lengkap stasiun untuk areas.txt
STATION_NAMES = {
    "DKA": "Dukuh Atas Bank Syariah Indonesia",
    "SET": "Setiabudi",
    "RAS": "Rasuna Said",
    "KUA": "Kuningan",
    "PAN": "Pancoran bank bjb",
    "CKK": "Cikoko",
    "CIL": "Ciliwung",
    "CWG": "Cawang",
    "HAL": "Halim",
    "JBU": "Jatibening Baru",
    "CK1": "Cikunir 1",
    "CK2": "Cikunir 2",
    "BEK": "Bekasi Barat",
    "JTM": "Jati Mulya",
    "HAR": "Harjamukti",
    "KAM": "Kampung Rambutan",
    "CRC": "Ciracas",
    "TMI": "TMII",
}


# Load tabel tarif

def load_fare_matrix(filepath):
    """
    Load file CSV matrix O-D tarif.
    Return: dict {(origin, destination): fare}
    """
    df = pd.read_csv(filepath, index_col=0)
    fares = {}
    for origin in df.index:
        for dest in df.columns:
            if origin != dest:
                fare = int(df.loc[origin, dest])
                fares[(str(origin), str(dest))] = fare
    return fares


# Fares V1

def generate_fare_attributes(fares_offpeak, output_dir):
    """
    Generate fare_attributes.txt (Fares V1).
    Menggunakan tarif offpeak sebagai tarif default untuk OTP.
    Setiap nilai tarif unik menjadi satu fare_id.
    """
    unique_fares = sorted(set(fares_offpeak.values()))

    rows = []
    for fare in unique_fares:
        rows.append({
            "fare_id"        : f"LRTJAB_{fare}",
            "price"          : fare,
            "currency_type"  : CURRENCY,
            "payment_method" : 1,   # bayar sebelum boarding (tap di gate)
            "transfers"      : 0,   # tidak ada transfer gratis
            "agency_id"      : AGENCY_ID,
        })

    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, "fare_attributes.txt")
    df.to_csv(path, index=False, encoding="utf-8",
              lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    return rows, path


def generate_fare_rules_v1(fares_offpeak, output_dir):
    """
    Generate fare_rules.txt (Fares V1).
    Menggunakan tarif offpeak sebagai tarif default.
    origin_id & destination_id merujuk ke zone_id di stops.txt (= kode stasiun).
    """
    rows = []
    for (origin, dest), fare in fares_offpeak.items():
        rows.append({
            "fare_id"       : f"LRTJAB_{fare}",
            "route_id"      : "",
            "origin_id"     : origin,
            "destination_id": dest,
            "contains_id"   : "",
        })

    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, "fare_rules.txt")
    df.to_csv(path, index=False, encoding="utf-8",
              lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    return rows, path


# Fares V2

def generate_timeframes(output_dir):
    """
    Generate timeframes.txt (Fares V2).

    Tiga timeframe:
      - PEAK    : Senin-Jumat 06:00-08:59 dan 16:00-19:59
      - OFFPEAK : Senin-Jumat di luar jam sibuk
      - HOLIDAY : Sabtu, Minggu & Hari Libur (semua jam)
    """
    rows = [
        # Peak pagi — Senin s/d Jumat
        {"timeframe_group_id": "PEAK", "start_time": "06:00:00",
         "end_time": "09:00:00", "service_id": "WD"},
        # Peak sore — Senin s/d Jumat
        {"timeframe_group_id": "PEAK", "start_time": "16:00:00",
         "end_time": "20:00:00", "service_id": "WD"},
        # Offpeak pagi sebelum peak — Senin s/d Jumat
        {"timeframe_group_id": "OFFPEAK", "start_time": "05:00:00",
         "end_time": "06:00:00", "service_id": "WD"},
        # Offpeak siang — Senin s/d Jumat
        {"timeframe_group_id": "OFFPEAK", "start_time": "09:00:00",
         "end_time": "16:00:00", "service_id": "WD"},
        # Offpeak malam — Senin s/d Jumat
        {"timeframe_group_id": "OFFPEAK", "start_time": "20:00:00",
         "end_time": "24:00:00", "service_id": "WD"},
        # Holiday — Sabtu & Minggu (semua jam operasi)
        {"timeframe_group_id": "HOLIDAY", "start_time": "05:00:00",
         "end_time": "24:00:00", "service_id": "WE"},
    ]

    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, "timeframes.txt")
    df.to_csv(path, index=False, encoding="utf-8",
              lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    return rows, path


def generate_areas(output_dir):
    """
    Generate areas.txt (Fares V2).
    Setiap stasiun menjadi satu area.
    """
    rows = [
        {"area_id": code, "area_name": name}
        for code, name in STATION_NAMES.items()
    ]

    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, "areas.txt")
    df.to_csv(path, index=False, encoding="utf-8",
              lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    return rows, path


def generate_stop_areas(output_dir):
    """
    Generate stop_areas.txt (Fares V2).
    Mapping semua platform stop_id ke area_id stasiun.
    """
    rows = []
    for area_id, stop_ids in STOP_AREA_MAP.items():
        for stop_id in stop_ids:
            rows.append({"area_id": area_id, "stop_id": stop_id})

    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, "stop_areas.txt")
    df.to_csv(path, index=False, encoding="utf-8",
              lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    return rows, path


def generate_fare_products(fares_offpeak, fares_peak, fares_holiday, output_dir):
    """
    Generate fare_products.txt (Fares V2).
    Setiap kombinasi (periode, nilai tarif) menjadi satu fare_product_id.
    """
    rows = []
    seen = set()

    for period, fares in [
        ("OFFPEAK", fares_offpeak),
        ("PEAK",    fares_peak),
        ("HOLIDAY", fares_holiday),
    ]:
        for fare in sorted(set(fares.values())):
            pid = f"LRTJAB_{period}_{fare}"
            if pid not in seen:
                seen.add(pid)
                rows.append({
                    "fare_product_id"  : pid,
                    "fare_product_name": f"LRT Jabodebek {'Jam Sibuk' if period == 'PEAK' else 'Luar Jam Sibuk' if period == 'OFFPEAK' else 'Hari Libur'} Rp{fare:,}",
                    "amount"           : f"{float(fare):.2f}",
                    # "amount"           : fare,
                    "currency"         : CURRENCY,
                })

    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, "fare_products.txt")
    df.to_csv(path, index=False, encoding="utf-8",
              lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    return rows, path


def generate_fare_leg_rules(fares_offpeak, fares_peak, fares_holiday, output_dir):
    """
    Generate fare_leg_rules.txt (Fares V2).
    Setiap kombinasi (O-D, periode waktu) menjadi satu baris.
    """
    rows = []

    configs = [
        ("OFFPEAK", fares_offpeak, "OFFPEAK"),
        ("PEAK",    fares_peak,    "PEAK"),
        ("HOLIDAY", fares_holiday, "HOLIDAY"),
    ]

    for period, fares, timeframe_id in configs:
        for (origin, dest), fare in fares.items():
            rows.append({
                "leg_group_id"          : f"{origin}_{dest}_{period}",
                "network_id"            : NETWORK_ID,
                "from_area_id"          : origin,
                "to_area_id"            : dest,
                "fare_product_id"       : f"LRTJAB_{period}_{fare}",
                "from_timeframe_group_id": timeframe_id,
                "to_timeframe_group_id" : "",
            })

    df = pd.DataFrame(rows)
    path = os.path.join(output_dir, "fare_leg_rules.txt")
    df.to_csv(path, index=False, encoding="utf-8",
              lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
    return rows, path


def main():
    print("=" * 60)
    print(" Generate Fare Files  |  LRT Jabodebek")
    print(f" {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f" Fares V1 : fare_attributes.txt, fare_rules.txt")
    print(f" Fares V2 : fare_products.txt, fare_leg_rules.txt,")
    print(f"            timeframes.txt, areas.txt, stop_areas.txt")
    print("=" * 60)

    # Validasi file input
    for period, filepath in FARE_FILES.items():
        if not os.path.exists(filepath):
            print(f"\n❌ File tidak ditemukan: {filepath}")
            print(f"   Pastikan file CSV tabel tarif tersedia.")
            return

    # Load tabel tarif
    print("\n📂 Loading tabel tarif ...")
    fares_offpeak = load_fare_matrix(FARE_FILES["offpeak"])
    fares_peak    = load_fare_matrix(FARE_FILES["peak"])
    fares_holiday = load_fare_matrix(FARE_FILES["holiday"])

    n_od = len(fares_offpeak)
    print(f"   Pasangan O-D : {n_od}")
    print(f"   Tarif unik offpeak : {sorted(set(fares_offpeak.values()))}")
    print(f"   Tarif unik peak    : {sorted(set(fares_peak.values()))}")
    print(f"   Tarif unik holiday : {sorted(set(fares_holiday.values()))}")

    # Fares V1
    print(f"\n{'─'*55}")
    print("📄 Generating Fares V1 ...")

    _, path = generate_fare_attributes(fares_offpeak, OUTPUT_DIR)
    print(f"   ✅ fare_attributes.txt → {path}")

    rows_v1, path = generate_fare_rules_v1(fares_offpeak, OUTPUT_DIR)
    print(f"   ✅ fare_rules.txt      → {path} ({len(rows_v1)} baris)")

    # Fares V2
    print(f"\n{'─'*55}")
    print("📄 Generating Fares V2 ...")

    _, path = generate_timeframes(OUTPUT_DIR)
    print(f"   ✅ timeframes.txt      → {path}")

    _, path = generate_areas(OUTPUT_DIR)
    print(f"   ✅ areas.txt           → {path} ({len(STATION_NAMES)} area)")

    rows_sa, path = generate_stop_areas(OUTPUT_DIR)
    print(f"   ✅ stop_areas.txt      → {path} ({len(rows_sa)} baris)")

    rows_fp, path = generate_fare_products(fares_offpeak, fares_peak, fares_holiday, OUTPUT_DIR)
    print(f"   ✅ fare_products.txt   → {path} ({len(rows_fp)} produk)")

    rows_flr, path = generate_fare_leg_rules(fares_offpeak, fares_peak, fares_holiday, OUTPUT_DIR)
    print(f"   ✅ fare_leg_rules.txt  → {path} ({len(rows_flr)} baris)")

    print(f"\n{'='*55}")
    print(f"✅ Semua file tarif berhasil dibuat!")
    print()
    print(f"   {'File':<25} {'Keterangan'}")
    print(f"   {'─'*50}")
    print(f"   {'fare_attributes.txt':<25} V1 — {len(set(fares_offpeak.values()))} kelas tarif (offpeak sebagai default)")
    print(f"   {'fare_rules.txt':<25} V1 — {len(rows_v1)} pasangan O-D")
    print(f"   {'timeframes.txt':<25} V2 — 3 periode (PEAK, OFFPEAK, HOLIDAY)")
    print(f"   {'areas.txt':<25} V2 — {len(STATION_NAMES)} area stasiun")
    print(f"   {'stop_areas.txt':<25} V2 — {len(rows_sa)} mapping stop → area")
    print(f"   {'fare_products.txt':<25} V2 — {len(rows_fp)} produk tarif")
    print(f"   {'fare_leg_rules.txt':<25} V2 — {len(rows_flr)} aturan tarif")

if __name__ == "__main__":
    main()

 Generate Fare Files  |  LRT Jabodebek
 2026-05-03 22:32:43
 Fares V1 : fare_attributes.txt, fare_rules.txt
 Fares V2 : fare_products.txt, fare_leg_rules.txt,
            timeframes.txt, areas.txt, stop_areas.txt

📂 Loading tabel tarif ...
   Pasangan O-D : 306
   Tarif unik offpeak : [5000, 5700, 6400, 7100, 7800, 8500, 9200, 9900, 10000]
   Tarif unik peak    : [5000, 5700, 6400, 7100, 7800, 8500, 9200, 9900, 10600, 11300, 12000, 12700, 13400, 14100, 14800, 15500, 16200, 16900, 17600, 18300, 19000, 19700, 20000]
   Tarif unik holiday : [5000, 5700, 6400, 7100, 7800, 8500, 9200, 9900, 10000]

───────────────────────────────────────────────────────
📄 Generating Fares V1 ...
   ✅ fare_attributes.txt → ./fare_attributes.txt
   ✅ fare_rules.txt      → ./fare_rules.txt (306 baris)

───────────────────────────────────────────────────────
📄 Generating Fares V2 ...
   ✅ timeframes.txt      → ./timeframes.txt
   ✅ areas.txt           → ./areas.txt (18 area)
   ✅ stop_areas.txt      → ./stop_ar